In [0]:
CREATE OR REPLACE TEMPORARY VIEW global_runtime_dates AS
SELECT 
    /* ELAPRASE */
    DATE('2020-08-01') AS elaprase_dx_start_date,
    DATE('2023-08-01') AS elaprase_tx_start_date,

    /* AVLAYAH */
    -- DATE('2022-08-01') AS avlayah_dx_start_date,
    DATE('2026-03-01') AS avlayah_tx_start_date;

In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
%sql

WITH elaprase_patients AS (

    SELECT DISTINCT patient_id

    FROM com_edp_prd.com_raw.kom_medical_events

    WHERE procedure_code = 'J1743'
       OR ndc11 IN ('54092070001','540920700')

),

other_jcode_2026 AS (

    SELECT DISTINCT patient_id

    FROM com_edp_prd.com_raw.kom_medical_events

    WHERE YEAR(service_date) = 2026
      AND procedure_code LIKE 'J%'
      AND procedure_code <> 'J1743'

)

SELECT DISTINCT
    p360.patient_id

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master p360

INNER JOIN elaprase_patients ep
    ON p360.patient_id = ep.patient_id

INNER JOIN other_jcode_2026 oj
    ON p360.patient_id = oj.patient_id;

In [0]:
Select * from cmpa_insights_internal_schema.patient360_master
WHERE patient_id IN (
'MXMH1384',
'33PEP1TE',
'SJBF10CK',
'MP1CXKGN',
'QWZMHPTJ',
'X5Z8G6EH',
'YZKGS5C3',
'K3VNKN2P',
'39BDW50Z',
'BTYB2XRN',
'C0501NXE',
'GYK64FYG',
'NLG0YH5M',
'DKQL3NMK',
'03DRQPXS',
'BGYXB8PE',
'755R9342',
'SFC8E2KF',
'5B5YJ0BJ',
'R4VLPN8E',
'HPS005NZ',
'Y1WFSTMH',
'VHXPQ5P2',
'M53EJ7KW',
'MLZEPGWG',
'CSJM9SYS',
'X6PF2Z3C',
'7SBZKMP0',
'T3E9D98L',
'Q9KQ4DD7',
'7TH16LJF',
'P80QX3EB',
'YRCV19C9',
'5HREMYP4',
'GPHH8T0K',
'905ESMNC',
'2XPN1XBS',
'J0CRCXC9',
'7W3FYDPQ',
'00P0S2C0',
'SFKC682M',
'3HKL7WPG',
'0CFBS3QP',
'R3XPVFMF',
'L0V77BWT',
'6YWE9Z2W',
'JJPL15PT',
'GSQG36ZD',
'96TFJT9N',
'MS7PBQ8Z',
'RBJZ2XC4',
'261HTDTY',
'MHJ1435L',
'PCKXDFEC',
'9GX8BDT6',
'MMMLTS10',
'SH1YT7BM',
'0JXVTG3V',
'5Z4BWQ9S',
'47DCKLNL',
'PH2YLV53',
'EP1WGTKX',
'LHETQKY2',
'G0MWMV8M',
'D8NGFVRN',
'3KYL0H3S',
'LS4M6FMC',
'LZT622TW',
'5YWQKG7T',
'WHVW1WH9',
'K0K723RQ',
'84HGEGMM',
'M4V6S82Z',
'EE8C86W3',
'W8VRKMD4'
);

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim AS
WITH base AS (
    SELECT DISTINCT
        CAST(payer_id AS STRING) AS payer_id,
        PAYER_NAME,
        CASE
            WHEN UPPER(PAYER_NAME) RLIKE 'UNITED|OPTUM' THEN 'UHC/Optum'
            WHEN UPPER(PAYER_NAME) RLIKE 'AETNA|CVS' THEN 'Aetna/CVS'
            WHEN UPPER(PAYER_NAME) RLIKE 'CIGNA|ESI|EVERNORTH' THEN 'Cigna/ESI'
            WHEN UPPER(PAYER_NAME) RLIKE 'ANTHEM|ELEVANCE|CARELON' THEN 'Elevance/Carelon'
            WHEN UPPER(PAYER_NAME) RLIKE 'ILLINOIS|TEXAS|OKLAHOMA|NEW MEXICO' THEN 'Prime Therapeutics / HCSC'
            ELSE PAYER_NAME
        END AS payer_display_name
    FROM com_edp_prd.com_raw.kom_plans
),

rollup_ids AS (
    SELECT
        payer_display_name,
        DENSE_RANK() OVER (ORDER BY payer_display_name) + 1000 AS payer_display_id
    FROM (
        SELECT DISTINCT payer_display_name FROM base
    )
)
SELECT
    b.PAYER_ID AS source_payer_id,
    b.PAYER_NAME AS source_payer_name,
    r.payer_display_id,
    r.payer_display_name,
    r.payer_display_id AS canonical_payer_id,
    r.payer_display_name AS canonical_payer_name
FROM base b
LEFT JOIN rollup_ids r ON b.payer_display_name = r.payer_display_name;


CREATE OR REPLACE TEMPORARY VIEW cohort_universe AS
SELECT DISTINCT patient_id
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master;


CREATE OR REPLACE TEMPORARY VIEW total_lives AS
WITH all_claims AS (
    SELECT DISTINCT * FROM (
        SELECT DISTINCT
            PATIENT_ID,
            COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
            KH_PLAN_ID AS plan_id,
            MEDICAL_EVENT_ID AS claim_id
        FROM com_edp_prd.com_raw.kom_medical_events
        UNION ALL
        SELECT DISTINCT
            PATIENT_ID,
            PRESCRIBER_NPI AS NPI,
            COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
            PHARMACY_EVENT_ID AS claim_id
        FROM com_edp_prd.com_raw.kom_pharmacy_events
    )
),
tagging_zip_v1 AS (
    SELECT
        a.*,
        LPAD(COALESCE(
            NULLIF(b.hco_zip, '-'),
            NULLIF(b.hcp_zip, '-'),
            c.PROVIDER_ZIP
        ), 5, '0') AS final_zip
    FROM all_claims AS a
    LEFT JOIN cmpa_insights_internal_schema.reference_file AS b ON a.npi = b.hcp_npi
    LEFT JOIN com_raw.kom_providers AS c ON a.npi = c.npi AND c.PROVIDER_TYPE = 'INDIVIDUAL'
),
tagging_territory AS (
    SELECT
        a.*,
        COALESCE(CAST(b.territory_id AS STRING), 'Unknown') AS territory_id,
        COALESCE(b.territory_name, 'Unknown') AS territory
    FROM tagging_zip_v1 a
    LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON a.final_zip = LPAD(TRY_CAST(b.zipcode AS STRING), 5, '0')
),
tagging_payer AS (
    SELECT
        a.*,
        COALESCE(CAST(r.canonical_payer_id AS STRING), CAST(b.PAYER_ID AS STRING)) AS PAYER_ID,
        COALESCE(r.canonical_payer_name, b.PAYER_NAME) AS PAYER_NAME
    FROM tagging_territory a
    LEFT JOIN com_raw.kom_plans b ON a.plan_id = b.KH_PLAN_ID
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
        ON b.PAYER_ID = r.source_payer_id
),
agg AS (
    SELECT
        COALESCE(territory_id, 'ALL Territories') AS territory_id,
        COALESCE(territory, 'All Territories') AS territory,
        COALESCE(payer_id, 'ALL Payers') AS payer_id,
        COALESCE(payer_name, 'All Payers') AS payer_name,
        COUNT(DISTINCT patient_id) AS total_lives,
        CASE
            WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
            WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
            WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
            WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
        END AS rollup_level
    FROM tagging_payer
    GROUP BY GROUPING SETS (
        (territory_id, territory, payer_id, payer_name),
        (payer_id, payer_name),
        (territory_id, territory),
        ()
    )
)
SELECT *
FROM agg
ORDER BY rollup_level, total_lives DESC;


CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
  AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
UNION
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    PHARMACY_EVENT_ID AS claim_id,
    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
    'PHARMACY' AS CLAIM_SOURCE,
    TRANSACTION_RESULT AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
  AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe);


CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    NDC11 AS CODE,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700','8497600101')
  AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
  AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
UNION
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    PROCEDURE_CODE AS CODE,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
  AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
UNION
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    PHARMACY_EVENT_ID AS claim_id,
    NDC11 AS CODE,
    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
    'PHARMACY' AS CLAIM_SOURCE,
    TRANSACTION_RESULT AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700','8497600101')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
  AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
UNION
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    PROCEDURE_CODE AS CODE,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
  AND SERVICE_DATE BETWEEN '2026-03-01' AND (SELECT end_date FROM runtime_parameters)
  AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe);


CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN all_tx_claims t ON e.PATIENT_ID = t.PATIENT_ID;


CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;


CREATE OR REPLACE TEMPORARY VIEW elaprase_tx AS
SELECT DISTINCT PATIENT_ID
FROM all_tx_claims
WHERE CODE IN ('54092070001', '540920700', 'J1743','99601', '99602', '96365', '96366', 'J1743','S9357', 'S9379', '38206', '38230', '38232','38240', '38241', '38242', '38243', '38250');

CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);


CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT patient_id AS PATIENT_ID FROM cohort_universe;


CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
SELECT DISTINCT
    PATIENT_ID, NPI, FILL_DATE, claim_id, plan_id, CLAIM_SOURCE, TRANSACTION_STATUS
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
UNION
SELECT DISTINCT
    PATIENT_ID, NPI, FILL_DATE, claim_id, plan_id, CLAIM_SOURCE, TRANSACTION_STATUS
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT patient_id FROM cohort_universe);


CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%' THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%' THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%' THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%' THEN 'NPPA'
            WHEN a.NPI IS NULL THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%' THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%' THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%' THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%' THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%' THEN 5
            WHEN a.NPI IS NULL THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT
    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p ON a.NPI = p.NPI
    GROUP BY a.PATIENT_ID, a.NPI, p.primary_specialty, p.secondary_specialty
),
ranked_hcps AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY SPECIALTY_PRIORITY ASC, NO_OF_VISITS DESC, MOST_RECENT_VISIT DESC, NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
),
hco_addition AS (
    SELECT
        a.PATIENT_ID AS patient_id,
        a.NPI AS hcp_npi,
        a.SPECIALTY AS hcp_specialty,
        CASE
            WHEN b.hcp_name IS NOT NULL THEN b.hcp_name
            ELSE CONCAT(c.FIRST_NAME, " ", c.LAST_NAME)
        END AS hcp_name,
        b.hco_veeva_crm_id,
        b.hco_name
    FROM ranked_hcps a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file b ON a.NPI = b.hcp_npi
    LEFT JOIN com_raw.kom_providers c ON a.NPI = c.NPI AND c.PROVIDER_TYPE = 'INDIVIDUAL'
    WHERE a.HCP_RANK = 1
)
SELECT DISTINCT * FROM hco_addition;


CREATE OR REPLACE TEMP VIEW avlayah_u17_first_date AS
SELECT
    t.PATIENT_ID,
    MIN(t.FILL_DATE) AS FIRST_AVLAYAH_U17_DATE
FROM all_tx_claims t
INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p
    ON t.PATIENT_ID = p.PATIENT_ID
WHERE t.CODE IN ('8497600101','J3490','J3590','J9999')
  AND p.patient_age < 17
GROUP BY t.PATIENT_ID;


CREATE OR REPLACE TEMP VIEW new_patient_flags AS
SELECT
    a.PATIENT_ID,
    MIN(a.FILL_DATE) AS FIRST_EVENT_DATE,
    u.FIRST_AVLAYAH_U17_DATE,
    CASE
        WHEN MIN(a.FILL_DATE) >= DATEADD(month, -1, (SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0
    END AS NEW_PATIENT_R1M,
    CASE
        WHEN MIN(a.FILL_DATE) >= DATEADD(month, -3, (SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0
    END AS NEW_PATIENT_R3M,
    CASE
        WHEN u.FIRST_AVLAYAH_U17_DATE >= DATEADD(month, -1, (SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0
    END AS NEW_PATIENT_R1M_AVLAYAH_U17,
    CASE
        WHEN u.FIRST_AVLAYAH_U17_DATE >= DATEADD(month, -3, (SELECT end_date FROM runtime_parameters))
        THEN 1 ELSE 0
    END AS NEW_PATIENT_R3M_AVLAYAH_U17
FROM all_patient_claims a
LEFT JOIN avlayah_u17_first_date u ON a.PATIENT_ID = u.PATIENT_ID
WHERE a.PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
GROUP BY a.PATIENT_ID, u.FIRST_AVLAYAH_U17_DATE;


CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level AS
WITH eligible_patient_universe AS (
    SELECT DISTINCT patient_id FROM cohort_universe
),
patient_claim_counts AS (
    SELECT patient_id, COUNT(DISTINCT claim_id) AS claims_count
    FROM all_patient_claims
    WHERE patient_id IN (SELECT patient_id FROM cohort_universe)
    GROUP BY 1
),
eligible_patients_with_claims AS (
    SELECT DISTINCT
        a.patient_id,
        b.claims_count
    FROM eligible_patient_universe a
    LEFT JOIN patient_claim_counts b ON a.patient_id = b.patient_id
),
patients_with_primary_hcp AS (
    SELECT
        a.*,
        b.* EXCEPT (b.patient_id)
    FROM eligible_patients_with_claims a
    LEFT JOIN primary_hcp b ON a.patient_id = b.patient_id
),
patients_with_final_zip AS (
    SELECT
        a.*,
        LPAD(COALESCE(
            NULLIF(b.hco_zip, '-'),
            NULLIF(b.hcp_zip, '-'),
            c.PROVIDER_ZIP
        ), 5, '0') AS final_zip
    FROM patients_with_primary_hcp a
    LEFT JOIN cmpa_insights_internal_schema.reference_file b ON a.hcp_npi = b.hcp_npi
    LEFT JOIN com_raw.kom_providers c ON a.hcp_npi = c.NPI AND c.PROVIDER_TYPE = 'INDIVIDUAL'
),
patients_with_territory_region AS (
    SELECT
        a.*,
        b.territory_id,
        b.territory_name AS territory,
        b.region_id,
        b.region_name AS region
    FROM patients_with_final_zip a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b
        ON a.final_zip = LPAD(TRY_CAST(b.zipcode AS STRING), 5, '0')
),
latest_plan_per_patient AS (
    SELECT b.patient_id, b.plan_id
    FROM (
        SELECT
            a.patient_id,
            a.plan_id,
            ROW_NUMBER() OVER (
                PARTITION BY a.patient_id
                ORDER BY a.fill_date DESC, a.npi ASC
            ) AS rn
        FROM all_patient_claims a
        WHERE a.plan_id IS NOT NULL
          AND a.patient_id IN (SELECT patient_id FROM cohort_universe)
    ) b
    WHERE b.rn = 1
),
patients_with_latest_plan AS (
    SELECT a.*, b.plan_id
    FROM patients_with_territory_region a
    LEFT JOIN latest_plan_per_patient b ON a.patient_id = b.patient_id
),
patients_with_payer_attributes AS (
    SELECT DISTINCT
        a.*,
        COALESCE(r.canonical_payer_id, b.PAYER_ID, 0) AS PAYER_ID,
        COALESCE(r.canonical_payer_name, b.PAYER_NAME, 'Unknown') AS PAYER_NAME,
        COALESCE(r.canonical_payer_id, b.PAYER_ID, 0) AS PARENT_ID,
        COALESCE(r.canonical_payer_name, b.PAYER_NAME, 'Unknown') AS PARENT_NAME,
        b.INSURANCE_SEGMENT,
        b.INSURANCE_GROUP
    FROM patients_with_latest_plan a
    LEFT JOIN com_edp_prd.com_raw.kom_plans b ON a.plan_id = b.KH_PLAN_ID
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
        ON b.PAYER_ID = r.source_payer_id
),
patients_with_tx_flags AS (
    SELECT
        p.*,
        CASE
            WHEN p360.patient_age < 17
             AND UPPER(p360.latest_treatment_type) = 'AVLAYAH'
            THEN 1 ELSE 0
        END AS avlayah_pt_lt_17,
        CASE
            WHEN p360.patient_age < 17
             AND UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT')
            THEN 1 ELSE 0
        END AS elaprase_pt_lt_17
    FROM patients_with_payer_attributes p
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p360
        ON p.patient_id = p360.patient_id
),
patients_dedup AS (
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY patient_id
                ORDER BY
                    CASE WHEN PAYER_NAME = 'Unknown' THEN 2 ELSE 1 END,
                    CASE WHEN territory_id IS NULL OR territory = 'Unknown' THEN 2 ELSE 1 END,
                    plan_id DESC NULLS LAST
            ) AS dedup_rn
        FROM patients_with_tx_flags
    )
    WHERE dedup_rn = 1
)
SELECT
    patient_id,
    claims_count,
    hcp_npi,
    hcp_specialty,
    final_zip,
    hcp_name,
    hco_veeva_crm_id,
    hco_name,
    COALESCE(CAST(territory_id AS STRING), 'Unknown') AS territory_id,
    COALESCE(territory, 'Unknown') AS territory,
    region_id,
    COALESCE(region, 'Unknown') AS region,
    plan_id,
    COALESCE(PAYER_ID, 'Unknown') AS PAYER_ID,
    COALESCE(PAYER_NAME, 'Unknown') AS PAYER_NAME,
    COALESCE(PARENT_ID, 'Unknown') AS PARENT_ID,
    COALESCE(PARENT_NAME, 'Unknown') AS PARENT_NAME,
    COALESCE(INSURANCE_SEGMENT, 'Unknown') AS INSURANCE_SEGMENT,
    COALESCE(INSURANCE_GROUP, 'Unknown') AS INSURANCE_GROUP,
    avlayah_pt_lt_17,
    elaprase_pt_lt_17
FROM patients_dedup;


CREATE OR REPLACE TEMP VIEW elaprase_provider_universe AS
WITH raw_provider_claims AS (
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
        BILLING_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (
            DIAGNOSIS_CODES LIKE '%E761%'
         OR DIAGNOSIS_CODES LIKE '%E763%'
         OR NDC11 IN ('54092070001','540920700')
         OR PROCEDURE_CODE IN (
             '99601','99602','96365','96366','J1743',
             'S9357','S9379','38206','38230','38232',
             '38240','38241','38242','38243','38250'
         )
    )
    AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
    UNION
    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS HCP_NPI,
        PHARMACY_NPI AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE (
            DIAGNOSIS_CODE IN ('E761','E763')
         OR NDC11 IN ('54092070001','540920700')
    )
    AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND PATIENT_ID IN (SELECT patient_id FROM cohort_universe)
)
SELECT DISTINCT
    rpc.PATIENT_ID,
    rpc.HCP_NPI,
    rpc.HCO_NPI,
    COALESCE(p.PAYER_ID, 0) AS payer_id,
    COALESCE(p.PAYER_NAME, 'Unknown') AS payer_name,
    COALESCE(p.PARENT_ID, 0) AS parent_id,
    COALESCE(p.PARENT_NAME, 'Unknown') AS parent_name,
    p.territory_id,
    p.territory,
    p.region_id,
    p.region
FROM raw_provider_claims rpc
JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    ON rpc.PATIENT_ID = p.patient_id;


CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer360_master AS
WITH tx_classification AS (
    SELECT
        patient_id,
        claim_id,
        drug_type,
        is_denial,
        fill_date
    FROM (
        SELECT
            patient_id,
            medical_event_id AS claim_id,
            service_date AS fill_date,
            CASE
                WHEN ndc11 IN ('54092070001','540920700') THEN 'ELAPRASE'
                WHEN procedure_code IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250') THEN 'ELAPRASE'
               WHEN ndc11 = '8497600101' THEN 'AVLAYAH'
                WHEN procedure_code IN ('J3490','J3590','J9999')
                    AND service_date >= '2026-03-01'
                THEN 'AVLAYAH'
            END AS drug_type,
            0 AS is_denial
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE patient_id IN (SELECT patient_id FROM cohort_universe)
        UNION ALL
        SELECT
        patient_id,
        pharmacy_event_id AS claim_id,
        fill_date,
            CASE
                WHEN ndc11 IN ('54092070001','540920700') THEN 'ELAPRASE'
                WHEN ndc11 IN ('8497600101') THEN 'AVLAYAH'
            END AS drug_type,
            CASE
                WHEN UPPER(transaction_result) = 'REJECTED' THEN 1
                ELSE 0
            END AS is_denial
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE patient_id IN (SELECT patient_id FROM cohort_universe)
    ) t
    WHERE drug_type IS NOT NULL
),

patient_age AS (
    SELECT
        p.patient_id,
        YEAR(CURRENT_DATE) - YEAR(d.patient_yob) AS CURRENT_AGE
    FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    LEFT JOIN (
        SELECT patient_id, MAX(patient_yob) AS patient_yob
        FROM com_edp_prd.com_raw.kom_patient_demographics
        GROUP BY patient_id
    ) d ON p.patient_id = d.patient_id
),
new_patient_flags AS (
    SELECT
        a.patient_id,

        MIN(a.fill_date) AS first_event_date,

        MIN(
            CASE
                WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH'
                 AND pa.current_age < 17
                 AND t.drug_type = 'AVLAYAH'
                THEN t.fill_date
            END
        ) AS first_avlayah_u17_date,

        CASE
            WHEN MIN(a.fill_date) >= ADD_MONTHS(
                    (SELECT end_date FROM runtime_parameters),
                    -1
                 )
            THEN 1 ELSE 0
        END AS new_patient_r1m,

        CASE
            WHEN MIN(a.fill_date) >= ADD_MONTHS(
                    (SELECT end_date FROM runtime_parameters),
                    -3
                 )
            THEN 1 ELSE 0
        END AS new_patient_r3m,

        CASE
            WHEN MIN(
                    CASE
                        WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH'
                         AND pa.current_age < 17
                         AND t.drug_type = 'AVLAYAH'
                        THEN t.fill_date
                    END
                 ) >= ADD_MONTHS(
                        (SELECT end_date FROM runtime_parameters),
                        -1
                    )
            THEN 1 ELSE 0
        END AS new_patient_r1m_avlayah_u17,

        CASE
            WHEN MIN(
                    CASE
                        WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH'
                         AND pa.current_age < 17
                         AND t.drug_type = 'AVLAYAH'
                        THEN t.fill_date
                    END
                 ) >= ADD_MONTHS(
                        (SELECT end_date FROM runtime_parameters),
                        -3
                    )
            THEN 1 ELSE 0
        END AS new_patient_r3m_avlayah_u17

    FROM all_patient_claims a

    LEFT JOIN tx_classification t
        ON a.patient_id = t.patient_id

    LEFT JOIN patient_age pa
        ON a.patient_id = pa.patient_id

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p360
        ON a.patient_id = p360.patient_id

    WHERE a.patient_id IN (
        SELECT patient_id
        FROM cohort_universe
    )
    GROUP BY a.patient_id
),
patient_claim_metrics AS (
    SELECT
        p.patient_id,
        COUNT(DISTINCT a.claim_id) AS TOTAL_CLAIMS,
        COUNT(DISTINCT ph.PHARMACY_EVENT_ID) AS PHARMACY_TOTAL_CLAIMS,
        COUNT(DISTINCT CASE
            WHEN UPPER(ph.TRANSACTION_RESULT) = 'PAID' AND ph.ndc11 IN ('54092070001','540920700')
            THEN ph.PHARMACY_EVENT_ID END) AS ELAPRASE_APPROVED_FILLS,
        COUNT(DISTINCT CASE
            WHEN UPPER(ph.TRANSACTION_RESULT) = 'REJECTED' AND ph.ndc11 IN ('54092070001','540920700')
            THEN ph.PHARMACY_EVENT_ID END) AS ELAPRASE_REJECTED_FILLS,
        COUNT(DISTINCT CASE
            WHEN UPPER(ph.TRANSACTION_RESULT) = 'REVERSED' AND ph.ndc11 IN ('54092070001','540920700')
            THEN ph.PHARMACY_EVENT_ID END) AS ELAPRASE_REVERSED_FILLS,
        COUNT(DISTINCT CASE
            WHEN ph.ndc11 IN ('54092070001','540920700')
            THEN ph.PHARMACY_EVENT_ID END) AS ELAPRASE_PHARMACY_CLAIMS,
        COUNT(DISTINCT CASE
            WHEN UPPER(ph.TRANSACTION_RESULT) = 'PAID' AND ph.ndc11 IN ('8497600101')
            THEN ph.PHARMACY_EVENT_ID END) AS AVLAYAH_APPROVED_FILLS,
        COUNT(DISTINCT CASE
            WHEN UPPER(ph.TRANSACTION_RESULT) = 'REJECTED' AND ph.ndc11 IN ('8497600101')
            THEN ph.PHARMACY_EVENT_ID END) AS AVLAYAH_REJECTED_FILLS,
        COUNT(DISTINCT CASE
            WHEN UPPER(ph.TRANSACTION_RESULT) = 'REVERSED' AND ph.ndc11 IN ('8497600101')
            THEN ph.PHARMACY_EVENT_ID END) AS AVLAYAH_REVERSED_FILLS,
        COUNT(DISTINCT CASE
            WHEN ph.ndc11 IN ('8497600101')
            THEN ph.PHARMACY_EVENT_ID END) AS AVLAYAH_PHARMACY_CLAIMS
    FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    LEFT JOIN (
        SELECT patient_id, claim_id FROM all_patient_claims
        WHERE patient_id IN (SELECT patient_id FROM cohort_universe)
    ) a ON p.patient_id = a.patient_id
    LEFT JOIN com_edp_prd.com_raw.kom_pharmacy_events ph
        ON p.patient_id = ph.patient_id
       AND ph.fill_date BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
       AND (
            ph.diagnosis_code IN ('E761','E763')
            OR ph.ndc11 IN ('54092070001','540920700','8497600101')
           )
    GROUP BY p.patient_id
),
payer_group_map AS (
    SELECT
        p.*,
        p.payer_id AS payer_display_id,
        p.payer_name AS payer_display_name,
        p.payer_name AS payer_group
    FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
),

patient_drug_metrics AS (
    SELECT
        p360.patient_id,
        CASE WHEN a.current_age < 17 THEN 1 ELSE 0 END AS is_lt17,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') THEN 1 ELSE 0 END) AS elaprase_flag,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' THEN 1 ELSE 0 END) AS avlayah_flag,
        COUNT(DISTINCT CASE WHEN UPPER(t.drug_type) IN ('ELAPRASE','OTHER ERT') THEN t.claim_id END) AS elaprase_claims_ct,
        COUNT(DISTINCT CASE WHEN UPPER(t.drug_type) = 'AVLAYAH' THEN t.claim_id END) AS avlayah_claims_ct,
        SUM(CASE WHEN UPPER(t.drug_type) IN ('ELAPRASE','OTHER ERT') AND t.is_denial = 1 THEN 1 ELSE 0 END) AS elaprase_denials,
        SUM(CASE WHEN UPPER(t.drug_type) = 'AVLAYAH' AND t.is_denial = 1 THEN 1 ELSE 0 END) AS avlayah_denials,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') AND p.insurance_group = 'COMMERCIAL' THEN 1 ELSE 0 END) AS elaprase_comm,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') AND p.insurance_group = 'MEDICARE' THEN 1 ELSE 0 END) AS elaprase_medicare,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') AND p.insurance_group = 'MEDICAID' THEN 1 ELSE 0 END) AS elaprase_medicaid,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') AND p.insurance_group NOT IN ('COMMERCIAL','MEDICARE','MEDICAID') THEN 1 ELSE 0 END) AS elaprase_other,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' AND p.insurance_group = 'COMMERCIAL' THEN 1 ELSE 0 END) AS avlayah_comm,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' AND p.insurance_group = 'MEDICARE' THEN 1 ELSE 0 END) AS avlayah_medicare,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' AND p.insurance_group = 'MEDICAID' THEN 1 ELSE 0 END) AS avlayah_medicaid,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' AND p.insurance_group NOT IN ('COMMERCIAL','MEDICARE','MEDICAID') THEN 1 ELSE 0 END) AS avlayah_other,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') AND a.current_age < 17 THEN 1 ELSE 0 END) AS elaprase_pt_lt_17,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' AND a.current_age < 17 THEN 1 ELSE 0 END) AS avlayah_pt_lt_17,
        COUNT(DISTINCT CASE
            WHEN UPPER(t.drug_type) IN ('ELAPRASE','OTHER ERT') AND a.current_age < 17
            THEN t.claim_id END) AS elaprase_claims_lt_17,
        COUNT(DISTINCT CASE
            WHEN UPPER(t.drug_type) = 'AVLAYAH' AND a.current_age < 17
            THEN t.claim_id END) AS avlayah_claims_lt_17,
        SUM(CASE
            WHEN UPPER(t.drug_type) IN ('ELAPRASE','OTHER ERT') AND t.is_denial = 1 AND a.current_age < 17
            THEN 1 ELSE 0 END) AS elaprase_denial_ct_lt_17,
        SUM(CASE
            WHEN UPPER(t.drug_type) = 'AVLAYAH' AND t.is_denial = 1 AND a.current_age < 17
            THEN 1 ELSE 0 END) AS avlayah_denial_ct_lt_17,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') AND a.current_age < 5 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_LT_5_YRS,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') AND a.current_age BETWEEN 5 AND 10 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_5_TO_10_YRS,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') AND a.current_age BETWEEN 11 AND 16 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_11_TO_16_YRS,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') AND a.current_age >= 17 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_17NGRT_YRS,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' AND a.current_age < 5 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_LT_5_YRS,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' AND a.current_age BETWEEN 5 AND 10 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_5_TO_10_YRS,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' AND a.current_age BETWEEN 11 AND 16 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_11_TO_16_YRS,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' AND a.current_age >= 17 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_17NGRT_YRS
        
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master p360
LEFT JOIN payer_group_map p ON p360.patient_id = p.patient_id
LEFT JOIN tx_classification t ON p360.patient_id = t.patient_id
LEFT JOIN patient_age a ON p360.patient_id = a.patient_id
GROUP BY
    p360.patient_id,
    a.current_age
),
patient_drug_hcp_hco AS (
    SELECT
        p.patient_id,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') THEN p.hcp_npi END) AS elaprase_hcp_npi,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' THEN p.hcp_npi END) AS avlayah_hcp_npi,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) IN ('ELAPRASE','OTHER ERT') THEN p.hco_veeva_crm_id END) AS elaprase_hco_id,
        MAX(CASE WHEN UPPER(p360.latest_treatment_type) = 'AVLAYAH' THEN p.hco_veeva_crm_id END) AS avlayah_hco_id
    FROM payer_group_map p
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p360
    ON p.patient_id = p360.patient_id
LEFT JOIN tx_classification t
    ON p.patient_id = t.patient_id
GROUP BY p.patient_id
),
payer_info_normalized AS (
    SELECT
        CASE
            WHEN UPPER(payer_account_name) RLIKE 'UNITED|OPTUM' THEN 'UHC/Optum'
            WHEN UPPER(payer_account_name) RLIKE 'AETNA|CVS' THEN 'Aetna/CVS'
            WHEN UPPER(payer_account_name) RLIKE 'CIGNA|ESI|EVERNORTH|EXPRESS SCRIPTS' THEN 'Cigna/ESI'
            WHEN UPPER(payer_account_name) RLIKE 'ANTHEM|ELEVANCE|CARELON|AMERIGROUP|WELLPOINT|EMPIRE.*BLUE' THEN 'Elevance/Carelon'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*ILLINOIS|BCBS.*ILLINOIS' THEN 'Prime Therapeutics / HCSC'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*TEXAS|BCBS.*TEXAS' THEN 'Prime Therapeutics / HCSC'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*OKLAHOMA|BCBS.*OKLAHOMA' THEN 'Prime Therapeutics / HCSC'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*NEW MEXICO|BCBS.*NEW MEXICO' THEN 'Prime Therapeutics / HCSC'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*NORTH CAROLINA' THEN 'BCBS NC'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*TENNESSEE' THEN 'BCBS TN'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*MASSACHUSETTS' THEN 'BCBS MA'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*MINNESOTA' THEN 'BCBS MN'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*ARIZONA' THEN 'BCBS AZ'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*ARKANSAS' THEN 'BCBS Arkansas'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*KANSAS CITY' THEN 'BCBS Kansas City'
            WHEN UPPER(payer_account_name) RLIKE 'BLUE.*RHODE ISLAND' THEN 'BCBS RI'
            WHEN UPPER(payer_account_name) RLIKE 'HORIZON' THEN 'Horizon BCBS NJ'
            WHEN UPPER(payer_account_name) RLIKE 'CAREFIRST' THEN 'Carefirst BCBS (CVS)'
            WHEN UPPER(payer_account_name) RLIKE 'EXCELLUS' THEN 'Excellus'
            WHEN UPPER(payer_account_name) RLIKE 'PREMERA' THEN 'Premera'
            WHEN UPPER(payer_account_name) RLIKE 'REGENCE' THEN 'Regence'
            WHEN UPPER(payer_account_name) RLIKE 'FLORIDA BLUE' THEN 'Florida Blue'
            WHEN UPPER(payer_account_name) RLIKE 'HIGHMARK' THEN 'Highmark'
            ELSE payer_account_name
        END AS normalized_payer_display_name,
        MAX(pie_completed) AS pie_completed,
        MAX(account_director) AS account_director
    FROM com_edp_prd.cmpa_insights_internal_schema.payer_info
    GROUP BY 1
),
base_enriched AS (
    SELECT
        p.*,

        p360.latest_treatment_type,

        COALESCE(i.pie_completed, 'NO') AS pie_completed,
        COALESCE(i.account_director, '-') AS account_director,

        COALESCE(n.new_patient_r1m, 0) AS new_patient_r1m,
        COALESCE(n.new_patient_r3m, 0) AS new_patient_r3m,
        COALESCE(n.new_patient_r1m_avlayah_u17, 0) AS new_patient_r1m_avlayah_u17,
        COALESCE(n.new_patient_r3m_avlayah_u17, 0) AS new_patient_r3m_avlayah_u17,

        COALESCE(c.total_claims, 0) AS total_claims,
        COALESCE(c.pharmacy_total_claims, 0) AS pharmacy_total_claims,

        COALESCE(c.elaprase_pharmacy_claims, 0) AS elaprase_pharmacy_claims,
        COALESCE(c.elaprase_approved_fills, 0) AS elaprase_approved_fills,
        COALESCE(c.elaprase_rejected_fills, 0) AS elaprase_rejected_fills,
        COALESCE(c.elaprase_reversed_fills, 0) AS elaprase_reversed_fills,

        COALESCE(c.avlayah_pharmacy_claims, 0) AS avlayah_pharmacy_claims,
        COALESCE(c.avlayah_approved_fills, 0) AS avlayah_approved_fills,
        COALESCE(c.avlayah_rejected_fills, 0) AS avlayah_rejected_fills,
        COALESCE(c.avlayah_reversed_fills, 0) AS avlayah_reversed_fills,

        COALESCE(d.is_lt17, 0) AS is_lt17,

        COALESCE(d.elaprase_claims_ct, 0) AS elaprase_claims_ct,
        COALESCE(d.avlayah_claims_ct, 0) AS avlayah_claims_ct,

        COALESCE(d.elaprase_denials, 0) AS elaprase_denials,
        COALESCE(d.avlayah_denials, 0) AS avlayah_denials,

        hh.elaprase_hcp_npi,
        hh.avlayah_hcp_npi,
        hh.elaprase_hco_id,
        hh.avlayah_hco_id,

        COALESCE(d.elaprase_comm, 0) AS elaprase_comm,
        COALESCE(d.elaprase_medicare, 0) AS elaprase_medicare,
        COALESCE(d.elaprase_medicaid, 0) AS elaprase_medicaid,
        COALESCE(d.elaprase_other, 0) AS elaprase_other,

        COALESCE(d.avlayah_comm, 0) AS avlayah_comm,
        COALESCE(d.avlayah_medicare, 0) AS avlayah_medicare,
        COALESCE(d.avlayah_medicaid, 0) AS avlayah_medicaid,
        COALESCE(d.avlayah_other, 0) AS avlayah_other,

        COALESCE(d.elaprase_flag, 0) AS elaprase_flag,
        COALESCE(d.avlayah_flag, 0) AS avlayah_flag,

        COALESCE(d.elaprase_claims_lt_17, 0) AS elaprase_claims_lt_17,
        COALESCE(d.avlayah_claims_lt_17, 0) AS avlayah_claims_lt_17,

        COALESCE(d.elaprase_denial_ct_lt_17, 0) AS elaprase_denial_ct_lt_17,
        COALESCE(d.avlayah_denial_ct_lt_17, 0) AS avlayah_denial_ct_lt_17,

        COALESCE(d.ELAPRASE_AGE_LT_5_YRS, 0) AS ELAPRASE_AGE_LT_5_YRS,
        COALESCE(d.ELAPRASE_AGE_5_TO_10_YRS, 0) AS ELAPRASE_AGE_5_TO_10_YRS,
        COALESCE(d.ELAPRASE_AGE_11_TO_16_YRS, 0) AS ELAPRASE_AGE_11_TO_16_YRS,
        COALESCE(d.ELAPRASE_AGE_17NGRT_YRS, 0) AS ELAPRASE_AGE_17NGRT_YRS,

        COALESCE(d.AVLAYAH_AGE_LT_5_YRS, 0) AS AVLAYAH_AGE_LT_5_YRS,
        COALESCE(d.AVLAYAH_AGE_5_TO_10_YRS, 0) AS AVLAYAH_AGE_5_TO_10_YRS,
        COALESCE(d.AVLAYAH_AGE_11_TO_16_YRS, 0) AS AVLAYAH_AGE_11_TO_16_YRS,
        COALESCE(d.AVLAYAH_AGE_17NGRT_YRS, 0) AS AVLAYAH_AGE_17NGRT_YRS,

        a.current_age,

        CASE WHEN a.current_age < 5 THEN 1 ELSE 0 END AS age_lt_5_yrs,
        CASE WHEN a.current_age BETWEEN 5 AND 10 THEN 1 ELSE 0 END AS age_5_to_10_yrs,
        CASE WHEN a.current_age BETWEEN 11 AND 16 THEN 1 ELSE 0 END AS AGE_11_TO_16_YRS,
        CASE WHEN a.current_age >= 17 THEN 1 ELSE 0 END AS age_17ngrt_yrs

    FROM payer_group_map p

    LEFT JOIN new_patient_flags n USING (patient_id)
    LEFT JOIN patient_claim_metrics c USING (patient_id)
    LEFT JOIN patient_age a USING (patient_id)
    LEFT JOIN patient_drug_metrics d USING (patient_id)
    LEFT JOIN patient_drug_hcp_hco hh USING (patient_id)

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p360
        ON p.patient_id = p360.patient_id

    LEFT JOIN payer_info_normalized i
        ON UPPER(p.payer_display_name) = UPPER(i.normalized_payer_display_name)
),

rollup_territory_payer AS (
    SELECT
        territory_id,
        territory AS territory_name,
        CAST(payer_id AS STRING) AS payer_id,
        payer_group AS payer_name,
        payer_group,
        CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(parent_id))) AS parent_id,
        CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(parent_name))) AS parent_name,
        COUNT(DISTINCT patient_id) AS total_elaprase_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE' THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID' THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT patient_id) AS elaprase_total_patients,
        COUNT(DISTINCT CASE WHEN avlayah_flag = 1 THEN patient_id END) AS avlayah_pt_total,
        SUM(total_claims) AS total_claims_lt17,
        SUM(elaprase_claims_ct) AS elaprase_claims_ct,
        SUM(avlayah_claims_ct) AS avlayah_claims_ct,
        COUNT(DISTINCT elaprase_hcp_npi) AS elaprase_hcp,
        COUNT(DISTINCT avlayah_hcp_npi) AS avlayah_hcp,
        COUNT(DISTINCT elaprase_hco_id) AS elaprase_hco,
        COUNT(DISTINCT avlayah_hco_id) AS avlayah_hco,
        SUM(elaprase_comm) AS elaprase_commercial_pt_ct,
        SUM(elaprase_medicare) AS elaprase_medicare_pt_ct,
        SUM(elaprase_medicaid) AS elaprase_medicaid_pt_ct,
        SUM(elaprase_other) AS elaprase_other_pt_ct,
        SUM(avlayah_comm) AS avlayah_commercial_pt_ct,
        SUM(avlayah_medicare) AS avlayah_medicare_pt_ct,
        SUM(avlayah_medicaid) AS avlayah_medicaid_pt_ct,
        SUM(avlayah_other) AS avlayah_other_pt_ct,
        SUM(elaprase_denials) AS elaprase_denials,
        SUM(avlayah_denials) AS avlayah_denials,
        SUM(elaprase_claims_lt_17) AS elaprase_claims_lt_17,
        SUM(avlayah_claims_lt_17) AS avlayah_claims_lt_17,
        SUM(elaprase_denial_ct_lt_17) AS elaprase_denial_ct_lt_17,
        SUM(avlayah_denial_ct_lt_17) AS avlayah_denial_ct_lt_17,
        SUM(ELAPRASE_AGE_LT_5_YRS) AS ELAPRASE_AGE_LT_5_YRS,
        SUM(ELAPRASE_AGE_5_TO_10_YRS) AS ELAPRASE_AGE_5_TO_10_YRS,
        SUM(ELAPRASE_AGE_11_TO_16_YRS) AS ELAPRASE_AGE_11_TO_16_YRS,
        SUM(ELAPRASE_AGE_17NGRT_YRS) AS ELAPRASE_AGE_17NGRT_YRS,
        SUM(AVLAYAH_AGE_LT_5_YRS) AS AVLAYAH_AGE_LT_5_YRS,
        SUM(AVLAYAH_AGE_5_TO_10_YRS) AS AVLAYAH_AGE_5_TO_10_YRS,
        SUM(AVLAYAH_AGE_11_TO_16_YRS) AS AVLAYAH_AGE_11_TO_16_YRS,
        SUM(AVLAYAH_AGE_17NGRT_YRS) AS AVLAYAH_AGE_17NGRT_YRS,
        COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
            OR insurance_group IS NULL THEN patient_id END) AS other_patients,
        SUM(new_patient_r1m) AS new_elaprase_patients_r1m,
        SUM(new_patient_r3m) AS new_elaprase_patients_r3m,
        SUM(new_patient_r1m_avlayah_u17) AS new_avlayah_u17_patients_r1m,
        SUM(new_patient_r3m_avlayah_u17) AS new_avlayah_u17_patients_r3m,
        COUNT(DISTINCT CASE WHEN age_lt_5_yrs = 1 THEN patient_id END) AS age_lt_5_yrs,
        COUNT(DISTINCT CASE WHEN age_5_to_10_yrs = 1 THEN patient_id END) AS age_5_to_10_yrs,
        COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS = 1 THEN patient_id END) AS AGE_11_TO_16_YRS,
        COUNT(DISTINCT CASE WHEN age_17ngrt_yrs = 1 THEN patient_id END) AS age_17ngrt_yrs,
        SUM(COALESCE(avlayah_pt_lt_17, 0)) AS avlayah_pt_lt_17,
        SUM(COALESCE(elaprase_pt_lt_17, 0)) AS elaprase_pt_lt_17,
        COUNT(DISTINCT hcp_npi) AS total_primary_hcps,
        COUNT(DISTINCT hco_veeva_crm_id) AS total_primary_hcos,
        SUM(total_claims) AS total_claims,
        SUM(pharmacy_total_claims) AS pharmacy_total_claims,
        SUM(elaprase_pharmacy_claims) AS elaprase_pharmacy_claims,
        SUM(elaprase_approved_fills) AS elaprase_approved_fills,
        SUM(elaprase_rejected_fills) AS elaprase_rejected_fills,
        SUM(elaprase_reversed_fills) AS elaprase_reversed_fills,
        SUM(avlayah_pharmacy_claims) AS avlayah_pharmacy_claims,
        SUM(avlayah_approved_fills) AS avlayah_approved_fills,
        SUM(avlayah_rejected_fills) AS avlayah_rejected_fills,
        SUM(avlayah_reversed_fills) AS avlayah_reversed_fills,
        MAX(pie_completed) AS pie_completed,
        MAX(account_director) AS account_director,
        'TERRITORY_PAYER' AS rollup_level
    FROM base_enriched
    GROUP BY territory_id, territory, payer_id, payer_group
),
rollup_payer_all_territory AS (
    SELECT
        'ALL Territories' AS territory_id,
        'All Territories' AS territory_name,
        CAST(payer_id AS STRING) AS payer_id,
        payer_group AS payer_name,
        payer_group,
        'ALL Parents' AS parent_id,
        'All Parents' AS parent_name,
        COUNT(DISTINCT patient_id) AS total_elaprase_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE' THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID' THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT patient_id) AS elaprase_total_patients,
        COUNT(DISTINCT CASE WHEN avlayah_flag = 1 THEN patient_id END) AS avlayah_pt_total,
        SUM(total_claims) AS total_claims_lt17,
        SUM(elaprase_claims_ct) AS elaprase_claims_ct,
        SUM(avlayah_claims_ct) AS avlayah_claims_ct,
        COUNT(DISTINCT elaprase_hcp_npi) AS elaprase_hcp,
        COUNT(DISTINCT avlayah_hcp_npi) AS avlayah_hcp,
        COUNT(DISTINCT elaprase_hco_id) AS elaprase_hco,
        COUNT(DISTINCT avlayah_hco_id) AS avlayah_hco,
        SUM(elaprase_comm) AS elaprase_commercial_pt_ct,
        SUM(elaprase_medicare) AS elaprase_medicare_pt_ct,
        SUM(elaprase_medicaid) AS elaprase_medicaid_pt_ct,
        SUM(elaprase_other) AS elaprase_other_pt_ct,
        SUM(avlayah_comm) AS avlayah_commercial_pt_ct,
        SUM(avlayah_medicare) AS avlayah_medicare_pt_ct,
        SUM(avlayah_medicaid) AS avlayah_medicaid_pt_ct,
        SUM(avlayah_other) AS avlayah_other_pt_ct,
        SUM(elaprase_denials) AS elaprase_denials,
        SUM(avlayah_denials) AS avlayah_denials,
        SUM(elaprase_claims_lt_17) AS elaprase_claims_lt_17,
        SUM(avlayah_claims_lt_17) AS avlayah_claims_lt_17,
        SUM(elaprase_denial_ct_lt_17) AS elaprase_denial_ct_lt_17,
        SUM(avlayah_denial_ct_lt_17) AS avlayah_denial_ct_lt_17,
        SUM(ELAPRASE_AGE_LT_5_YRS) AS ELAPRASE_AGE_LT_5_YRS,
        SUM(ELAPRASE_AGE_5_TO_10_YRS) AS ELAPRASE_AGE_5_TO_10_YRS,
        SUM(ELAPRASE_AGE_11_TO_16_YRS) AS ELAPRASE_AGE_11_TO_16_YRS,
        SUM(ELAPRASE_AGE_17NGRT_YRS) AS ELAPRASE_AGE_17NGRT_YRS,
        SUM(AVLAYAH_AGE_LT_5_YRS) AS AVLAYAH_AGE_LT_5_YRS,
        SUM(AVLAYAH_AGE_5_TO_10_YRS) AS AVLAYAH_AGE_5_TO_10_YRS,
        SUM(AVLAYAH_AGE_11_TO_16_YRS) AS AVLAYAH_AGE_11_TO_16_YRS,
        SUM(AVLAYAH_AGE_17NGRT_YRS) AS AVLAYAH_AGE_17NGRT_YRS,
        COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
            OR insurance_group IS NULL THEN patient_id END) AS other_patients,
        SUM(new_patient_r1m) AS new_elaprase_patients_r1m,
        SUM(new_patient_r3m) AS new_elaprase_patients_r3m,
        SUM(new_patient_r1m_avlayah_u17) AS new_avlayah_u17_patients_r1m,
        SUM(new_patient_r3m_avlayah_u17) AS new_avlayah_u17_patients_r3m,
        COUNT(DISTINCT CASE WHEN age_lt_5_yrs = 1 THEN patient_id END) AS age_lt_5_yrs,
        COUNT(DISTINCT CASE WHEN age_5_to_10_yrs = 1 THEN patient_id END) AS age_5_to_10_yrs,
        COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS = 1 THEN patient_id END) AS AGE_11_TO_16_YRS,
        COUNT(DISTINCT CASE WHEN age_17ngrt_yrs = 1 THEN patient_id END) AS age_17ngrt_yrs,
        SUM(COALESCE(avlayah_pt_lt_17, 0)) AS avlayah_pt_lt_17,
        SUM(COALESCE(elaprase_pt_lt_17, 0)) AS elaprase_pt_lt_17,
        COUNT(DISTINCT hcp_npi) AS total_primary_hcps,
        COUNT(DISTINCT hco_veeva_crm_id) AS total_primary_hcos,
        SUM(total_claims) AS total_claims,
        SUM(pharmacy_total_claims) AS pharmacy_total_claims,
        SUM(elaprase_pharmacy_claims) AS elaprase_pharmacy_claims,
        SUM(elaprase_approved_fills) AS elaprase_approved_fills,
        SUM(elaprase_rejected_fills) AS elaprase_rejected_fills,
        SUM(elaprase_reversed_fills) AS elaprase_reversed_fills,
        SUM(avlayah_pharmacy_claims) AS avlayah_pharmacy_claims,
        SUM(avlayah_approved_fills) AS avlayah_approved_fills,
        SUM(avlayah_rejected_fills) AS avlayah_rejected_fills,
        SUM(avlayah_reversed_fills) AS avlayah_reversed_fills,
        MAX(pie_completed) AS pie_completed,
        MAX(account_director) AS account_director,
        'PAYER_ALL_TERRITORY' AS rollup_level
    FROM base_enriched
    GROUP BY payer_id, payer_group
),
rollup_territory_all_payer AS (
    SELECT
        territory_id,
        territory AS territory_name,
        'ALL Payers' AS payer_id,
        'All Payers' AS payer_name,
        'All Payers' AS payer_group,
        'ALL Parents' AS parent_id,
        'All Parents' AS parent_name,
        COUNT(DISTINCT patient_id) AS total_elaprase_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE' THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID' THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT patient_id) AS elaprase_total_patients,
        COUNT(DISTINCT CASE WHEN avlayah_flag = 1 THEN patient_id END) AS avlayah_pt_total,
        SUM(total_claims) AS total_claims_lt17,
        SUM(elaprase_claims_ct) AS elaprase_claims_ct,
        SUM(avlayah_claims_ct) AS avlayah_claims_ct,
        COUNT(DISTINCT elaprase_hcp_npi) AS elaprase_hcp,
        COUNT(DISTINCT avlayah_hcp_npi) AS avlayah_hcp,
        COUNT(DISTINCT elaprase_hco_id) AS elaprase_hco,
        COUNT(DISTINCT avlayah_hco_id) AS avlayah_hco,
        SUM(elaprase_comm) AS elaprase_commercial_pt_ct,
        SUM(elaprase_medicare) AS elaprase_medicare_pt_ct,
        SUM(elaprase_medicaid) AS elaprase_medicaid_pt_ct,
        SUM(elaprase_other) AS elaprase_other_pt_ct,
        SUM(avlayah_comm) AS avlayah_commercial_pt_ct,
        SUM(avlayah_medicare) AS avlayah_medicare_pt_ct,
        SUM(avlayah_medicaid) AS avlayah_medicaid_pt_ct,
        SUM(avlayah_other) AS avlayah_other_pt_ct,
        SUM(elaprase_denials) AS elaprase_denials,
        SUM(avlayah_denials) AS avlayah_denials,
        SUM(elaprase_claims_lt_17) AS elaprase_claims_lt_17,
        SUM(avlayah_claims_lt_17) AS avlayah_claims_lt_17,
        SUM(elaprase_denial_ct_lt_17) AS elaprase_denial_ct_lt_17,
        SUM(avlayah_denial_ct_lt_17) AS avlayah_denial_ct_lt_17,
        SUM(ELAPRASE_AGE_LT_5_YRS) AS ELAPRASE_AGE_LT_5_YRS,
        SUM(ELAPRASE_AGE_5_TO_10_YRS) AS ELAPRASE_AGE_5_TO_10_YRS,
        SUM(ELAPRASE_AGE_11_TO_16_YRS) AS ELAPRASE_AGE_11_TO_16_YRS,
        SUM(ELAPRASE_AGE_17NGRT_YRS) AS ELAPRASE_AGE_17NGRT_YRS,
        SUM(AVLAYAH_AGE_LT_5_YRS) AS AVLAYAH_AGE_LT_5_YRS,
        SUM(AVLAYAH_AGE_5_TO_10_YRS) AS AVLAYAH_AGE_5_TO_10_YRS,
        SUM(AVLAYAH_AGE_11_TO_16_YRS) AS AVLAYAH_AGE_11_TO_16_YRS,
        SUM(AVLAYAH_AGE_17NGRT_YRS) AS AVLAYAH_AGE_17NGRT_YRS,
        COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
            OR insurance_group IS NULL THEN patient_id END) AS other_patients,
        SUM(new_patient_r1m) AS new_elaprase_patients_r1m,
        SUM(new_patient_r3m) AS new_elaprase_patients_r3m,
        SUM(new_patient_r1m_avlayah_u17) AS new_avlayah_u17_patients_r1m,
        SUM(new_patient_r3m_avlayah_u17) AS new_avlayah_u17_patients_r3m,
        COUNT(DISTINCT CASE WHEN age_lt_5_yrs = 1 THEN patient_id END) AS age_lt_5_yrs,
        COUNT(DISTINCT CASE WHEN age_5_to_10_yrs = 1 THEN patient_id END) AS age_5_to_10_yrs,
        COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS = 1 THEN patient_id END) AS AGE_11_TO_16_YRS,
        COUNT(DISTINCT CASE WHEN age_17ngrt_yrs = 1 THEN patient_id END) AS age_17ngrt_yrs,
        SUM(COALESCE(avlayah_pt_lt_17, 0)) AS avlayah_pt_lt_17,
        SUM(COALESCE(elaprase_pt_lt_17, 0)) AS elaprase_pt_lt_17,
        COUNT(DISTINCT hcp_npi) AS total_primary_hcps,
        COUNT(DISTINCT hco_veeva_crm_id) AS total_primary_hcos,
        SUM(total_claims) AS total_claims,
        SUM(pharmacy_total_claims) AS pharmacy_total_claims,
        SUM(elaprase_pharmacy_claims) AS elaprase_pharmacy_claims,
        SUM(elaprase_approved_fills) AS elaprase_approved_fills,
        SUM(elaprase_rejected_fills) AS elaprase_rejected_fills,
        SUM(elaprase_reversed_fills) AS elaprase_reversed_fills,
        SUM(avlayah_pharmacy_claims) AS avlayah_pharmacy_claims,
        SUM(avlayah_approved_fills) AS avlayah_approved_fills,
        SUM(avlayah_rejected_fills) AS avlayah_rejected_fills,
        SUM(avlayah_reversed_fills) AS avlayah_reversed_fills,
        CAST(NULL AS STRING) AS pie_completed,
        CAST(NULL AS STRING) AS account_director,
        'TERRITORY_ALL_PAYER' AS rollup_level
    FROM base_enriched
    GROUP BY territory_id, territory
),
rollup_national AS (
    SELECT
        'ALL Territories' AS territory_id,
        'All Territories' AS territory_name,
        'ALL Payers' AS payer_id,
        'All Payers' AS payer_name,
        'All Payers' AS payer_group,
        'ALL Parents' AS parent_id,
        'All Parents' AS parent_name,
        COUNT(DISTINCT patient_id) AS total_elaprase_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE' THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID' THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT patient_id) AS elaprase_total_patients,
        COUNT(DISTINCT CASE WHEN avlayah_flag = 1 THEN patient_id END) AS avlayah_pt_total,
        SUM(total_claims) AS total_claims_lt17,
        SUM(elaprase_claims_ct) AS elaprase_claims_ct,
        SUM(avlayah_claims_ct) AS avlayah_claims_ct,
        COUNT(DISTINCT elaprase_hcp_npi) AS elaprase_hcp,
        COUNT(DISTINCT avlayah_hcp_npi) AS avlayah_hcp,
        COUNT(DISTINCT elaprase_hco_id) AS elaprase_hco,
        COUNT(DISTINCT avlayah_hco_id) AS avlayah_hco,
        SUM(elaprase_comm) AS elaprase_commercial_pt_ct,
        SUM(elaprase_medicare) AS elaprase_medicare_pt_ct,
        SUM(elaprase_medicaid) AS elaprase_medicaid_pt_ct,
        SUM(elaprase_other) AS elaprase_other_pt_ct,
        SUM(avlayah_comm) AS avlayah_commercial_pt_ct,
        SUM(avlayah_medicare) AS avlayah_medicare_pt_ct,
        SUM(avlayah_medicaid) AS avlayah_medicaid_pt_ct,
        SUM(avlayah_other) AS avlayah_other_pt_ct,
        SUM(elaprase_denials) AS elaprase_denials,
        SUM(avlayah_denials) AS avlayah_denials,
        SUM(elaprase_claims_lt_17) AS elaprase_claims_lt_17,
        SUM(avlayah_claims_lt_17) AS avlayah_claims_lt_17,
        SUM(elaprase_denial_ct_lt_17) AS elaprase_denial_ct_lt_17,
        SUM(avlayah_denial_ct_lt_17) AS avlayah_denial_ct_lt_17,
        SUM(ELAPRASE_AGE_LT_5_YRS) AS ELAPRASE_AGE_LT_5_YRS,
        SUM(ELAPRASE_AGE_5_TO_10_YRS) AS ELAPRASE_AGE_5_TO_10_YRS,
        SUM(ELAPRASE_AGE_11_TO_16_YRS) AS ELAPRASE_AGE_11_TO_16_YRS,
        SUM(ELAPRASE_AGE_17NGRT_YRS) AS ELAPRASE_AGE_17NGRT_YRS,
        SUM(AVLAYAH_AGE_LT_5_YRS) AS AVLAYAH_AGE_LT_5_YRS,
        SUM(AVLAYAH_AGE_5_TO_10_YRS) AS AVLAYAH_AGE_5_TO_10_YRS,
        SUM(AVLAYAH_AGE_11_TO_16_YRS) AS AVLAYAH_AGE_11_TO_16_YRS,
        SUM(AVLAYAH_AGE_17NGRT_YRS) AS AVLAYAH_AGE_17NGRT_YRS,
        COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
            OR insurance_group IS NULL THEN patient_id END) AS other_patients,
        SUM(new_patient_r1m) AS new_elaprase_patients_r1m,
        SUM(new_patient_r3m) AS new_elaprase_patients_r3m,
        SUM(new_patient_r1m_avlayah_u17) AS new_avlayah_u17_patients_r1m,
        SUM(new_patient_r3m_avlayah_u17) AS new_avlayah_u17_patients_r3m,
        COUNT(DISTINCT CASE WHEN age_lt_5_yrs = 1 THEN patient_id END) AS age_lt_5_yrs,
        COUNT(DISTINCT CASE WHEN age_5_to_10_yrs = 1 THEN patient_id END) AS age_5_to_10_yrs,
        COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS = 1 THEN patient_id END) AS AGE_11_TO_16_YRS,
        COUNT(DISTINCT CASE WHEN age_17ngrt_yrs = 1 THEN patient_id END) AS age_17ngrt_yrs,
        SUM(COALESCE(avlayah_pt_lt_17, 0)) AS avlayah_pt_lt_17,
        SUM(COALESCE(elaprase_pt_lt_17, 0)) AS elaprase_pt_lt_17,
        COUNT(DISTINCT hcp_npi) AS total_primary_hcps,
        COUNT(DISTINCT hco_veeva_crm_id) AS total_primary_hcos,
        SUM(total_claims) AS total_claims,
        SUM(pharmacy_total_claims) AS pharmacy_total_claims,
        SUM(elaprase_pharmacy_claims) AS elaprase_pharmacy_claims,
        SUM(elaprase_approved_fills) AS elaprase_approved_fills,
        SUM(elaprase_rejected_fills) AS elaprase_rejected_fills,
        SUM(elaprase_reversed_fills) AS elaprase_reversed_fills,
        SUM(avlayah_pharmacy_claims) AS avlayah_pharmacy_claims,
        SUM(avlayah_approved_fills) AS avlayah_approved_fills,
        SUM(avlayah_rejected_fills) AS avlayah_rejected_fills,
        SUM(avlayah_reversed_fills) AS avlayah_reversed_fills,
        CAST(NULL AS STRING) AS pie_completed,
        CAST(NULL AS STRING) AS account_director,
        'NATIONAL' AS rollup_level
    FROM base_enriched
),
all_rollups AS (
    SELECT * FROM rollup_territory_payer
    UNION ALL
    SELECT * FROM rollup_payer_all_territory
    UNION ALL
    SELECT * FROM rollup_territory_all_payer
    UNION ALL
    SELECT * FROM rollup_national
),
final_with_lives AS (
    SELECT
        r.*,
        COALESCE(t.total_lives, 0) AS total_lives
    FROM all_rollups r
    LEFT JOIN total_lives t
        ON r.rollup_level = t.rollup_level
       AND r.territory_id = t.territory_id
       AND r.payer_id = t.payer_id
),
final_with_share AS (
    SELECT
        f.*,
        100.0 * f.total_lives /
        NULLIF(
            CASE
                WHEN f.rollup_level = 'TERRITORY_PAYER'
                THEN SUM(f.total_lives) OVER (PARTITION BY f.rollup_level, f.territory_id)
                ELSE SUM(f.total_lives) OVER (PARTITION BY f.rollup_level)
            END
        , 0) AS payer_market_share_pct
    FROM final_with_lives f
),
ranked_payers AS (
    SELECT
        f.*,
        RANK() OVER (
            PARTITION BY
                CASE
                    WHEN rollup_level = 'TERRITORY_PAYER' THEN territory_id
                    ELSE rollup_level
                END
            ORDER BY total_elaprase_patients DESC
        ) AS payer_rank,
        ROW_NUMBER() OVER (
            PARTITION BY
                CASE
                    WHEN rollup_level = 'TERRITORY_PAYER' THEN territory_id
                    ELSE rollup_level
                END
            ORDER BY total_elaprase_patients DESC
        ) AS row_num
    FROM final_with_share f
    WHERE payer_name NOT IN ('Unknown','All Parents')
),
rank_counts AS (
    SELECT
        CASE
            WHEN rollup_level = 'TERRITORY_PAYER' THEN territory_id
            ELSE rollup_level
        END AS grp,
        COUNT(*) AS payer_count
    FROM ranked_payers
    GROUP BY 1
),
special_rows AS (
    SELECT
        f.*,
        CASE
            WHEN payer_name = 'Unknown' THEN r.payer_count
            WHEN payer_name = 'All Parents' THEN r.payer_count + 1
        END AS payer_rank,
        NULL AS row_num
    FROM final_with_share f
    JOIN rank_counts r
        ON (
            CASE
                WHEN f.rollup_level = 'TERRITORY_PAYER' THEN f.territory_id
                ELSE f.rollup_level
            END = r.grp
        )
    WHERE payer_name IN ('Unknown','All Parents')
),
final_with_rank AS (
    SELECT * FROM ranked_payers
    UNION ALL
    SELECT * FROM special_rows
)
SELECT
    territory_id,
    territory_name,
    CAST(payer_id AS STRING) AS payer_id,
    payer_name,
    parent_id,
    parent_name,
    payer_market_share_pct,
    payer_rank,
    total_lives,
    total_elaprase_patients,
    medicare_patients,
    medicaid_patients,
    commercial_patients,
    other_patients,
    new_elaprase_patients_r1m AS NEW_ELAPRASE_PATIENTS_R1M,
    new_elaprase_patients_r3m AS NEW_ELAPRASE_PATIENTS_R3M,
    new_avlayah_u17_patients_r1m AS AVLAYAH_U17_NEW_PATIENTS_R1M,
    new_avlayah_u17_patients_r3m AS AVLAYAH_U17_NEW_PATIENTS_R3M,
    age_lt_5_yrs AS AGE_LT_5_YRS,
    age_5_to_10_yrs AS AGE_5_TO_10_YRS,
    AGE_11_TO_16_YRS AS AGE_11_TO_16_YRS,
    age_17ngrt_yrs AS AGE_17NGRT_YRS,
    avlayah_pt_lt_17 AS AVLAYAH_PT_LT_17,
    elaprase_pt_lt_17 AS ELAPRASE_PT_LT_17,
    total_primary_hcps AS TOTAL_PRIMARY_HCPS,
    total_primary_hcos AS TOTAL_PRIMARY_HCOS,
    total_claims AS TOTAL_CLAIMS,
    pharmacy_total_claims AS PHARMACY_TOTAL_CLAIMS,
    elaprase_pharmacy_claims AS ELAPRASE_PHARMACY_CLAIMS,
    elaprase_approved_fills AS ELAPRASE_APPROVED_FILLS,
    elaprase_rejected_fills AS ELAPRASE_REJECTED_FILLS,
    elaprase_reversed_fills AS ELAPRASE_REVERSED_FILLS,
    avlayah_pharmacy_claims AS AVLAYAH_PHARMACY_CLAIMS,
    avlayah_approved_fills AS AVLAYAH_APPROVED_FILLS,
    avlayah_rejected_fills AS AVLAYAH_REJECTED_FILLS,
    avlayah_reversed_fills AS AVLAYAH_REVERSED_FILLS,
    elaprase_total_patients,
    avlayah_pt_total,
    total_claims_lt17,
    elaprase_claims_ct,
    avlayah_claims_ct,
    elaprase_hcp,
    avlayah_hcp,
    elaprase_hco,
    avlayah_hco,
    elaprase_commercial_pt_ct,
    elaprase_medicare_pt_ct,
    elaprase_medicaid_pt_ct,
    elaprase_other_pt_ct,
    avlayah_commercial_pt_ct,
    avlayah_medicare_pt_ct,
    avlayah_medicaid_pt_ct,
    avlayah_other_pt_ct,
    elaprase_age_lt_5_yrs,
    elaprase_age_5_to_10_yrs,
    elaprase_age_11_to_16_yrs,
    elaprase_age_17ngrt_yrs,
    avlayah_age_lt_5_yrs,
    avlayah_age_5_to_10_yrs,
    avlayah_age_11_to_16_yrs,
    avlayah_age_17ngrt_yrs,
    elaprase_denials,
    avlayah_denials,
    elaprase_denial_ct_lt_17,
    avlayah_denial_ct_lt_17,
    elaprase_claims_lt_17,
    avlayah_claims_lt_17,
    CASE WHEN elaprase_claims_lt_17 = 0 THEN 0
        ELSE ROUND(100.0 * elaprase_denial_ct_lt_17 / elaprase_claims_lt_17, 2)
    END AS ELAPRASE_DENIAL_RATE_LT_17,
    CASE WHEN avlayah_claims_lt_17 = 0 THEN 0
        ELSE ROUND(100.0 * avlayah_denial_ct_lt_17 / avlayah_claims_lt_17, 2)
    END AS AVLAYAH_DENIAL_RATE_LT_17,
    CASE WHEN elaprase_claims_ct = 0 THEN 0
        ELSE ROUND(100.0 * elaprase_denials / elaprase_claims_ct, 2)
    END AS ELAPRASE_DENIAL_RATE,
    CASE WHEN avlayah_claims_ct = 0 THEN 0
        ELSE ROUND(100.0 * avlayah_denials / avlayah_claims_ct, 2)
    END AS AVLAYAH_DENIAL_RATE,
    CASE
        WHEN elaprase_pharmacy_claims = 0 THEN 0
        ELSE ROUND(100.0 * elaprase_approved_fills / elaprase_pharmacy_claims, 2)
    END AS ELAPRASE_APPROVAL_RATE,
    CASE
        WHEN elaprase_pharmacy_claims = 0 THEN 0
        ELSE ROUND(100.0 * elaprase_rejected_fills / elaprase_pharmacy_claims, 2)
    END AS ELAPRASE_REJECTION_RATE,
    CASE
        WHEN elaprase_pharmacy_claims = 0 THEN 0
        ELSE ROUND(100.0 * elaprase_reversed_fills / elaprase_pharmacy_claims, 2)
    END AS ELAPRASE_REVERSED_RATE,
    CASE
        WHEN avlayah_pharmacy_claims = 0 THEN 0
        ELSE ROUND(100.0 * avlayah_approved_fills / avlayah_pharmacy_claims, 2)
    END AS AVLAYAH_APPROVAL_RATE,
    CASE
        WHEN avlayah_pharmacy_claims = 0 THEN 0
        ELSE ROUND(100.0 * avlayah_rejected_fills / avlayah_pharmacy_claims, 2)
    END AS AVLAYAH_REJECTION_RATE,
    CASE
        WHEN avlayah_pharmacy_claims = 0 THEN 0
        ELSE ROUND(100.0 * avlayah_reversed_fills / avlayah_pharmacy_claims, 2)
    END AS AVLAYAH_REVERSED_RATE,
    pie_completed AS PIE_COMPLETED,
    account_director AS ACCOUNT_DIRECTOR,
    rollup_level
FROM final_with_rank;


In [0]:
select * from cmpa_insights_internal_schema.payer360_master

In [0]:
with tx_classification AS (
    SELECT
        patient_id,
        claim_id,
        drug_type,
        is_denial,
        fill_date
    FROM (
        SELECT
            patient_id,
            medical_event_id AS claim_id,
            service_date AS fill_date,
            CASE
                WHEN COALESCE(ndc11, procedure_code) IN
                    ('54092070001','540920700','99601','99602','96365','96366','J1743','S9357','S9379',
                     '38206','38230','38232','38240','38241','38242','38243','38250')
                THEN 'ELAPRASE'
                WHEN COALESCE(ndc11, procedure_code) IN ('8497600101','J3490','J3590','J9999')
                    AND service_date >= '2026-03-01'
                THEN 'AVLAYAH'
            END AS drug_type,
            0 AS is_denial
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE patient_id IN (SELECT patient_id FROM cohort_universe)
        UNION ALL
        SELECT
        patient_id,
        pharmacy_event_id AS claim_id,
        fill_date,
            CASE
                WHEN ndc11 IN ('54092070001','540920700') THEN 'ELAPRASE'
                WHEN ndc11 IN ('8497600101') THEN 'AVLAYAH'
            END AS drug_type,
            CASE
                WHEN UPPER(transaction_result) = 'REJECTED' THEN 1
                ELSE 0
            END AS is_denial
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE patient_id IN (SELECT patient_id FROM cohort_universe)
    ) t
    WHERE drug_type IS NOT NULL
),
patient_age AS (
    SELECT
        p.patient_id,
        YEAR(CURRENT_DATE) - YEAR(d.patient_yob) AS CURRENT_AGE
    FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    LEFT JOIN (
        SELECT patient_id, MAX(patient_yob) AS patient_yob
        FROM com_edp_prd.com_raw.kom_patient_demographics
        GROUP BY patient_id
    ) d ON p.patient_id = d.patient_id
)
-- SELECT
--     p360.patient_id,
--     p360.latest_treatment_type,
--     pa.current_age,
--     COUNT(DISTINCT CASE
--         WHEN t.drug_type='AVLAYAH'
--         THEN t.claim_id
--     END) avlayah_claims
-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master p360
-- LEFT JOIN patient_age pa
--     ON p360.patient_id = pa.patient_id
-- LEFT JOIN tx_classification t
--     ON p360.patient_id = t.patient_id
-- WHERE UPPER(p360.latest_treatment_type)='AVLAYAH'
--   AND pa.current_age < 17
-- GROUP BY
--     p360.patient_id,
--     p360.latest_treatment_type,
--     pa.current_age
-- HAVING avlayah_claims = 0;
SELECT
    patient_id,
    drug_type,
    COUNT(DISTINCT claim_id) AS claims
FROM tx_classification
WHERE patient_id IN
(
'33PEP1TE',
'261HTDTY',
'LS4M6FMC',
'7W3FYDPQ',
'MS7PBQ8Z'
)
GROUP BY 1,2
ORDER BY 1,2;

In [0]:
SELECT *
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
WHERE patient_id IN (
'33PEP1TE',
'261HTDTY',
'LS4M6FMC',
'7W3FYDPQ',
'MS7PBQ8Z'
);

In [0]:
SELECT
    patient_id,
    SERVICE_DATE as fill_date,
    COALESCE(ndc11, procedure_code) AS code
FROM com_edp_prd.com_raw.kom_medical_events
WHERE patient_id IN (
'33PEP1TE',
'261HTDTY',
'LS4M6FMC',
'7W3FYDPQ',
'MS7PBQ8Z'
)
AND COALESCE(ndc11, procedure_code)
    IN ('8497600101','J3490','J3590','J9999')

UNION ALL

SELECT
    patient_id,
    fill_date,
    ndc11 AS code
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE patient_id IN (
'33PEP1TE',
'261HTDTY',
'LS4M6FMC',
'7W3FYDPQ',
'MS7PBQ8Z'
)
AND ndc11='8497600101'
ORDER BY patient_id, fill_date;

In [0]:
WITH avlayah_claims_all AS (

SELECT
    patient_id,
    MIN(fill_date) first_avlayah_claim,
    MAX(fill_date) last_avlayah_claim,
    COUNT(*) claim_cnt
FROM (

    SELECT
        patient_id,
        service_date AS fill_date
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE COALESCE(ndc11, procedure_code)
          IN ('8497600101','J3490','J3590','J9999')

    UNION ALL

    SELECT
        patient_id,
        fill_date
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE ndc11='8497600101'

)
GROUP BY patient_id

)

SELECT
    p.patient_id,
    p.latest_treatment_type,
    a.first_avlayah_claim,
    a.last_avlayah_claim,
    a.claim_cnt
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master p
LEFT JOIN avlayah_claims_all a
    ON p.patient_id = a.patient_id
WHERE p.patient_id IN (
'33PEP1TE',
'261HTDTY',
'LS4M6FMC',
'7W3FYDPQ',
'MS7PBQ8Z'
);

In [0]:
SELECT
    patient_id,
    latest_treatment_type,
    latest_treatment_date
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
WHERE patient_id IN (
'261HTDTY',
'LS4M6FMC',
'MS7PBQ8Z'
);

In [0]:
SELECT
    patient_id,
    MAX(service_date) latest_jcode_date
FROM com_edp_prd.com_raw.kom_medical_events
WHERE patient_id IN
(
'33PEP1TE',
'261HTDTY',
'LS4M6FMC',
'7W3FYDPQ',
'MS7PBQ8Z'
)
AND procedure_code IN ('J3490','J3590','J9999')
GROUP BY 1;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer360_master

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim AS

-- WITH base AS (

-- SELECT DISTINCT
--     CAST(payer_id AS STRING) AS payer_id,
--     PAYER_NAME,

-- CASE

-- WHEN UPPER(PAYER_NAME) RLIKE 'UNITED|OPTUM'
--     THEN 'UHC/Optum'

-- WHEN UPPER(PAYER_NAME) RLIKE 'AETNA|CVS'
--     THEN 'Aetna/CVS'

-- WHEN UPPER(PAYER_NAME) RLIKE 'CIGNA|ESI|EVERNORTH'
--     THEN 'Cigna/ESI'

-- WHEN UPPER(PAYER_NAME) RLIKE 'ANTHEM|ELEVANCE|CARELON'
--     THEN 'Elevance/Carelon'

-- WHEN UPPER(PAYER_NAME) RLIKE 'ILLINOIS|TEXAS|OKLAHOMA|NEW MEXICO'
--     THEN 'Prime Therapeutics / HCSC'

-- ELSE PAYER_NAME

-- END AS payer_display_name

-- FROM com_edp_prd.com_raw.kom_plans
-- ),

-- rollup_ids AS (

-- SELECT
--     payer_display_name,
--     DENSE_RANK() OVER (ORDER BY payer_display_name) + 1000 AS payer_display_id
-- FROM (
--     SELECT DISTINCT payer_display_name
--     FROM base
-- )
-- )

-- SELECT
--     b.PAYER_ID AS source_payer_id,
--     b.PAYER_NAME AS source_payer_name,
--     r.payer_display_id,
--     r.payer_display_name,
--     r.payer_display_id AS canonical_payer_id,
--     r.payer_display_name AS canonical_payer_name
-- FROM base b
-- LEFT JOIN rollup_ids r
-- ON b.payer_display_name = r.payer_display_name;

In [0]:
-- /* 
-- PURPOSE
-- - Create a temporary view (total_lives) that calculates distinct patient lives 
--   across Territory and Payer with subtotal and national rollups.

-- BUSINESS LOGIC
-- - Combine medical and pharmacy claims into one unified claim universe.
-- - Attribute each claim to an NPI and Plan.
-- - Map NPI → Provider ZIP → Territory.
-- - Map Plan → Payer.
-- - Count DISTINCT patients.
-- - Produce 4 rollup levels:
--     1) Territory + Payer
--     2) Payer (All Territories)
--     3) Territory (All Payers)
--     4) National Total
-- */

-- CREATE OR REPLACE TEMPORARY VIEW total_lives AS

-- WITH all_claims AS (
--   /* 
--     Combine medical and pharmacy events into a single claims dataset.
--     Each row represents a distinct patient + NPI + plan + claim combination.
--   */
--   SELECT DISTINCT * FROM (
    
--     /* Medical claims: use Rendering NPI, else Referring NPI */
--     SELECT DISTINCT 
--       PATIENT_ID,
--       COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--       KH_PLAN_ID AS plan_id,
--       MEDICAL_EVENT_ID AS claim_id
--     FROM com_edp_prd.com_raw.kom_medical_events

--     UNION ALL

--     /* Pharmacy claims: use Prescriber NPI and primary/secondary plan */
--     SELECT DISTINCT 
--       PATIENT_ID,
--       PRESCRIBER_NPI AS NPI,
--       COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
--       PHARMACY_EVENT_ID AS claim_id
--     FROM com_edp_prd.com_raw.kom_pharmacy_events

--     -- UNION

--     -- /* Medical claims using Rendering NPI */
--     -- SELECT DISTINCT 
--     --   PATIENT_ID,
--     --   RENDERING_NPI AS NPI,
--     --   KH_PLAN_ID AS plan_id,
--     --   MEDICAL_EVENT_ID AS claim_id
--     -- FROM com_edp_prd.com_raw.kom_medical_events
--   )
-- ),

-- -- tagging_zip AS (
-- --   /* Attach provider ZIP based on NPI */
-- --   SELECT 
-- --     a.*, 
-- --     b.PROVIDER_ZIP AS hcp_zip
-- --   FROM all_claims a
-- --   LEFT JOIN com_raw.kom_providers b
-- --     ON a.npi = b.NPI 
-- --    AND b.PROVIDER_TYPE = 'INDIVIDUAL'
-- -- ),

-- -- tagging_territory AS (
-- --   /* Map provider ZIP to territory */
-- --   SELECT 
-- --     a.*, 
-- --     COALESCE(CAST(b.territory_id AS STRING), 'Unknown') AS territory_id,
-- --     COALESCE(b.territory_name, 'Unknown') AS territory
-- --   FROM tagging_zip a
-- --   LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping b
-- --     ON TRY_CAST(a.hcp_zip AS STRING) = TRY_CAST(b.zipcode AS STRING)
-- -- ),

-- tagging_zip_v1 AS (
--   SELECT 
--     a.*,
--     LPAD(COALESCE(
--       NULLIF(b.hco_zip, '-'), 
--       NULLIF(b.hcp_zip, '-'), 
--       c.PROVIDER_ZIP
--     ), 5, '0') AS final_zip
--   FROM all_claims AS a
--   LEFT JOIN cmpa_insights_internal_schema.reference_file AS b ON a.npi = b.hcp_npi
--   LEFT JOIN com_raw.kom_providers AS c ON a.npi = c.npi AND c.PROVIDER_TYPE = 'INDIVIDUAL'
-- ),

-- tagging_territory AS (
--   /* Map provider ZIP to territory */
--   SELECT 
--     a.*, 
--     COALESCE(CAST(b.territory_id AS STRING), 'Unknown') AS territory_id,
--     COALESCE(b.territory_name, 'Unknown') AS territory
--   FROM tagging_zip_v1 a
--   LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping b
--     ON a.final_zip = LPAD(TRY_CAST(b.zipcode AS STRING), 5, '0')
-- ),

-- tagging_payer AS (
--   SELECT 
--     a.*,
--     COALESCE(CAST(r.canonical_payer_id AS STRING), CAST(b.PAYER_ID AS STRING)) AS PAYER_ID,
--     COALESCE(r.canonical_payer_name, b.PAYER_NAME) AS PAYER_NAME
--   FROM tagging_territory a
--   LEFT JOIN com_raw.kom_plans b
--       ON a.plan_id = b.KH_PLAN_ID
--   LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
--       ON b.PAYER_ID = r.source_payer_id
-- ),

-- agg AS (
--   /* 
--     Aggregate distinct patients and create rollup levels 
--     using GROUPING SETS.
--   */
--   SELECT
--     COALESCE(territory_id, 'ALL Territories') AS territory_id,
--     COALESCE(territory, 'All Territories') AS territory,
--     COALESCE(payer_id, 'ALL Payers') AS payer_id,
--     COALESCE(payer_name, 'All Payers') AS payer_name,
--     COUNT(DISTINCT patient_id) AS total_lives,
--     CASE
--       WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 0 THEN 'TERRITORY_PAYER'
--       WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 0 THEN 'PAYER_ALL_TERRITORY'
--       WHEN GROUPING(territory_id) = 0 AND GROUPING(payer_id) = 1 THEN 'TERRITORY_ALL_PAYER'
--       WHEN GROUPING(territory_id) = 1 AND GROUPING(payer_id) = 1 THEN 'NATIONAL'
--     END AS rollup_level
--   FROM tagging_payer
--   GROUP BY GROUPING SETS (
--     (territory_id, territory, payer_id, payer_name),
--     (payer_id, payer_name),
--     (territory_id, territory),
--     ()
--   )
-- )

-- /* Final output ordered by rollup level and total lives */
-- SELECT *
-- FROM agg
-- ORDER BY rollup_level, total_lives DESC;

In [0]:
select * from total_lives;

In [0]:
-- /*
-- PURPOSE
-- - Identify eligible patients based on diagnosis and treatment criteria.
-- - Pull all diagnosis and treatment claims for those eligible patients.

--   BUSINESS LOGIC SUMMARY
--   1) Pull diagnosis claims (E761, E763).
--   2) Pull treatment claims (specific NDCs and procedure codes).
--   3) Identify:
--     - E761 patients with ≥2 diagnosis dates + at least one treatment → specified_patients.
--     - E763 patients with ≥2 diagnosis dates + Elaprase treatment,
--       excluding already specified patients → incremental_patients.
--   4) Eligible patients = specified + incremental.
--   5) Return all diagnosis and treatment claims for eligible patients.
--   */


-- /* ============================================================
--    1) ALL DIAGNOSIS CLAIMS (E761 / E763)
--    ============================================================ */
-- CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS
-- /* Medical diagnosis claims */
-- SELECT DISTINCT
--     PATIENT_ID,
--     COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,  -- Provider attribution
--     SERVICE_DATE AS FILL_DATE,
--     MEDICAL_EVENT_ID AS claim_id,
--     KH_PLAN_ID AS plan_id,
--     'MEDICAL' AS CLAIM_SOURCE,
--     'MEDICAL' AS TRANSACTION_STATUS
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- /* Pharmacy diagnosis claims (paid only) */
-- SELECT DISTINCT
--     PATIENT_ID,
--     PRESCRIBER_NPI AS NPI,
--     FILL_DATE,
--     PHARMACY_EVENT_ID AS claim_id,
--     COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
--     'PHARMACY' AS CLAIM_SOURCE,
--     TRANSACTION_RESULT AS TRANSACTION_STATUS
-- FROM com_edp_prd.com_raw.kom_pharmacy_events
-- WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
--   AND TRANSACTION_STATUS = 'PAID'
--   AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);



-- /* ============================================================
--    2) ALL TREATMENT CLAIMS
--    ============================================================ */
-- CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- /* Medical NDC-based treatment */
-- SELECT DISTINCT
--     PATIENT_ID,
--     COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     MEDICAL_EVENT_ID AS claim_id,
--     NDC11 AS CODE,
--     KH_PLAN_ID AS plan_id,
--     'MEDICAL' AS CLAIM_SOURCE,
--     'MEDICAL' AS TRANSACTION_STATUS
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE NDC11 IN ('54092070001', '540920700','8497600101')
--   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- /* Medical procedure-based treatment */
-- SELECT DISTINCT
--     PATIENT_ID,
--     RENDERING_NPI AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     MEDICAL_EVENT_ID AS claim_id,
--     PROCEDURE_CODE AS CODE,
--     KH_PLAN_ID AS plan_id,
--     'MEDICAL' AS CLAIM_SOURCE,
--     'MEDICAL' AS TRANSACTION_STATUS
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
--                          'S9357', 'S9379', '38206', '38230', '38232',
--                          '38240', '38241', '38242', '38243', '38250')
--   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- /* Pharmacy treatment (paid only) */
-- SELECT DISTINCT
--     PATIENT_ID,
--     PRESCRIBER_NPI AS NPI,
--     FILL_DATE,
--     PHARMACY_EVENT_ID AS claim_id,
--     NDC11 AS CODE,
--     COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
--     'PHARMACY' AS CLAIM_SOURCE,
--     TRANSACTION_RESULT AS TRANSACTION_STATUS
-- FROM com_edp_prd.com_raw.kom_pharmacy_events
-- WHERE NDC11 IN ('54092070001', '540920700','8497600101')
--   AND TRANSACTION_RESULT = 'PAID'
--   AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- /* Medical procedure-based treatment */
-- SELECT DISTINCT
--     PATIENT_ID,
--     RENDERING_NPI AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     MEDICAL_EVENT_ID AS claim_id,
--     PROCEDURE_CODE AS CODE,
--     KH_PLAN_ID AS plan_id,
--     'MEDICAL' AS CLAIM_SOURCE,
--     'MEDICAL' AS TRANSACTION_STATUS
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE PROCEDURE_CODE IN ('J3490','J3590','J9999')
--   AND SERVICE_DATE BETWEEN '2026-03-01' AND (SELECT end_date FROM runtime_parameters);



-- /* ============================================================
--    3) E761 PATIENTS WITH ≥2 DIAGNOSIS DATES
--    ============================================================ */
-- CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
-- SELECT PATIENT_ID
-- FROM (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E761%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

--     UNION

--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E761'
--       AND TRANSACTION_STATUS = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- )
-- GROUP BY PATIENT_ID
-- HAVING COUNT(DISTINCT FILL_DATE) >= 2;



-- /* ============================================================
--    4) SPECIFIED PATIENTS
--    - E761 with ≥2 dx
--    - AND at least one treatment claim
--    ============================================================ */
-- CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
-- SELECT DISTINCT e.PATIENT_ID
-- FROM e761_patients_2dx e
-- INNER JOIN all_tx_claims t 
--     ON e.PATIENT_ID = t.PATIENT_ID;



-- /* ============================================================
--    5) E763 PATIENTS WITH ≥2 DIAGNOSIS DATES
--    ============================================================ */
-- CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
-- SELECT PATIENT_ID
-- FROM (
--     SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE DIAGNOSIS_CODES LIKE '%E763%'
--       AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

--     UNION

--     SELECT DISTINCT PATIENT_ID, FILL_DATE
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE DIAGNOSIS_CODE = 'E763'
--       AND TRANSACTION_STATUS = 'PAID'
--       AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
-- )
-- GROUP BY PATIENT_ID
-- HAVING COUNT(DISTINCT FILL_DATE) >= 2;



-- /* ============================================================
--    6) ELAPRASE TREATMENT PATIENTS
--    ============================================================ */
-- CREATE OR REPLACE TEMPORARY VIEW elaprase_tx AS
-- SELECT DISTINCT PATIENT_ID
-- FROM all_tx_claims
-- WHERE CODE IN ('54092070001', '540920700', 'J1743');



-- /* ============================================================
--    7) INCREMENTAL PATIENTS
--    - E763 with ≥2 dx
--    - AND Elaprase treatment
--    - NOT already in specified_patients
--    ============================================================ */
-- CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
-- SELECT DISTINCT e.PATIENT_ID
-- FROM e763_patients_2dx e
-- INNER JOIN elaprase_tx t 
--     ON e.PATIENT_ID = t.PATIENT_ID
-- WHERE e.PATIENT_ID NOT IN (
--     SELECT PATIENT_ID FROM specified_patients
-- );



-- /* ============================================================
--    8) ELIGIBLE PATIENTS
--    ============================================================ */
-- CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
-- SELECT PATIENT_ID FROM specified_patients
-- UNION
-- SELECT PATIENT_ID FROM incremental_patients;



-- /* ============================================================
--    9) ALL CLAIMS FOR ELIGIBLE PATIENTS
--    ============================================================ */
-- CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

-- /* Diagnosis claims */
-- SELECT DISTINCT
--     PATIENT_ID,
--     NPI,
--     FILL_DATE,
--     claim_id,
--     plan_id,
--     CLAIM_SOURCE,
--     TRANSACTION_STATUS
-- FROM all_dx_claims
-- WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

-- UNION

-- /* Treatment claims */
-- SELECT DISTINCT
--     PATIENT_ID,
--     NPI,
--     FILL_DATE,
--     claim_id,
--     plan_id,
--     CLAIM_SOURCE,
--     TRANSACTION_STATUS
-- FROM all_tx_claims
-- WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);

In [0]:
-- /*
-- PURPOSE
-- - Assign a single primary HCP per eligible patient.
-- - Rank HCPs using specialty priority, visit volume, and recency.
-- - Attach HCP and associated HCO details.

-- BUSINESS LOGIC
-- 1) For each patient–NPI combination:
--    - Classify provider into a specialty bucket.
--    - Assign a specialty priority (lower value = higher priority).
--    - Count distinct visit dates (Dx + Tx combined).
--    - Capture most recent visit date.
-- 2) Rank HCPs per patient using:
--    - Specialty priority (ascending)
--    - Number of visits (descending)
--    - Most recent visit (descending)
--    - NPI (ascending tie-breaker)
-- 3) Select the top-ranked HCP (rank = 1) per patient.
-- 4) Attach HCP ZIP, HCP name, and HCO details.
-- */


-- CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS

-- /* ============================================================
--    1) Build HCP-level metrics per patient
--    ============================================================ */
-- WITH hcp_metrics AS (
--     SELECT
--         a.PATIENT_ID,
--         a.NPI,

--         /* Specialty bucket (reporting label) */
--         CASE
--             WHEN p.primary_specialty LIKE '%Genetic%'
--               OR p.secondary_specialty LIKE '%Genetic%'
--                 THEN 'Geneticist'
--             WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
--               OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
--               OR p.primary_specialty LIKE '%Neurological Surgery%'
--                 THEN 'Psychiatry & Neurology'
--             WHEN p.primary_specialty LIKE '%Pediatrics%'
--                 THEN 'Pediatrician'
--             WHEN p.primary_specialty LIKE '%Internal Medicine%'
--               OR p.secondary_specialty LIKE '%Internal Medicine%'
--               OR p.primary_specialty LIKE '%Family Medicine%'
--               OR p.secondary_specialty LIKE '%Family Medicine%'
--                 THEN 'PCP'
--             WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
--               OR p.primary_specialty LIKE '%Physician Assistant%'
--                 THEN 'NPPA'
--             WHEN a.NPI IS NULL
--                 THEN 'NA'
--             ELSE 'Others'
--         END AS SPECIALTY,

--         /* Tier 1: Specialty priority (lower = higher priority) */
--         CASE
--             WHEN p.primary_specialty LIKE '%Genetic%'
--               OR p.secondary_specialty LIKE '%Genetic%'
--                 THEN 1
--             WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
--               OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
--               OR p.primary_specialty LIKE '%Neurological Surgery%'
--                 THEN 2
--             WHEN p.primary_specialty LIKE '%Pediatrics%'
--                 THEN 3
--             WHEN p.primary_specialty LIKE '%Internal Medicine%'
--               OR p.secondary_specialty LIKE '%Internal Medicine%'
--               OR p.primary_specialty LIKE '%Family Medicine%'
--               OR p.secondary_specialty LIKE '%Family Medicine%'
--                 THEN 4
--             WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
--               OR p.primary_specialty LIKE '%Physician Assistant%'
--                 THEN 5
--             WHEN a.NPI IS NULL
--                 THEN 7
--             ELSE 6
--         END AS SPECIALTY_PRIORITY,

--         /* Tier 2: Total distinct visit dates */
--         COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

--         /* Tier 3: Most recent visit */
--         MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

--     FROM all_patient_claims a
--     LEFT JOIN com_edp_prd.com_raw.kom_providers p
--         ON a.NPI = p.NPI

--     GROUP BY
--         a.PATIENT_ID,
--         a.NPI,
--         p.primary_specialty,
--         p.secondary_specialty
-- ),

-- /* ============================================================
--    2) Rank HCPs per patient
--    ============================================================ */
-- ranked_hcps AS (
--     SELECT
--         *,
--         ROW_NUMBER() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY
--                 SPECIALTY_PRIORITY ASC,
--                 NO_OF_VISITS DESC,
--                 MOST_RECENT_VISIT DESC,
--                 NPI ASC
--         ) AS HCP_RANK
--     FROM hcp_metrics
-- ),

-- /* ============================================================
--    3) Attach HCP + HCO details for top-ranked HCP
--    ============================================================ */
-- hco_addition AS (
--     SELECT
--         a.PATIENT_ID AS patient_id,
--         a.NPI AS hcp_npi,
--         a.SPECIALTY AS hcp_specialty,

--         -- /* ZIP preference: reference file first, else provider table */
--         -- CASE 
--         --     WHEN b.hcp_zip IS NULL OR b.hcp_zip = '-' THEN c.PROVIDER_ZIP
--         --     ELSE b.hcp_zip 
--         -- END AS hcp_zip,

--         /* Name preference: reference file first, else provider table */
--         CASE
--             WHEN b.hcp_name IS NOT NULL
--                 THEN b.hcp_name
--             ELSE CONCAT(c.FIRST_NAME, " ", c.LAST_NAME)
--         END AS hcp_name,

--         b.hco_veeva_crm_id,
--         b.hco_name

--     FROM ranked_hcps a
--     LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.reference_file b
--         ON a.NPI = b.hcp_npi
--     LEFT JOIN com_raw.kom_providers c
--         ON a.NPI = c.NPI
--        AND c.PROVIDER_TYPE = 'INDIVIDUAL'

--     WHERE a.HCP_RANK = 1
-- )

-- /* ============================================================
--    Final Output: One primary HCP per patient
--    ============================================================ */
-- SELECT DISTINCT *
-- FROM hco_addition;

In [0]:
-- SELECT
--   a.patient_id,
--   a.hcp_npi,
--   a.hcp_specialty,
--   a.hcp_name,
--   a.hco_veeva_crm_id,
--   a.hco_name,

--   LPAD(COALESCE(
--     NULLIF(b.hco_zip, '-'),
--     NULLIF(b.hcp_zip, '-'),
--     c.PROVIDER_ZIP
--   ), 5, '0') AS final_zip,

--   COALESCE(CAST(t.territory_id AS STRING), 'Unknown') AS territory_id,
--   COALESCE(t.territory_name, 'Unknown')               AS territory,
--   t.region_id,
--   COALESCE(t.region_name, 'Unknown')                  AS region

-- FROM primary_hcp a

-- LEFT JOIN cmpa_insights_internal_schema.reference_file b
--   ON a.hcp_npi = b.hcp_npi

-- LEFT JOIN com_raw.kom_providers c
--   ON a.hcp_npi = c.NPI
--  AND c.PROVIDER_TYPE = 'INDIVIDUAL'

-- LEFT JOIN cmpa_insights_internal_schema.zip_to_territory_mapping t
--   ON LPAD(
--        COALESCE(NULLIF(b.hco_zip, '-'), NULLIF(b.hcp_zip, '-'), c.PROVIDER_ZIP),
--      5, '0') = LPAD(TRY_CAST(t.zipcode AS STRING), 5, '0');

In [0]:

-- SELECT
--   COUNT(*) AS total_rows,
--   COUNT(DISTINCT patient_id) AS distinct_patients
-- FROM primary_hcp;

In [0]:
-- CREATE OR REPLACE TEMP VIEW avlayah_u17_first_date AS
-- SELECT
--     t.PATIENT_ID,
--     MIN(t.FILL_DATE) AS FIRST_AVLAYAH_U17_DATE
-- FROM all_tx_claims t
-- INNER JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p
--     ON t.PATIENT_ID = p.PATIENT_ID
-- WHERE t.CODE IN ('8497600101')   -- 👈 Avlayah code (confirm if more)
--   AND p.patient_age < 17
-- GROUP BY t.PATIENT_ID;

-- CREATE OR REPLACE TEMP VIEW new_patient_flags AS
-- SELECT
--     a.PATIENT_ID,

--     /* Overall first event */
--     MIN(a.FILL_DATE) AS FIRST_EVENT_DATE,

--     /* Avlayah U17 first event */
--     u.FIRST_AVLAYAH_U17_DATE,

--     /* Existing flags */
--     CASE 
--         WHEN MIN(a.FILL_DATE) >= DATEADD(month, -1, (SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 
--     END AS NEW_PATIENT_R1M,

--     CASE 
--         WHEN MIN(a.FILL_DATE) >= DATEADD(month, -3, (SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 
--     END AS NEW_PATIENT_R3M,

--     /* ✅ Avlayah U17 flags */
--     CASE 
--         WHEN u.FIRST_AVLAYAH_U17_DATE >= DATEADD(month, -1, (SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 
--     END AS NEW_PATIENT_R1M_AVLAYAH_U17,

--     CASE 
--         WHEN u.FIRST_AVLAYAH_U17_DATE >= DATEADD(month, -3, (SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 
--     END AS NEW_PATIENT_R3M_AVLAYAH_U17

-- FROM all_patient_claims a
-- LEFT JOIN avlayah_u17_first_date u
--     ON a.PATIENT_ID = u.PATIENT_ID
-- GROUP BY a.PATIENT_ID, u.FIRST_AVLAYAH_U17_DATE;  

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level AS

-- /* ============================================================
--    1) Eligible patient universe
--    ============================================================ */
-- WITH eligible_patient_universe AS (
--   SELECT DISTINCT patient_id
--   FROM eligible_patients
-- ),

-- /* ============================================================
--    2) Count distinct claims per patient
--    ============================================================ */
-- patient_claim_counts AS (
--   SELECT 
--     patient_id, 
--     COUNT(DISTINCT claim_id) AS claims_count
--   FROM all_patient_claims
--   GROUP BY 1
-- ),

-- /* ============================================================
--    3) Attach claims count to eligible patients
--    ============================================================ */
-- eligible_patients_with_claims AS (
--   SELECT DISTINCT 
--     a.patient_id, 
--     b.claims_count
--   FROM eligible_patient_universe a 
--   LEFT JOIN patient_claim_counts b 
--     ON a.patient_id = b.patient_id
-- ),

-- /* ============================================================
--    4) Attach primary HCP details
--    ============================================================ */
-- patients_with_primary_hcp AS (
--   SELECT 
--     a.*, 
--     b.* EXCEPT (b.patient_id)
--   FROM eligible_patients_with_claims a
--   LEFT JOIN primary_hcp b 
--     ON a.patient_id = b.patient_id
-- ),

-- /* ============================================================
--    5) Enrich HCP ZIP using reference file with fallback chain:
--       reference_file.hco_zip → reference_file.hcp_zip → kom_providers ZIP
--       ('-' values in reference file treated as NULL
--    ============================================================ */
-- patients_with_final_zip AS (
--   SELECT
--     a.*,
--     LPAD(COALESCE(
--       NULLIF(b.hco_zip, '-'),
--       NULLIF(b.hcp_zip, '-'),
--       c.PROVIDER_ZIP
--     ), 5, '0') AS final_zip
--   FROM patients_with_primary_hcp a
--   LEFT JOIN cmpa_insights_internal_schema.reference_file b
--     ON a.hcp_npi = b.hcp_npi
--   LEFT JOIN com_raw.kom_providers c
--     ON a.hcp_npi = c.NPI
--    AND c.PROVIDER_TYPE = 'INDIVIDUAL'
-- ),

-- /* ============================================================
--    6) Map HCP ZIP to territory and region
--    ============================================================ */
-- patients_with_territory_region AS (
--   SELECT 
--     a.*, 
--     b.territory_id, 
--     b.territory_name AS territory, 
--     b.region_id, 
--     b.region_name AS region
--   FROM patients_with_final_zip a
--   LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping b 
--     ON a.final_zip = LPAD(TRY_CAST(b.zipcode AS STRING), 5, '0')
-- ),

-- /* ============================================================
--    7) Determine latest plan per patient
--       (Most recent claim by fill_date; tie-break by NPI)
--    ============================================================ */
-- latest_plan_per_patient AS (
--   SELECT 
--     b.patient_id, 
--     b.plan_id
--   FROM ( 
--     SELECT 
--       a.patient_id, 
--       a.plan_id, 
--       ROW_NUMBER() OVER (
--         PARTITION BY a.patient_id 
--         ORDER BY a.fill_date DESC, a.npi ASC
--       ) AS rn
--     FROM all_patient_claims a
--     WHERE a.plan_id IS NOT NULL
--   ) b
--   WHERE b.rn = 1
-- ),

-- /* ============================================================
--    8) Attach latest plan to patient record
--    ============================================================ */
-- patients_with_latest_plan AS (
--   SELECT 
--     a.*, 
--     b.plan_id
--   FROM patients_with_territory_region a
--   LEFT JOIN latest_plan_per_patient b 
--     ON a.patient_id = b.patient_id
-- ),

-- /* ============================================================
--    9) Attach payer attributes from plan table
--    ============================================================ */
-- patients_with_payer_attributes AS (
--   SELECT DISTINCT
--     a.*,
--     COALESCE(r.canonical_payer_id, b.PAYER_ID, 0) AS PAYER_ID,
--     COALESCE(r.canonical_payer_name, b.PAYER_NAME, 'Unknown') AS PAYER_NAME,
--     COALESCE(r.canonical_payer_id, b.PAYER_ID, 0) AS PARENT_ID,
--     COALESCE(r.canonical_payer_name, b.PAYER_NAME, 'Unknown') AS PARENT_NAME,
--     b.INSURANCE_SEGMENT,
--     b.INSURANCE_GROUP
--   FROM patients_with_latest_plan a
--   LEFT JOIN com_edp_prd.com_raw.kom_plans b
--     ON a.plan_id = b.KH_PLAN_ID
--   LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
--     ON b.PAYER_ID = r.source_payer_id
-- ),

-- /* ============================================================
--    10) Add Avlayah / Elaprase patient flags
--    ============================================================ */
-- patients_with_tx_flags AS (
--   SELECT
--     p.*,
--     CASE
--       WHEN p360.patient_age < 17
--        AND UPPER(p360.latest_mpsii_tx_type) = 'AVLAYAH'
--       THEN 1
--       ELSE 0
--     END AS avlayah_pt_lt_17,
--     CASE
--       WHEN p360.patient_age < 17
--        AND UPPER(p360.latest_mpsii_tx_type) IN ('ELAPRASE', 'OTHER ERT PROC')
--       THEN 1
--       ELSE 0
--     END AS elaprase_pt_lt_17
--   FROM patients_with_payer_attributes p
--   LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p360
--     ON p.patient_id = p360.patient_id
-- )

-- /* ============================================================
--    Final Patient-Level Output
--    ============================================================ */
-- SELECT
--   patient_id,
--   claims_count,

--   hcp_npi,
--   hcp_specialty,
--   final_zip,        -- rename final_zip to hcp_zip for consistency
--   hcp_name,
--   hco_veeva_crm_id,
--   hco_name,

--   COALESCE(CAST(territory_id AS STRING), 'Unknown') AS territory_id,
--   COALESCE(territory, 'Unknown') AS territory,

--   region_id,
--   COALESCE(region, 'Unknown') AS region,

--   plan_id,
--   COALESCE(PAYER_ID, 'Unknown') AS PAYER_ID,
--   COALESCE(PAYER_NAME, 'Unknown') AS PAYER_NAME,
--   COALESCE(PARENT_ID, 'Unknown') AS PARENT_ID,
--   COALESCE(PARENT_NAME, 'Unknown') AS PARENT_NAME,
--   COALESCE(INSURANCE_SEGMENT, 'Unknown') AS INSURANCE_SEGMENT,
--   COALESCE(INSURANCE_GROUP, 'Unknown') AS INSURANCE_GROUP,

--   avlayah_pt_lt_17,
--   elaprase_pt_lt_17

-- FROM patients_with_tx_flags;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level

In [0]:

-- CREATE OR REPLACE TEMP VIEW elaprase_provider_universe AS

-- WITH raw_provider_claims AS (

--     -- ===============================
--     -- MEDICAL CLAIMS (DX + NDC + PROC)
--     -- ===============================
--     SELECT DISTINCT
--         PATIENT_ID,
--         COALESCE(RENDERING_NPI, REFERRING_NPI) AS HCP_NPI,
--         BILLING_NPI AS HCO_NPI
--     FROM com_edp_prd.com_raw.kom_medical_events
--     WHERE (
--             DIAGNOSIS_CODES LIKE '%E761%'
--          OR DIAGNOSIS_CODES LIKE '%E763%'
--          OR NDC11 IN ('54092070001','540920700')
--          OR PROCEDURE_CODE IN (
--              '99601','99602','96365','96366','J1743',
--              'S9357','S9379','38206','38230','38232',
--              '38240','38241','38242','38243','38250'
--          )
--     )
--     AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

--     UNION

--     -- ===============================
--     -- PHARMACY CLAIMS
--     -- ===============================
--     SELECT DISTINCT
--         PATIENT_ID,
--         PRESCRIBER_NPI AS HCP_NPI,
--         PHARMACY_NPI AS HCO_NPI
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
--     WHERE (
--             DIAGNOSIS_CODE IN ('E761','E763')
--          OR NDC11 IN ('54092070001','540920700')
--     )
--     AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--     AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
-- )

-- -- ===============================
-- -- Attach Payer + Territory from MASTER
-- -- ===============================
-- SELECT DISTINCT
--     rpc.PATIENT_ID,
--     rpc.HCP_NPI,
--     rpc.HCO_NPI,

--     /* canonical payer from patient master */
--     COALESCE(p.PAYER_ID,0) AS payer_id,
--     COALESCE(p.PAYER_NAME,'Unknown') AS payer_name,

--     COALESCE(p.PARENT_ID,0) AS parent_id,
--     COALESCE(p.PARENT_NAME,'Unknown') AS parent_name,

--     p.territory_id,
--     p.territory,
--     p.region_id,
--     p.region

-- FROM raw_provider_claims rpc

-- JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
--     ON rpc.PATIENT_ID = p.patient_id;

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.payer360_master AS

-- /* ============================================================
-- 1. NEW PATIENT FLAGS
-- ============================================================ */
-- -- WITH new_patient_flags AS (

-- -- SELECT
-- --     patient_id,
-- --     MIN(fill_date) FIRST_EVENT_DATE,

-- --     CASE WHEN MIN(fill_date) >= DATEADD(month,-1,(SELECT end_date FROM runtime_parameters))
-- --         THEN 1 ELSE 0 END NEW_PATIENT_R1M,

-- --     CASE WHEN MIN(fill_date) >= DATEADD(month,-3,(SELECT end_date FROM runtime_parameters))
-- --         THEN 1 ELSE 0 END NEW_PATIENT_R3M

-- -- FROM all_patient_claims
-- -- GROUP BY patient_id
-- -- ),

-- /* ============================================================
-- 1. NEW PATIENT FLAGS (UPDATED WITH AVLAYAH U17)
-- ============================================================ */
-- WITH tx_classification AS (

-- SELECT 
--     patient_id,
--     claim_id,
--     drug_type,
--     is_denial,
--     fill_date 
-- FROM (

--     /* ================= MEDICAL ================= */
--     SELECT 
--         patient_id,
--         medical_event_id AS claim_id,
--         service_date AS fill_date,   -- ✅ bring it here

--         CASE 
--             WHEN COALESCE(ndc11, procedure_code) IN 
--                 ('54092070001','540920700','99601','99602','96365','96366','J1743','S9357','S9379',
--                  '38206','38230','38232','38240','38241','38242','38243','38250') 
--             THEN 'ELAPRASE'

--             WHEN COALESCE(ndc11, procedure_code) IN 
--                 ('8479600101','J3490','J3590','J9999') 
--                 AND service_date >= '2026-03-01' 
--             THEN 'AVLAYAH'
--         END AS drug_type,

--         0 AS is_denial

--     FROM com_edp_prd.com_raw.kom_medical_events


--     UNION ALL


--     /* ================= PHARMACY ================= */
--     SELECT 
--         patient_id,
--         pharmacy_event_id AS claim_id,
--         fill_date,   -- ✅ already exists here

--         CASE 
--             WHEN ndc11 IN ('54092070001','540920700') THEN 'ELAPRASE'
--             WHEN ndc11 IN ('8479600101') THEN 'AVLAYAH'
--         END AS drug_type,

--         CASE WHEN UPPER(transaction_result)='REJECTED' THEN 1 ELSE 0 END

--     FROM com_edp_prd.com_raw.kom_pharmacy_events

-- ) t

-- WHERE drug_type IS NOT NULL
-- ),

-- patient_age AS (

-- SELECT
--     p.patient_id,
--     YEAR(CURRENT_DATE)-YEAR(d.patient_yob) CURRENT_AGE

-- FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p

-- LEFT JOIN (
--     SELECT patient_id, MAX(patient_yob) patient_yob
--     FROM com_edp_prd.com_raw.kom_patient_demographics
--     GROUP BY patient_id
-- ) d
-- ON p.patient_id=d.patient_id
-- ),

-- new_patient_flags AS (

-- SELECT
--     a.patient_id,

--     /* Overall first event */
--     MIN(a.fill_date) AS first_event_date,

--     /* Avlayah U17 first event */
--     MIN(CASE 
--             WHEN t.drug_type = 'AVLAYAH' AND pa.current_age < 17
--             THEN t.fill_date 
--         END) AS first_avlayah_u17_date,

--     /* ================= OVERALL FLAGS ================= */
--     CASE 
--         WHEN MIN(a.fill_date) >= DATEADD(month,-1,(SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 
--     END AS new_patient_r1m,

--     CASE 
--         WHEN MIN(a.fill_date) >= DATEADD(month,-3,(SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 
--     END AS new_patient_r3m,

--     /* ================= AVLAYAH U17 FLAGS ================= */
--     CASE 
--         WHEN MIN(CASE 
--                     WHEN t.drug_type = 'AVLAYAH' AND pa.current_age < 17
--                     THEN t.fill_date 
--                  END)
--              >= DATEADD(month,-1,(SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 
--     END AS new_patient_r1m_avlayah_u17,

--     CASE 
--         WHEN MIN(CASE 
--                     WHEN t.drug_type = 'AVLAYAH' AND pa.current_age < 17
--                     THEN t.fill_date 
--                  END)
--              >= DATEADD(month,-3,(SELECT end_date FROM runtime_parameters))
--         THEN 1 ELSE 0 
--     END AS new_patient_r3m_avlayah_u17

-- FROM all_patient_claims a

-- LEFT JOIN tx_classification t 
--     ON a.patient_id = t.patient_id

-- LEFT JOIN patient_age pa
--     ON a.patient_id = pa.patient_id

-- GROUP BY a.patient_id
-- ),

-- /* ============================================================
-- 2. CLAIM METRICS
-- ============================================================ */
-- patient_claim_metrics AS (

-- SELECT
--     p.patient_id,

--     COUNT(DISTINCT a.claim_id) TOTAL_CLAIMS,
--     COUNT(DISTINCT ph.PHARMACY_EVENT_ID) PHARMACY_TOTAL_CLAIMS,

--     COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='PAID'
--         THEN ph.PHARMACY_EVENT_ID END) APPROVED_FILLS,

--     COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='REJECTED'
--         THEN ph.PHARMACY_EVENT_ID END) REJECTED_FILLS,

--     COUNT(DISTINCT CASE WHEN UPPER(ph.TRANSACTION_RESULT)='REVERSED'
--         THEN ph.PHARMACY_EVENT_ID END) REVERSED_FILLS

-- FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p

-- LEFT JOIN (
--     SELECT patient_id, claim_id
--     FROM all_patient_claims
-- ) a
-- ON p.patient_id=a.patient_id

-- LEFT JOIN com_edp_prd.com_raw.kom_pharmacy_events ph
--     ON p.patient_id=ph.patient_id
--    AND ph.fill_date BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
--    AND (
--         ph.diagnosis_code IN ('E761','E763')
--         OR ph.ndc11 IN ('54092070001','540920700')
--        )

-- GROUP BY p.patient_id
-- ),

-- /* ============================================================
-- 3. AGE
-- ============================================================ */


-- payer_group_map AS (

-- SELECT
--     p.*,

--     /* payer is already canonical in patient master */
--     p.payer_id AS payer_display_id,
--     p.payer_name AS payer_display_name,

--     /* group name equals payer name */
--     p.payer_name AS payer_group

-- FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
-- ),

-- patient_drug_metrics AS (

-- SELECT 
--     p.patient_id,

--     /* age flag */
--     CASE WHEN a.current_age < 17 THEN 1 ELSE 0 END AS is_lt17,

--     /* ================= PATIENT FLAGS ================= */
--     MAX(CASE WHEN t.drug_type='ELAPRASE' THEN 1 ELSE 0 END) AS elaprase_flag,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' THEN 1 ELSE 0 END) AS avlayah_flag,

--     /* ================= CLAIM COUNTS ================= */
--     COUNT(DISTINCT CASE WHEN t.drug_type='ELAPRASE' THEN t.claim_id END) AS elaprase_claims_ct,
--     COUNT(DISTINCT CASE WHEN t.drug_type='AVLAYAH' THEN t.claim_id END) AS avlayah_claims_ct,

--     /* ================= DENIALS ================= */
--     SUM(CASE WHEN t.drug_type='ELAPRASE' AND t.is_denial=1 THEN 1 ELSE 0 END) AS elaprase_denials,
--     SUM(CASE WHEN t.drug_type='AVLAYAH' AND t.is_denial=1 THEN 1 ELSE 0 END) AS avlayah_denials,

--     /* ================= PRIMARY HCP / HCO ================= */
--     count(CASE WHEN t.drug_type='ELAPRASE' THEN p.hcp_npi END) AS elaprase_hcp,
--     count(CASE WHEN t.drug_type='AVLAYAH' THEN p.hcp_npi END) AS avlayah_hcp,

--     count(CASE WHEN t.drug_type='ELAPRASE' THEN p.hco_veeva_crm_id END) AS elaprase_hco,
--     count(CASE WHEN t.drug_type='AVLAYAH' THEN p.hco_veeva_crm_id END) AS avlayah_hco,

--     /* ================= PAYER SPLIT (PATIENT LEVEL) ================= */
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND p.insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) elaprase_comm,
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND p.insurance_group='MEDICARE' THEN 1 ELSE 0 END) elaprase_medicare,
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND p.insurance_group='MEDICAID' THEN 1 ELSE 0 END) elaprase_medicaid,
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND p.insurance_group NOT IN ('COMMERCIAL','MEDICARE','MEDICAID') THEN 1 ELSE 0 END) elaprase_other,

--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND p.insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) avlayah_comm,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND p.insurance_group='MEDICARE' THEN 1 ELSE 0 END) avlayah_medicare,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND p.insurance_group='MEDICAID' THEN 1 ELSE 0 END) avlayah_medicaid,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND p.insurance_group NOT IN ('COMMERCIAL','MEDICARE','MEDICAID') THEN 1 ELSE 0 END) avlayah_other,

--     /* ================= LT 17 FLAGS ================= */
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age < 17 THEN 1 ELSE 0 END) AS elaprase_pt_lt_17,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age < 17 THEN 1 ELSE 0 END) AS avlayah_pt_lt_17,

--     /* ================= LT17 CLAIMS ================= */
--     COUNT(DISTINCT CASE 
--         WHEN t.drug_type='ELAPRASE' AND a.current_age < 17 
--         THEN t.claim_id END) AS elaprase_claims_lt_17,

--     COUNT(DISTINCT CASE 
--         WHEN t.drug_type='AVLAYAH' AND a.current_age < 17 
--         THEN t.claim_id END) AS avlayah_claims_lt_17,

--     /* ================= LT17 DENIALS ================= */
--     SUM(CASE 
--         WHEN t.drug_type='ELAPRASE' AND t.is_denial=1 AND a.current_age < 17 
--         THEN 1 ELSE 0 END) AS elaprase_denial_lt_17,

--     SUM(CASE 
--         WHEN t.drug_type='AVLAYAH' AND t.is_denial=1 AND a.current_age < 17 
--         THEN 1 ELSE 0 END) AS avlayah_denial_lt_17,

--     /* ================= ELAPRASE AGE SPLIT ================= */
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age < 5 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_LT_5_YRS,
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age BETWEEN 5 AND 10 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_5_TO_10_YRS,
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age BETWEEN 11 AND 16 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_11_TO_16_YRS,
--     MAX(CASE WHEN t.drug_type='ELAPRASE' AND a.current_age >= 17 THEN 1 ELSE 0 END) AS ELAPRASE_AGE_17NGRT_YRS,

--     /* ================= AVLAYAH AGE SPLIT ================= */
--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age < 5 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_LT_5_YRS,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age BETWEEN 5 AND 10 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_5_TO_10_YRS,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age BETWEEN 11 AND 16 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_11_TO_16_YRS,
--     MAX(CASE WHEN t.drug_type='AVLAYAH' AND a.current_age >= 17 THEN 1 ELSE 0 END) AS AVLAYAH_AGE_17NGRT_YRS

-- FROM payer_group_map p
-- LEFT JOIN tx_classification t 
--     ON p.patient_id = t.patient_id
-- LEFT JOIN patient_age a 
--     ON p.patient_id = a.patient_id

-- GROUP BY p.patient_id, a.current_age
-- ), 

-- /* ============================================================
-- 3B. DRUG METRICS (ELAPRASE vs AVLAYAH)
-- ============================================================ */

-- -- tx_classification AS (

-- -- SELECT 
-- --     patient_id,
-- --     claim_id,
    
-- --     CASE 
-- --         WHEN code IN ('54092070001','540920700','J1743') THEN 'ELAPRASE'
-- --         WHEN code = '8479600101' OR code IN ('J3490','J3590','J9999') THEN 'AVLAYAH'
-- --     END AS drug_type,

-- --     CASE WHEN UPPER(transaction_status)='REJECTED' THEN 1 ELSE 0 END AS is_denial

-- -- FROM (
-- --     SELECT 
-- --         patient_id,
-- --         medical_event_id AS claim_id,
-- --         COALESCE(ndc11, procedure_code) AS code,
-- --         'PAID' AS transaction_status
-- --     FROM com_edp_prd.com_raw.kom_medical_events

-- --     UNION ALL

-- --     SELECT 
-- --         patient_id,
-- --         pharmacy_event_id,
-- --         ndc11,
-- --         transaction_result
-- --     FROM com_edp_prd.com_raw.kom_pharmacy_events
-- -- )

-- -- WHERE code IS NOT NULL
-- -- ),

-- -- patient_drug_metrics AS (

-- -- SELECT 
-- --     p.patient_id,

-- --     /* ================= <17 ================= */
-- --     CASE WHEN a.current_age < 17 THEN 1 ELSE 0 END AS is_lt17,

-- --     MAX(CASE WHEN t.drug_type='ELAPRASE' THEN 1 ELSE 0 END) elaprase_flag,
-- --     MAX(CASE WHEN t.drug_type='AVLAYAH' THEN 1 ELSE 0 END) avlayah_flag,

-- --     COUNT(DISTINCT CASE WHEN t.drug_type='ELAPRASE' THEN t.claim_id END) elaprase_claims_ct,
-- --     COUNT(DISTINCT CASE WHEN t.drug_type='AVLAYAH' THEN t.claim_id END) avlayah_claims_ct,

-- --     SUM(CASE WHEN t.drug_type='ELAPRASE' AND t.is_denial=1 THEN 1 ELSE 0 END) elaprase_denials,
-- --     SUM(CASE WHEN t.drug_type='AVLAYAH' AND t.is_denial=1 THEN 1 ELSE 0 END) avlayah_denials,

-- --     COUNT(DISTINCT CASE WHEN t.drug_type='ELAPRASE' THEN p.hcp_npi END) elaprase_hcp,
-- --     COUNT(DISTINCT CASE WHEN t.drug_type='AVLAYAH' THEN p.hcp_npi END) avlayah_hcp,

-- --     COUNT(DISTINCT CASE WHEN t.drug_type='ELAPRASE' THEN p.hco_veeva_crm_id END) elaprase_hco,
-- --     COUNT(DISTINCT CASE WHEN t.drug_type='AVLAYAH' THEN p.hco_veeva_crm_id END) avlayah_hco,

-- --     /* payer splits */
-- --     MAX(CASE WHEN t.drug_type='ELAPRASE' AND insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) elaprase_comm,
-- --     MAX(CASE WHEN t.drug_type='ELAPRASE' AND insurance_group='MEDICARE' THEN 1 ELSE 0 END) elaprase_medicare,

-- --     MAX(CASE WHEN t.drug_type='AVLAYAH' AND insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) avlayah_comm,
-- --     MAX(CASE WHEN t.drug_type='AVLAYAH' AND insurance_group='MEDICARE' THEN 1 ELSE 0 END) avlayah_medicare

-- -- FROM payer_group_map p
-- -- LEFT JOIN tx_classification t ON p.patient_id=t.patient_id
-- -- LEFT JOIN patient_age a ON p.patient_id=a.patient_id

-- -- GROUP BY p.patient_id, a.current_age
-- -- ),

-- /* ============================================================
-- 4. DISPLAY PAYER ROLLUP
-- ============================================================ */


-- /* ============================================================
-- 5. PAYER INFO NORMALIZATION (SAFE BCBS MATCH)
-- ============================================================ */
-- payer_info_normalized AS (

-- SELECT

-- CASE

-- /* =========================
-- UHC / OPTUM
-- ========================= */
-- WHEN UPPER(payer_account_name) RLIKE 'UNITED|OPTUM'
-- THEN 'UHC/Optum'

-- /* =========================
-- AETNA / CVS
-- ========================= */
-- WHEN UPPER(payer_account_name) RLIKE 'AETNA|CVS'
-- THEN 'Aetna/CVS'

-- /* =========================
-- CIGNA / ESI
-- ========================= */
-- WHEN UPPER(payer_account_name) RLIKE 'CIGNA|ESI|EVERNORTH|EXPRESS SCRIPTS'
-- THEN 'Cigna/ESI'

-- /* =========================
-- ELEVANCE / CARELON
-- ========================= */
-- WHEN UPPER(payer_account_name) RLIKE 'ANTHEM|ELEVANCE|CARELON|AMERIGROUP|WELLPOINT|EMPIRE.*BLUE'
-- THEN 'Elevance/Carelon'

-- /* =========================
-- HCSC PLANS (BCBS + STATE)
-- ========================= */
-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*ILLINOIS|BCBS.*ILLINOIS'
-- THEN 'Prime Therapeutics / HCSC'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*TEXAS|BCBS.*TEXAS'
-- THEN 'Prime Therapeutics / HCSC'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*OKLAHOMA|BCBS.*OKLAHOMA'
-- THEN 'Prime Therapeutics / HCSC'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*NEW MEXICO|BCBS.*NEW MEXICO'
-- THEN 'Prime Therapeutics / HCSC'

-- /* =========================
-- REGIONAL BCBS
-- ========================= */
-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*NORTH CAROLINA'
-- THEN 'BCBS NC'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*TENNESSEE'
-- THEN 'BCBS TN'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*MASSACHUSETTS'
-- THEN 'BCBS MA'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*MINNESOTA'
-- THEN 'BCBS MN'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*ARIZONA'
-- THEN 'BCBS AZ'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*ARKANSAS'
-- THEN 'BCBS Arkansas'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*KANSAS CITY'
-- THEN 'BCBS Kansas City'

-- WHEN UPPER(payer_account_name) RLIKE 'BLUE.*RHODE ISLAND'
-- THEN 'BCBS RI'

-- /* =========================
-- OTHER BCBS LICENSEES
-- ========================= */
-- WHEN UPPER(payer_account_name) RLIKE 'HORIZON'
-- THEN 'Horizon BCBS NJ'

-- WHEN UPPER(payer_account_name) RLIKE 'CAREFIRST'
-- THEN 'Carefirst BCBS (CVS)'

-- WHEN UPPER(payer_account_name) RLIKE 'EXCELLUS'
-- THEN 'Excellus'

-- WHEN UPPER(payer_account_name) RLIKE 'PREMERA'
-- THEN 'Premera'

-- WHEN UPPER(payer_account_name) RLIKE 'REGENCE'
-- THEN 'Regence'

-- WHEN UPPER(payer_account_name) RLIKE 'FLORIDA BLUE'
-- THEN 'Florida Blue'

-- WHEN UPPER(payer_account_name) RLIKE 'HIGHMARK'
-- THEN 'Highmark'

-- /* =========================
-- DEFAULT
-- ========================= */
-- ELSE payer_account_name

-- END AS normalized_payer_display_name,

-- MAX(pie_completed) pie_completed,
-- MAX(account_director) account_director

-- FROM com_edp_prd.cmpa_insights_internal_schema.payer_info
-- GROUP BY 1

-- ),

-- /* ============================================================
-- 6. BASE ENRICHMENT
-- ============================================================ */
-- base_enriched AS (

-- SELECT
--     p.*,

--     COALESCE(i.pie_completed,'NO') pie_completed,
--     COALESCE(i.account_director,'-') account_director,

--     COALESCE(n.new_patient_r1m,0) new_patient_r1m,
--     COALESCE(n.new_patient_r3m,0) new_patient_r3m,

--     COALESCE(n.new_patient_r1m_avlayah_u17,0) new_patient_r1m_avlayah_u17,
--     COALESCE(n.new_patient_r3m_avlayah_u17,0) new_patient_r3m_avlayah_u17,

--     COALESCE(c.total_claims,0) total_claims,
--     COALESCE(c.pharmacy_total_claims,0) pharmacy_total_claims,
--     COALESCE(c.approved_fills,0) approved_fills,
--     COALESCE(c.rejected_fills,0) rejected_fills,
--     COALESCE(c.reversed_fills,0) reversed_fills,

--     COALESCE(d.is_lt17,0) is_lt17,

--     COALESCE(d.elaprase_claims_ct,0) elaprase_claims_ct,
--     COALESCE(d.avlayah_claims_ct,0) avlayah_claims_ct,

--     COALESCE(d.elaprase_denials,0) elaprase_denials,
--     COALESCE(d.avlayah_denials,0) avlayah_denials,

--     COALESCE(d.elaprase_hcp,0) elaprase_hcp,
--     COALESCE(d.avlayah_hcp,0) avlayah_hcp,

--     COALESCE(d.elaprase_hco,0) elaprase_hco,
--     COALESCE(d.avlayah_hco,0) avlayah_hco,

--     COALESCE(d.elaprase_comm,0) elaprase_comm,
--     COALESCE(d.elaprase_medicare,0) elaprase_medicare,
--     COALESCE(d.elaprase_medicaid,0) elaprase_medicaid,
--     COALESCE(d.elaprase_other,0) elaprase_other,

--     COALESCE(d.avlayah_comm,0) avlayah_comm,
--     COALESCE(d.avlayah_medicare,0) avlayah_medicare,
--     COALESCE(d.avlayah_medicaid,0) avlayah_medicaid,
--     COALESCE(d.avlayah_other,0) avlayah_other,

--     COALESCE(d.elaprase_flag,0) elaprase_flag,  
--     COALESCE(d.avlayah_flag,0) avlayah_flag,

--     COALESCE(d.elaprase_claims_lt_17,0) elaprase_claims_lt_17,
--     COALESCE(d.avlayah_claims_lt_17,0) avlayah_claims_lt_17,

--     COALESCE(d.elaprase_denial_lt_17,0) elaprase_denial_lt_17,
--     COALESCE(d.avlayah_denial_lt_17,0) avlayah_denial_lt_17,

--     COALESCE(d.ELAPRASE_AGE_LT_5_YRS,0) ELAPRASE_AGE_LT_5_YRS,
--     COALESCE(d.ELAPRASE_AGE_5_TO_10_YRS,0) ELAPRASE_AGE_5_TO_10_YRS,
--     COALESCE(d.ELAPRASE_AGE_11_TO_16_YRS,0) ELAPRASE_AGE_11_TO_16_YRS,
--     COALESCE(d.ELAPRASE_AGE_17NGRT_YRS,0) ELAPRASE_AGE_17NGRT_YRS,

--     COALESCE(d.AVLAYAH_AGE_LT_5_YRS,0) AVLAYAH_AGE_LT_5_YRS,
--     COALESCE(d.AVLAYAH_AGE_5_TO_10_YRS,0) AVLAYAH_AGE_5_TO_10_YRS,
--     COALESCE(d.AVLAYAH_AGE_11_TO_16_YRS,0) AVLAYAH_AGE_11_TO_16_YRS,
--     COALESCE(d.AVLAYAH_AGE_17NGRT_YRS,0) AVLAYAH_AGE_17NGRT_YRS,

--     a.current_age,

--     CASE WHEN a.current_age<5 THEN 1 ELSE 0 END age_lt_5_yrs,
--     CASE WHEN a.current_age BETWEEN 5 AND 10 THEN 1 ELSE 0 END age_5_to_10_yrs,
--     CASE WHEN a.current_age BETWEEN 11 AND 16 THEN 1 ELSE 0 END AGE_11_TO_16_YRS,
--     CASE WHEN a.current_age>=17 THEN 1 ELSE 0 END age_17ngrt_yrs

-- FROM payer_group_map p

-- LEFT JOIN new_patient_flags n USING(patient_id)
-- LEFT JOIN patient_claim_metrics c USING(patient_id)
-- LEFT JOIN patient_age a USING(patient_id)
-- LEFT JOIN patient_drug_metrics d USING(patient_id)

-- LEFT JOIN payer_info_normalized i
--     ON UPPER(p.payer_display_name)=UPPER(i.normalized_payer_display_name)
-- ),

-- /* ============================================================
-- 7. TERRITORY + PAYER
-- ============================================================ */
-- rollup_territory_payer AS (

-- SELECT


-- territory_id,
-- territory AS territory_name,
-- CAST(payer_id AS STRING) AS payer_id,
-- payer_group AS payer_name,
-- payer_group,
-- CONCAT_WS(' | ',SORT_ARRAY(COLLECT_SET(parent_id))) parent_id,
-- CONCAT_WS(' | ',SORT_ARRAY(COLLECT_SET(parent_name))) parent_name,

-- COUNT(DISTINCT patient_id) total_elaprase_patients,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

-- COUNT(DISTINCT  patient_id ) elaprase_total_patients,
-- -- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN patient_id END) elaprase_pt_total,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN patient_id END) avlayah_pt_total,
-- SUM(total_claims) total_claims_lt17,
-- SUM(elaprase_claims_ct) elaprase_claims_ct,
-- SUM(avlayah_claims_ct ) avlayah_claims_ct,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hcp_npi END) elaprase_hcp,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hcp_npi END) avlayah_hcp,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hco_veeva_crm_id END) elaprase_hco,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hco_veeva_crm_id END) avlayah_hco,

-- SUM(elaprase_comm) elaprase_commercial_pt_ct,
-- SUM(elaprase_medicare) elaprase_medicare_pt_ct,
-- SUM(elaprase_medicaid) elaprase_medicaid_pt_ct,
-- SUM(elaprase_other) elaprase_other_pt_ct,

-- SUM(avlayah_comm) avlayah_commercial_pt_ct,
-- SUM(avlayah_medicare) avlayah_medicare_pt_ct,
-- SUM(avlayah_medicaid) avlayah_medicaid_pt_ct,
-- SUM(avlayah_other) avlayah_other_pt_ct,
-- SUM(elaprase_denials) elaprase_denials,
-- SUM(avlayah_denials) avlayah_denials,

-- SUM(elaprase_claims_lt_17) elaprase_claims_lt_17,
-- SUM(avlayah_claims_lt_17) avlayah_claims_lt_17,

-- CASE 
--     WHEN SUM(elaprase_claims_lt_17) = 0 THEN 0
--     ELSE ROUND(100.0 * SUM(elaprase_denial_lt_17) / SUM(elaprase_claims_lt_17), 2)
-- END AS elaprase_denial_lt_17,

-- CASE 
--     WHEN SUM(avlayah_claims_lt_17) = 0 THEN 0
--     ELSE ROUND(100.0 * SUM(avlayah_denial_lt_17) / SUM(avlayah_claims_lt_17), 2)
-- END AS avlayah_denial_lt_17,

-- SUM(ELAPRASE_AGE_LT_5_YRS) ELAPRASE_AGE_LT_5_YRS,
-- SUM(ELAPRASE_AGE_5_TO_10_YRS) ELAPRASE_AGE_5_TO_10_YRS,
-- SUM(ELAPRASE_AGE_11_TO_16_YRS) ELAPRASE_AGE_11_TO_16_YRS,
-- SUM(ELAPRASE_AGE_17NGRT_YRS) ELAPRASE_AGE_17NGRT_YRS,

-- SUM(AVLAYAH_AGE_LT_5_YRS) AVLAYAH_AGE_LT_5_YRS,
-- SUM(AVLAYAH_AGE_5_TO_10_YRS) AVLAYAH_AGE_5_TO_10_YRS,
-- SUM(AVLAYAH_AGE_11_TO_16_YRS) AVLAYAH_AGE_11_TO_16_YRS,
-- SUM(AVLAYAH_AGE_17NGRT_YRS) AVLAYAH_AGE_17NGRT_YRS,

-- COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--     OR insurance_group IS NULL THEN patient_id END) other_patients,

-- SUM(new_patient_r1m) new_elaprase_patients_r1m,
-- SUM(new_patient_r3m) new_elaprase_patients_r3m,

-- SUM(new_patient_r1m_avlayah_u17) new_avlayah_u17_patients_r1m,
-- SUM(new_patient_r3m_avlayah_u17) new_avlayah_u17_patients_r3m,

-- COUNT(DISTINCT CASE WHEN age_lt_5_yrs=1 THEN patient_id END) age_lt_5_yrs,
-- COUNT(DISTINCT CASE WHEN age_5_to_10_yrs=1 THEN patient_id END) age_5_to_10_yrs,
-- COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS=1 THEN patient_id END) AGE_11_TO_16_YRS,
-- COUNT(DISTINCT CASE WHEN age_17ngrt_yrs=1 THEN patient_id END) age_17ngrt_yrs,

-- SUM(COALESCE(avlayah_pt_lt_17,0)) avlayah_pt_lt_17,
-- SUM(COALESCE(elaprase_pt_lt_17,0)) elaprase_pt_lt_17,

-- COUNT(DISTINCT hcp_npi) total_primary_hcps,
-- COUNT(DISTINCT hco_veeva_crm_id) total_primary_hcos,

-- SUM(total_claims) total_claims,
-- SUM(pharmacy_total_claims) pharmacy_total_claims,
-- SUM(approved_fills) approved_fills,
-- SUM(rejected_fills) rejected_fills,
-- SUM(reversed_fills) reversed_fills,

-- MAX(pie_completed) pie_completed,
-- MAX(account_director) account_director,

-- 'TERRITORY_PAYER' rollup_level

-- FROM base_enriched
-- GROUP BY territory_id,territory,payer_id,payer_group
-- ),

-- /* ============================================================
-- 8. PAYER ALL TERRITORY
-- ============================================================ */
-- rollup_payer_all_territory AS (

-- SELECT
-- 'ALL Territories' territory_id,
-- 'All Territories' territory_name,
-- CAST(payer_id AS STRING) AS payer_id,
-- payer_group payer_name,
-- payer_group,
-- 'ALL Parents' parent_id,
-- 'All Parents' parent_name,

-- COUNT(DISTINCT patient_id) total_elaprase_patients,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

-- COUNT(DISTINCT  patient_id ) elaprase_total_patients,
-- -- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN patient_id END) elaprase_pt_total,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN patient_id END) avlayah_pt_total,
-- SUM( total_claims ) total_claims_lt17,
-- SUM(elaprase_claims_ct) elaprase_claims_ct,
-- SUM(avlayah_claims_ct ) avlayah_claims_ct,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hcp_npi END) elaprase_hcp,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hcp_npi END) avlayah_hcp,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hco_veeva_crm_id END) elaprase_hco,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hco_veeva_crm_id END) avlayah_hco,

-- SUM(elaprase_comm) elaprase_commercial_pt_ct,
-- SUM(elaprase_medicare) elaprase_medicare_pt_ct,
-- SUM(elaprase_medicaid) elaprase_medicaid_pt_ct,
-- SUM(elaprase_other) elaprase_other_pt_ct,

-- SUM(avlayah_comm) avlayah_commercial_pt_ct,
-- SUM(avlayah_medicare) avlayah_medicare_pt_ct,
-- SUM(avlayah_medicaid) avlayah_medicaid_pt_ct,
-- SUM(avlayah_other) avlayah_other_pt_ct,
-- SUM(elaprase_denials) elaprase_denials,
-- SUM(avlayah_denials) avlayah_denials,

-- SUM(elaprase_claims_lt_17) elaprase_claims_lt_17,
-- SUM(avlayah_claims_lt_17) avlayah_claims_lt_17,

-- CASE 
--     WHEN SUM(elaprase_claims_lt_17) = 0 THEN 0
--     ELSE ROUND(100.0 * SUM(elaprase_denial_lt_17) / SUM(elaprase_claims_lt_17), 2)
-- END AS elaprase_denial_lt_17,

-- CASE 
--     WHEN SUM(avlayah_claims_lt_17) = 0 THEN 0
--     ELSE ROUND(100.0 * SUM(avlayah_denial_lt_17) / SUM(avlayah_claims_lt_17), 2)
-- END AS avlayah_denial_lt_17,

-- SUM(ELAPRASE_AGE_LT_5_YRS) ELAPRASE_AGE_LT_5_YRS,
-- SUM(ELAPRASE_AGE_5_TO_10_YRS) ELAPRASE_AGE_5_TO_10_YRS,
-- SUM(ELAPRASE_AGE_11_TO_16_YRS) ELAPRASE_AGE_11_TO_16_YRS,
-- SUM(ELAPRASE_AGE_17NGRT_YRS) ELAPRASE_AGE_17NGRT_YRS,

-- SUM(AVLAYAH_AGE_LT_5_YRS) AVLAYAH_AGE_LT_5_YRS,
-- SUM(AVLAYAH_AGE_5_TO_10_YRS) AVLAYAH_AGE_5_TO_10_YRS,
-- SUM(AVLAYAH_AGE_11_TO_16_YRS) AVLAYAH_AGE_11_TO_16_YRS,
-- SUM(AVLAYAH_AGE_17NGRT_YRS) AVLAYAH_AGE_17NGRT_YRS,

-- COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--     OR insurance_group IS NULL THEN patient_id END) other_patients,

-- SUM(new_patient_r1m) new_elaprase_patients_r1m,
-- SUM(new_patient_r3m) new_elaprase_patients_r3m,

-- SUM(new_patient_r1m_avlayah_u17) new_avlayah_u17_patients_r1m,
-- SUM(new_patient_r3m_avlayah_u17) new_avlayah_u17_patients_r3m,

-- COUNT(DISTINCT CASE WHEN age_lt_5_yrs=1 THEN patient_id END) age_lt_5_yrs,
-- COUNT(DISTINCT CASE WHEN age_5_to_10_yrs=1 THEN patient_id END) age_5_to_10_yrs,
-- COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS=1 THEN patient_id END) AGE_11_TO_16_YRS,
-- COUNT(DISTINCT CASE WHEN age_17ngrt_yrs=1 THEN patient_id END) age_17ngrt_yrs,

-- SUM(COALESCE(avlayah_pt_lt_17,0)) avlayah_pt_lt_17,
-- SUM(COALESCE(elaprase_pt_lt_17,0)) elaprase_pt_lt_17,

-- COUNT(DISTINCT hcp_npi) total_primary_hcps,
-- COUNT(DISTINCT hco_veeva_crm_id) total_primary_hcos,

-- SUM(total_claims) total_claims,
-- SUM(pharmacy_total_claims) pharmacy_total_claims,
-- SUM(approved_fills) approved_fills,
-- SUM(rejected_fills) rejected_fills,
-- SUM(reversed_fills) reversed_fills,

-- MAX(pie_completed) pie_completed,
-- MAX(account_director) account_director,

-- 'PAYER_ALL_TERRITORY' rollup_level

-- FROM base_enriched
-- GROUP BY payer_id,payer_group
-- ),

-- /* ============================================================
-- 9. TERRITORY ALL PAYER
-- ============================================================ */
-- rollup_territory_all_payer AS (

-- SELECT
-- territory_id,
-- territory territory_name,
-- 'ALL Payers' payer_id,
-- 'All Payers' payer_name,
-- 'All Payers' payer_group,
-- 'ALL Parents' parent_id,
-- 'All Parents' parent_name,

-- COUNT(DISTINCT patient_id) total_elaprase_patients,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

-- COUNT(DISTINCT  patient_id ) elaprase_total_patients,
-- -- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN patient_id END) elaprase_pt_total,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN patient_id END) avlayah_pt_total,
-- SUM( total_claims ) total_claims_lt17,
-- SUM(elaprase_claims_ct) elaprase_claims_ct,
-- SUM(avlayah_claims_ct ) avlayah_claims_ct,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hcp_npi END) elaprase_hcp,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hcp_npi END) avlayah_hcp,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hco_veeva_crm_id END) elaprase_hco,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hco_veeva_crm_id END) avlayah_hco,

-- SUM(elaprase_comm) elaprase_commercial_pt_ct,
-- SUM(elaprase_medicare) elaprase_medicare_pt_ct,
-- SUM(elaprase_medicaid) elaprase_medicaid_pt_ct,
-- SUM(elaprase_other) elaprase_other_pt_ct,

-- SUM(avlayah_comm) avlayah_commercial_pt_ct,
-- SUM(avlayah_medicare) avlayah_medicare_pt_ct,
-- SUM(avlayah_medicaid) avlayah_medicaid_pt_ct,
-- SUM(avlayah_other) avlayah_other_pt_ct,
-- SUM(elaprase_denials) elaprase_denials,
-- SUM(avlayah_denials) avlayah_denials,

-- SUM(elaprase_claims_lt_17) elaprase_claims_lt_17,
-- SUM(avlayah_claims_lt_17) avlayah_claims_lt_17,

-- CASE 
--     WHEN SUM(elaprase_claims_lt_17) = 0 THEN 0
--     ELSE ROUND(100.0 * SUM(elaprase_denial_lt_17) / SUM(elaprase_claims_lt_17), 2)
-- END AS elaprase_denial_lt_17,

-- CASE 
--     WHEN SUM(avlayah_claims_lt_17) = 0 THEN 0
--     ELSE ROUND(100.0 * SUM(avlayah_denial_lt_17) / SUM(avlayah_claims_lt_17), 2)
-- END AS avlayah_denial_lt_17,

-- SUM(ELAPRASE_AGE_LT_5_YRS) ELAPRASE_AGE_LT_5_YRS,
-- SUM(ELAPRASE_AGE_5_TO_10_YRS) ELAPRASE_AGE_5_TO_10_YRS,
-- SUM(ELAPRASE_AGE_11_TO_16_YRS) ELAPRASE_AGE_11_TO_16_YRS,
-- SUM(ELAPRASE_AGE_17NGRT_YRS) ELAPRASE_AGE_17NGRT_YRS,

-- SUM(AVLAYAH_AGE_LT_5_YRS) AVLAYAH_AGE_LT_5_YRS,
-- SUM(AVLAYAH_AGE_5_TO_10_YRS) AVLAYAH_AGE_5_TO_10_YRS,
-- SUM(AVLAYAH_AGE_11_TO_16_YRS) AVLAYAH_AGE_11_TO_16_YRS,
-- SUM(AVLAYAH_AGE_17NGRT_YRS) AVLAYAH_AGE_17NGRT_YRS,

-- COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--     OR insurance_group IS NULL THEN patient_id END) other_patients,

-- SUM(new_patient_r1m) new_elaprase_patients_r1m,
-- SUM(new_patient_r3m) new_elaprase_patients_r3m,

-- SUM(new_patient_r1m_avlayah_u17) new_avlayah_u17_patients_r1m,
-- SUM(new_patient_r3m_avlayah_u17) new_avlayah_u17_patients_r3m,

-- COUNT(DISTINCT CASE WHEN age_lt_5_yrs=1 THEN patient_id END) age_lt_5_yrs,
-- COUNT(DISTINCT CASE WHEN age_5_to_10_yrs=1 THEN patient_id END) age_5_to_10_yrs,
-- COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS=1 THEN patient_id END) AGE_11_TO_16_YRS,
-- COUNT(DISTINCT CASE WHEN age_17ngrt_yrs=1 THEN patient_id END) age_17ngrt_yrs,

-- SUM(COALESCE(avlayah_pt_lt_17,0)) avlayah_pt_lt_17,
-- SUM(COALESCE(elaprase_pt_lt_17,0)) elaprase_pt_lt_17,

-- COUNT(DISTINCT hcp_npi) total_primary_hcps,
-- COUNT(DISTINCT hco_veeva_crm_id) total_primary_hcos,

-- SUM(total_claims) total_claims,
-- SUM(pharmacy_total_claims) pharmacy_total_claims,
-- SUM(approved_fills) approved_fills,
-- SUM(rejected_fills) rejected_fills,
-- SUM(reversed_fills) reversed_fills,

-- CAST(NULL AS STRING) pie_completed,
-- CAST(NULL AS STRING) account_director,

-- 'TERRITORY_ALL_PAYER' rollup_level

-- FROM base_enriched
-- GROUP BY territory_id,territory
-- ),

-- /* ============================================================
-- 10. NATIONAL
-- ============================================================ */
-- rollup_national AS (

-- SELECT
-- 'ALL Territories' territory_id,
-- 'All Territories' territory_name,
-- 'ALL Payers' payer_id,
-- 'All Payers' payer_name,
-- 'All Payers' payer_group,
-- 'ALL Parents' parent_id,
-- 'All Parents' parent_name,

-- COUNT(DISTINCT patient_id) total_elaprase_patients,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

-- COUNT(DISTINCT  patient_id ) elaprase_total_patients,
-- -- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN patient_id END) elaprase_pt_total,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN patient_id END) avlayah_pt_total,
-- SUM( total_claims ) total_claims_lt17,
-- SUM(elaprase_claims_ct) elaprase_claims_ct,
-- SUM(avlayah_claims_ct ) avlayah_claims_ct,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hcp_npi END) elaprase_hcp,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hcp_npi END) avlayah_hcp,
-- COUNT(DISTINCT CASE WHEN elaprase_flag=1 THEN hco_veeva_crm_id END) elaprase_hco,
-- COUNT(DISTINCT CASE WHEN avlayah_flag=1 THEN hco_veeva_crm_id END) avlayah_hco,

-- SUM(elaprase_comm) elaprase_commercial_pt_ct,
-- SUM(elaprase_medicare) elaprase_medicare_pt_ct,
-- SUM(elaprase_medicaid) elaprase_medicaid_pt_ct,
-- SUM(elaprase_other) elaprase_other_pt_ct,

-- SUM(avlayah_comm) avlayah_commercial_pt_ct,
-- SUM(avlayah_medicare) avlayah_medicare_pt_ct,
-- SUM(avlayah_medicaid) avlayah_medicaid_pt_ct,
-- SUM(avlayah_other) avlayah_other_pt_ct,
-- SUM(elaprase_denials) elaprase_denials,
-- SUM(avlayah_denials) avlayah_denials,

-- SUM(elaprase_claims_lt_17) elaprase_claims_lt_17,
-- SUM(avlayah_claims_lt_17) avlayah_claims_lt_17,

-- CASE 
--     WHEN SUM(elaprase_claims_lt_17) = 0 THEN 0
--     ELSE ROUND(100.0 * SUM(elaprase_denial_lt_17) / SUM(elaprase_claims_lt_17), 2)
-- END AS elaprase_denial_lt_17,

-- CASE 
--     WHEN SUM(avlayah_claims_lt_17) = 0 THEN 0
--     ELSE ROUND(100.0 * SUM(avlayah_denial_lt_17) / SUM(avlayah_claims_lt_17), 2)
-- END AS avlayah_denial_lt_17,

-- SUM(ELAPRASE_AGE_LT_5_YRS) ELAPRASE_AGE_LT_5_YRS,
-- SUM(ELAPRASE_AGE_5_TO_10_YRS) ELAPRASE_AGE_5_TO_10_YRS,
-- SUM(ELAPRASE_AGE_11_TO_16_YRS) ELAPRASE_AGE_11_TO_16_YRS,
-- SUM(ELAPRASE_AGE_17NGRT_YRS) ELAPRASE_AGE_17NGRT_YRS,

-- SUM(AVLAYAH_AGE_LT_5_YRS) AVLAYAH_AGE_LT_5_YRS,
-- SUM(AVLAYAH_AGE_5_TO_10_YRS) AVLAYAH_AGE_5_TO_10_YRS,
-- SUM(AVLAYAH_AGE_11_TO_16_YRS) AVLAYAH_AGE_11_TO_16_YRS,
-- SUM(AVLAYAH_AGE_17NGRT_YRS) AVLAYAH_AGE_17NGRT_YRS,

-- COUNT(DISTINCT CASE WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
--     OR insurance_group IS NULL THEN patient_id END) other_patients,

-- SUM(new_patient_r1m) new_elaprase_patients_r1m,
-- SUM(new_patient_r3m) new_elaprase_patients_r3m,

-- SUM(new_patient_r1m_avlayah_u17) new_avlayah_u17_patients_r1m,
-- SUM(new_patient_r3m_avlayah_u17) new_avlayah_u17_patients_r3m,

-- COUNT(DISTINCT CASE WHEN age_lt_5_yrs=1 THEN patient_id END) age_lt_5_yrs,
-- COUNT(DISTINCT CASE WHEN age_5_to_10_yrs=1 THEN patient_id END) age_5_to_10_yrs,
-- COUNT(DISTINCT CASE WHEN AGE_11_TO_16_YRS=1 THEN patient_id END) AGE_11_TO_16_YRS,
-- COUNT(DISTINCT CASE WHEN age_17ngrt_yrs=1 THEN patient_id END) age_17ngrt_yrs,

-- SUM(COALESCE(avlayah_pt_lt_17,0)) avlayah_pt_lt_17,
-- SUM(COALESCE(elaprase_pt_lt_17,0)) elaprase_pt_lt_17,

-- COUNT(DISTINCT hcp_npi) total_primary_hcps,
-- COUNT(DISTINCT hco_veeva_crm_id) total_primary_hcos,

-- SUM(total_claims) total_claims,
-- SUM(pharmacy_total_claims) pharmacy_total_claims,
-- SUM(approved_fills) approved_fills,
-- SUM(rejected_fills) rejected_fills,
-- SUM(reversed_fills) reversed_fills,

-- CAST(NULL AS STRING) pie_completed,
-- CAST(NULL AS STRING) account_director,

-- 'NATIONAL' rollup_level

-- FROM base_enriched
-- ),

-- /* ============================================================
-- 11. UNION ALL ROLLUPS
-- ============================================================ */
-- all_rollups AS (

-- SELECT * FROM rollup_territory_payer
-- UNION ALL
-- SELECT * FROM rollup_payer_all_territory
-- UNION ALL
-- SELECT * FROM rollup_territory_all_payer
-- UNION ALL
-- SELECT * FROM rollup_national

-- ),

-- final_with_lives AS (

-- SELECT
-- r.*,
-- COALESCE(t.total_lives,0) total_lives

-- FROM all_rollups r

-- LEFT JOIN total_lives t
--   ON r.rollup_level=t.rollup_level
--  AND r.territory_id=t.territory_id
--  AND r.payer_id=t.payer_id

-- ),

-- final_with_share AS (

-- SELECT
-- f.*,

-- 100.0 * f.total_lives /
-- NULLIF(
-- CASE
-- WHEN f.rollup_level='TERRITORY_PAYER'
-- THEN SUM(f.total_lives) OVER(PARTITION BY f.rollup_level,f.territory_id)
-- ELSE SUM(f.total_lives) OVER(PARTITION BY f.rollup_level)
-- END
-- ,0) payer_market_share_pct

-- FROM final_with_lives f
-- ),

-- /* ============================================================
-- FINAL RANK CALCULATION (EXCLUDE UNKNOWN FROM RANKING)
-- ============================================================ */
-- final_with_rank AS (

-- WITH ranked_payers AS (

-- SELECT
-- f.*,

-- RANK() OVER(
-- PARTITION BY
-- CASE
--     WHEN rollup_level='TERRITORY_PAYER'
--         THEN territory_id
--     ELSE rollup_level
-- END
-- ORDER BY total_elaprase_patients DESC
-- ) AS payer_rank,

-- ROW_NUMBER() OVER(
-- PARTITION BY
-- CASE
--     WHEN rollup_level='TERRITORY_PAYER'
--         THEN territory_id
--     ELSE rollup_level
-- END
-- ORDER BY total_elaprase_patients DESC
-- ) AS row_num

-- FROM final_with_share f
-- WHERE payer_name NOT IN ('Unknown','All Parents')

-- ),

-- rank_counts AS (

-- SELECT
-- CASE
--     WHEN rollup_level='TERRITORY_PAYER'
--         THEN territory_id
--     ELSE rollup_level
-- END AS grp,
-- COUNT(*) AS payer_count
-- FROM ranked_payers
-- GROUP BY 1

-- ),

-- special_rows AS (

-- SELECT
-- f.*,

-- CASE
-- WHEN payer_name='Unknown'
-- THEN r.payer_count
-- WHEN payer_name='All Parents'
-- THEN r.payer_count + 1
-- END AS payer_rank,

-- NULL AS row_num

-- FROM final_with_share f
-- JOIN rank_counts r
-- ON (
-- CASE
--     WHEN f.rollup_level='TERRITORY_PAYER'
--         THEN f.territory_id
--     ELSE f.rollup_level
-- END
-- =
-- r.grp
-- )

-- WHERE payer_name IN ('Unknown','All Parents')

-- )

-- SELECT * FROM ranked_payers
-- UNION ALL
-- SELECT * FROM special_rows

-- )

-- SELECT

-- territory_id,
-- territory_name,
-- CAST(payer_id AS STRING) AS payer_id,
-- payer_name,
-- parent_id,
-- parent_name,

-- payer_market_share_pct,
-- payer_rank,
-- total_lives,

-- total_elaprase_patients,

-- medicare_patients,
-- medicaid_patients,
-- commercial_patients,
-- other_patients,

-- new_elaprase_patients_r1m      AS NEW_ELAPRASE_PATIENTS_R1M,
-- new_elaprase_patients_r3m      AS NEW_ELAPRASE_PATIENTS_R3M,

-- new_avlayah_u17_patients_r1m AS AVLAYAH_U17_NEW_PATIENTS_R1M,
-- new_avlayah_u17_patients_r3m AS AVLAYAH_U17_NEW_PATIENTS_R3M,

-- age_lt_5_yrs                   AS AGE_LT_5_YRS,
-- age_5_to_10_yrs                AS AGE_5_TO_10_YRS,
-- AGE_11_TO_16_YRS               AS AGE_11_TO_16_YRS,
-- age_17ngrt_yrs                 AS AGE_17NGRT_YRS,

-- avlayah_pt_lt_17   AS AVLAYAH_PT_LT_17,
-- elaprase_pt_lt_17  AS ELAPRASE_PT_LT_17,

-- total_primary_hcps             AS TOTAL_PRIMARY_HCPS,
-- total_primary_hcos             AS TOTAL_PRIMARY_HCOS,

-- total_claims                   AS TOTAL_CLAIMS,
-- pharmacy_total_claims          AS PHARMACY_TOTAL_CLAIMS,

-- approved_fills                 AS APPROVED_FILLS,
-- rejected_fills                 AS REJECTED_FILLS,
-- reversed_fills                 AS REVERSED_FILLS,

-- -- <17 metrics
-- elaprase_total_patients,
-- -- elaprase_pt_total,
-- avlayah_pt_total,
-- total_claims_lt17,

-- -- claims
-- elaprase_claims_ct,
-- avlayah_claims_ct,

-- -- HCP/HCO
-- elaprase_hcp,
-- avlayah_hcp,
-- elaprase_hco,
-- avlayah_hco,

-- -- payer split
-- elaprase_commercial_pt_ct,
-- elaprase_medicare_pt_ct,
-- elaprase_medicaid_pt_ct,
-- elaprase_other_pt_ct,

-- avlayah_commercial_pt_ct,
-- avlayah_medicare_pt_ct,
-- avlayah_medicaid_pt_ct,
-- avlayah_other_pt_ct,    

-- -- age
-- elaprase_age_lt_5_yrs,
-- elaprase_age_5_to_10_yrs,
-- elaprase_age_11_to_16_yrs,
-- elaprase_age_17ngrt_yrs,

-- avlayah_age_lt_5_yrs,
-- avlayah_age_5_to_10_yrs,
-- avlayah_age_11_to_16_yrs,
-- avlayah_age_17ngrt_yrs,

-- -- denials
-- elaprase_denial_lt_17,
-- avlayah_denial_lt_17,

-- elaprase_claims_lt_17,
-- avlayah_claims_lt_17,

-- -- denial rates
-- CASE WHEN elaprase_claims_ct=0 THEN 0
-- ELSE ROUND(100.0*elaprase_denials/elaprase_claims_ct,2)
-- END AS elaprase_denial_rate,

-- CASE WHEN avlayah_claims_ct=0 THEN 0
-- ELSE ROUND(100.0*avlayah_denials/avlayah_claims_ct,2)
-- END AS avlayah_denial_rate,

-- CASE
--     WHEN pharmacy_total_claims = 0 THEN 0
--     ELSE ROUND(100.0 * approved_fills / pharmacy_total_claims,2)
-- END AS ELAPRASE_APPROVAL_RATE,

-- CASE
--     WHEN pharmacy_total_claims = 0 THEN 0
--     ELSE ROUND(100.0 * rejected_fills / pharmacy_total_claims,2)
-- END AS ELAPRASE_REJECTION_RATE,

-- CASE
--     WHEN pharmacy_total_claims = 0 THEN 0
--     ELSE ROUND(100.0 * reversed_fills / pharmacy_total_claims,2)
-- END AS ELAPRASE_REVERSED_RATE,

-- pie_completed                  AS PIE_COMPLETED,
-- account_director               AS ACCOUNT_DIRECTOR,

-- rollup_level

-- FROM final_with_rank;

In [0]:
CREATE OR REPLACE TEMP VIEW tx_enriched AS
SELECT
    a.*,
    CASE
        WHEN t.code IN ('54092070001','540920700','J1743') THEN 'ELAPRASE'
        WHEN t.code = '8497600101'
             OR t.code IN ('J3490','J3590','J9999') THEN 'AVLAYAH'
    END AS drug_type,
    CASE
        WHEN UPPER(t.transaction_status) = 'REJECTED' THEN 1
        ELSE 0
    END AS is_denial
FROM all_patient_claims a
LEFT JOIN (
    SELECT
        patient_id,
        medical_event_id AS claim_id,
        service_date AS fill_date,
        COALESCE(ndc11, procedure_code) AS code,
        'PAID' AS transaction_status
    FROM com_edp_prd.com_raw.kom_medical_events
    UNION ALL
    SELECT
        patient_id,
        pharmacy_event_id AS claim_id,
        fill_date,
        ndc11 AS code,
        transaction_result AS transaction_status
    FROM com_edp_prd.com_raw.kom_pharmacy_events
) t
    ON a.patient_id = t.patient_id
   AND a.claim_id   = t.claim_id;
 
 
CREATE OR REPLACE TEMP VIEW all_patient_claims_expanded AS
SELECT
    c.patient_id,
    c.claim_id,
    c.fill_date,
    p.hcp_npi,
    p.hcp_name,
    p.hcp_specialty,
    CASE
        WHEN p.hco_veeva_crm_id IS NULL
          OR TRIM(p.hco_veeva_crm_id) = ''
          OR p.hco_veeva_crm_id = '-'
          OR UPPER(p.hco_veeva_crm_id) = 'UNKNOWN'
        THEN 'Unknown HCO'
        ELSE p.hco_veeva_crm_id
    END AS hco_veeva_crm_id,
    CASE
        WHEN p.hco_veeva_crm_id IS NULL
          OR TRIM(p.hco_veeva_crm_id) = ''
          OR p.hco_veeva_crm_id = '-'
          OR UPPER(p.hco_veeva_crm_id) = 'UNKNOWN'
        THEN 'Unknown HCO'
        ELSE COALESCE(p.hco_name, 'Unknown HCO')
    END AS hco_name,
    p.territory_id,
    p.territory AS territory_name,
    p.region_id,
    p.region AS region_name,
    COALESCE(r.canonical_payer_id, CAST(p.PAYER_ID AS STRING)) AS PAYER_ID,
    COALESCE(r.canonical_payer_name, p.PAYER_NAME) AS PAYER_NAME,
    COALESCE(CAST(r.canonical_payer_id AS STRING), 'Unknown') AS parent_id,
    COALESCE(r.canonical_payer_name, 'Unknown') AS parent_name,
    YEAR(CURRENT_DATE) - YEAR(pat_dem.patient_yob) AS patient_age,
    p.avlayah_pt_lt_17,
    p.elaprase_pt_lt_17
FROM all_patient_claims c
INNER JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    ON c.patient_id = p.patient_id
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
    ON p.PAYER_ID = r.source_payer_id
LEFT JOIN (
    SELECT patient_id, MAX(patient_yob) AS patient_yob
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
) pat_dem
    ON c.patient_id = pat_dem.patient_id;
 
 
CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL AS
WITH patient_age_map AS (
    SELECT
        patient_id,
        YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
),
base AS (
    SELECT
        a.patient_id,
        a.claim_id,
        a.fill_date,
        pm.hcp_npi,
        pm.hcp_name,
        pm.hcp_specialty,
        pm.territory_id,
        pm.territory AS territory_name,
        pm.hco_veeva_crm_id,
        pm.hco_name,
        pm.insurance_group,
        pm.payer_name AS payer_group,
        CAST(pm.payer_id AS STRING) AS payer_id_rollup,
        CAST(pm.parent_id AS STRING) AS parent_id_rollup,
        pm.parent_name AS parent_name_rollup,
        pa.patient_age,
        pm.avlayah_pt_lt_17,
        pm.elaprase_pt_lt_17
    FROM tx_enriched a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
        ON a.patient_id = pm.patient_id
    LEFT JOIN patient_age_map pa
        ON a.patient_id = pa.patient_id
),
territory_payer AS (
    SELECT
        territory_id,
        territory_name,
        payer_id_rollup AS payer_id,
        payer_group     AS payer_name,
        hcp_npi,
        hcp_name,
        hcp_specialty,
        hco_veeva_crm_id,
        MAX(hco_name)            AS hco_name,
        MAX(parent_id_rollup)    AS parent_id,
        MAX(parent_name_rollup)  AS parent_name,
        COUNT(DISTINCT patient_id) AS patient_count,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE
            WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
              OR insurance_group IS NULL
            THEN patient_id END) AS other_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 5                 THEN patient_id END) AS patient_count_age_less_than_5,
        COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5  AND 10   THEN patient_id END) AS patient_count_age_5_10,
        COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16   THEN patient_id END) AS patient_count_age_11_16,
        COUNT(DISTINCT CASE WHEN patient_age >= 17               THEN patient_id END) AS patient_count_age_greater_than_17,
        COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17  = 1 THEN patient_id END) AS avlayah_pt_lt_17,
        COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) AS elaprase_pt_lt_17,
        COUNT(DISTINCT claim_id)         AS claims_count,
        COUNT(DISTINCT hco_veeva_crm_id) AS total_hcos,
        MAX(fill_date)                   AS last_treatment_date,
        'TERRITORY_PAYER' AS rollup_level
    FROM base
    GROUP BY
        territory_id, territory_name,
        payer_id_rollup, payer_group,
        hcp_npi, hcp_name, hcp_specialty,
        hco_veeva_crm_id
),
payer_all_territory AS (
    SELECT
        'ALL Territories' AS territory_id,
        'All Territories' AS territory_name,
        payer_id_rollup   AS payer_id,
        payer_group       AS payer_name,
        hcp_npi,
        hcp_name,
        hcp_specialty,
        hco_veeva_crm_id,
        MAX(hco_name)           AS hco_name,
        MAX(parent_id_rollup)   AS parent_id,
        MAX(parent_name_rollup) AS parent_name,
        COUNT(DISTINCT patient_id) AS patient_count,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE
            WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
              OR insurance_group IS NULL
            THEN patient_id END) AS other_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 5                 THEN patient_id END) AS patient_count_age_less_than_5,
        COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5  AND 10   THEN patient_id END) AS patient_count_age_5_10,
        COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16   THEN patient_id END) AS patient_count_age_11_16,
        COUNT(DISTINCT CASE WHEN patient_age >= 17               THEN patient_id END) AS patient_count_age_greater_than_17,
        COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17  = 1 THEN patient_id END) AS avlayah_pt_lt_17,
        COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) AS elaprase_pt_lt_17,
        COUNT(DISTINCT claim_id)         AS claims_count,
        COUNT(DISTINCT hco_veeva_crm_id) AS total_hcos,
        MAX(fill_date)                   AS last_treatment_date,
        'PAYER_ALL_TERRITORY' AS rollup_level
    FROM base
    GROUP BY
        payer_id_rollup, payer_group,
        hcp_npi, hcp_name, hcp_specialty,
        hco_veeva_crm_id
),
territory_all_payer AS (
    SELECT
        territory_id,
        territory_name,
        'ALL Payers'  AS payer_id,
        'All Payers'  AS payer_name,
        'ALL HCPs'    AS hcp_npi,
        'All HCPs'    AS hcp_name,
        'All HCPs'    AS hcp_specialty,
        'ALL HCOs'    AS hco_veeva_crm_id,
        'All HCOs'    AS hco_name,
        'ALL Parents' AS parent_id,
        'All Parents' AS parent_name,
        COUNT(DISTINCT patient_id) AS patient_count,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE
            WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
              OR insurance_group IS NULL
            THEN patient_id END) AS other_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 5                 THEN patient_id END) AS patient_count_age_less_than_5,
        COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5  AND 10   THEN patient_id END) AS patient_count_age_5_10,
        COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16   THEN patient_id END) AS patient_count_age_11_16,
        COUNT(DISTINCT CASE WHEN patient_age >= 17               THEN patient_id END) AS patient_count_age_greater_than_17,
        COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17  = 1 THEN patient_id END) AS avlayah_pt_lt_17,
        COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) AS elaprase_pt_lt_17,
        COUNT(DISTINCT claim_id)         AS claims_count,
        COUNT(DISTINCT hco_veeva_crm_id) AS total_hcos,
        MAX(fill_date)                   AS last_treatment_date,
        'TERRITORY_ALL_PAYER' AS rollup_level
    FROM base
    GROUP BY territory_id, territory_name
),
national AS (
    SELECT
        'ALL Territories' AS territory_id,
        'All Territories' AS territory_name,
        'ALL Payers'      AS payer_id,
        'All Payers'      AS payer_name,
        'ALL HCPs'        AS hcp_npi,
        'All HCPs'        AS hcp_name,
        'All HCPs'        AS hcp_specialty,
        'ALL HCOs'        AS hco_veeva_crm_id,
        'All HCOs'        AS hco_name,
        'ALL Parents'     AS parent_id,
        'All Parents'     AS parent_name,
        COUNT(DISTINCT patient_id) AS patient_count,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE
            WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
              OR insurance_group IS NULL
            THEN patient_id END) AS other_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 5                 THEN patient_id END) AS patient_count_age_less_than_5,
        COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5  AND 10   THEN patient_id END) AS patient_count_age_5_10,
        COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16   THEN patient_id END) AS patient_count_age_11_16,
        COUNT(DISTINCT CASE WHEN patient_age >= 17               THEN patient_id END) AS patient_count_age_greater_than_17,
        COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17  = 1 THEN patient_id END) AS avlayah_pt_lt_17,
        COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) AS elaprase_pt_lt_17,
        COUNT(DISTINCT claim_id)         AS claims_count,
        COUNT(DISTINCT hco_veeva_crm_id) AS total_hcos,
        MAX(fill_date)                   AS last_treatment_date,
        'NATIONAL' AS rollup_level
    FROM base
)
SELECT * FROM territory_payer
UNION ALL
SELECT * FROM payer_all_territory
UNION ALL
SELECT * FROM territory_all_payer
UNION ALL
SELECT * FROM national;
 
 
CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL_PATIENT_LEVEL AS
WITH patient_age_map AS (
    SELECT
        patient_id,
        YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
),
base AS (
    SELECT
        a.*,
        pm.territory_id,
        pm.territory AS territory_name,
        pm.hcp_npi,
        pm.hcp_name,
        pm.hcp_specialty,
        pm.hco_veeva_crm_id,
        pm.hco_name,
        pm.insurance_group,
        pm.payer_name AS payer_group,
        CAST(pm.payer_id  AS STRING) AS payer_id_rollup,
        CAST(pm.parent_id AS STRING) AS parent_id_rollup,
        pm.parent_name              AS parent_name_rollup,
        pa.patient_age
    FROM tx_enriched a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
        ON a.patient_id = pm.patient_id
    LEFT JOIN patient_age_map pa
        ON a.patient_id = pa.patient_id
),
territory_payer AS (
    SELECT
        territory_id,
        territory_name,
        payer_id_rollup    AS payer_id,
        payer_group        AS payer_name,
        hcp_npi,
        hcp_name,
        hcp_specialty,
        hco_veeva_crm_id,
        hco_name,
        parent_id_rollup   AS parent_id,
        parent_name_rollup AS parent_name,
        patient_id,
        insurance_group,
        claim_id,
        fill_date,
        patient_age,
        'TERRITORY_PAYER' AS rollup_level
    FROM base
),
payer_all_territory AS (
    SELECT
        'ALL Territories'  AS territory_id,
        'All Territories'  AS territory_name,
        payer_id_rollup    AS payer_id,
        payer_group        AS payer_name,
        hcp_npi,
        hcp_name,
        hcp_specialty,
        hco_veeva_crm_id,
        hco_name,
        parent_id_rollup   AS parent_id,
        parent_name_rollup AS parent_name,
        patient_id,
        insurance_group,
        claim_id,
        fill_date,
        patient_age,
        'PAYER_ALL_TERRITORY' AS rollup_level
    FROM base
),
territory_all_payer AS (
    SELECT
        territory_id,
        territory_name,
        'ALL Payers'  AS payer_id,
        'All Payers'  AS payer_name,
        'ALL HCPs'    AS hcp_npi,
        'All HCPs'    AS hcp_name,
        'All HCPs'    AS hcp_specialty,
        'ALL HCOs'    AS hco_veeva_crm_id,
        'All HCOs'    AS hco_name,
        'ALL Parents' AS parent_id,
        'All Parents' AS parent_name,
        patient_id,
        insurance_group,
        claim_id,
        fill_date,
        patient_age,
        'TERRITORY_ALL_PAYER' AS rollup_level
    FROM base
),
national AS (
    SELECT
        'ALL Territories' AS territory_id,
        'All Territories' AS territory_name,
        'ALL Payers'      AS payer_id,
        'All Payers'      AS payer_name,
        'ALL HCPs'        AS hcp_npi,
        'All HCPs'        AS hcp_name,
        'All HCPs'        AS hcp_specialty,
        'ALL HCOs'        AS hco_veeva_crm_id,
        'All HCOs'        AS hco_name,
        'ALL Parents'     AS parent_id,
        'All Parents'     AS parent_name,
        patient_id,
        insurance_group,
        claim_id,
        fill_date,
        patient_age,
        'NATIONAL' AS rollup_level
    FROM base
),
final_rollups AS (
    SELECT * FROM territory_payer
    UNION ALL
    SELECT * FROM payer_all_territory
    UNION ALL
    SELECT * FROM territory_all_payer
    UNION ALL
    SELECT * FROM national
)
SELECT DISTINCT
    territory_id,
    territory_name,
    payer_id,
    payer_name,
    hcp_npi,
    hcp_name,
    hcp_specialty,
    hco_veeva_crm_id,
    hco_name,
    parent_id,
    parent_name,
    patient_id,
    insurance_group,
    fill_date,
    patient_age,
    rollup_level
FROM final_rollups;
 
 
CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL AS
WITH patient_age_map AS (
    SELECT
        patient_id,
        YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
),
base AS (
    SELECT
        a.*,
        pm.territory_id,
        pm.territory AS territory_name,
        pm.hcp_npi,
        pm.hcp_name,
        pm.hcp_specialty,
        pm.hco_veeva_crm_id,
        pm.hco_name,
        pm.insurance_group,
        pm.payer_name AS payer_group,
        pa.patient_age,
        CAST(pm.payer_id  AS STRING) AS payer_id_rollup,
        CAST(pm.parent_id AS STRING) AS parent_id_rollup,
        pm.parent_name              AS parent_name_rollup
    FROM tx_enriched a
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
        ON a.patient_id = pm.patient_id
    LEFT JOIN patient_age_map pa
        ON a.patient_id = pa.patient_id
),
territory_payer AS (
    SELECT
        territory_id,
        territory_name,
        payer_id_rollup AS payer_id,
        payer_group     AS payer_name,
        hco_veeva_crm_id,
        MAX(hco_name)           AS hco_name,
        MAX(parent_id_rollup)   AS parent_id,
        MAX(parent_name_rollup) AS parent_name,
        COUNT(DISTINCT patient_id) AS patient_count,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE
            WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
              OR insurance_group IS NULL
            THEN patient_id END) AS other_patients,
        COUNT(DISTINCT claim_id) AS claims_count,
        MAX(fill_date)           AS last_treatment_date,
        'TERRITORY_PAYER' AS rollup_level
    FROM base
    GROUP BY territory_id, territory_name, payer_id_rollup, payer_group, hco_veeva_crm_id
),
payer_all_territory AS (
    SELECT
        'ALL Territories' AS territory_id,
        'All Territories' AS territory_name,
        payer_id_rollup   AS payer_id,
        payer_group       AS payer_name,
        hco_veeva_crm_id,
        MAX(hco_name)           AS hco_name,
        MAX(parent_id_rollup)   AS parent_id,
        MAX(parent_name_rollup) AS parent_name,
        COUNT(DISTINCT patient_id) AS patient_count,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE
            WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
              OR insurance_group IS NULL
            THEN patient_id END) AS other_patients,
        COUNT(DISTINCT claim_id) AS claims_count,
        MAX(fill_date)           AS last_treatment_date,
        'PAYER_ALL_TERRITORY' AS rollup_level
    FROM base
    GROUP BY payer_id_rollup, payer_group, hco_veeva_crm_id
),
territory_all_payer AS (
    SELECT
        territory_id,
        territory_name,
        'ALL Payers'  AS payer_id,
        'All Payers'  AS payer_name,
        'ALL HCOs'    AS hco_veeva_crm_id,
        'All HCOs'    AS hco_name,
        'ALL Parents' AS parent_id,
        'All Parents' AS parent_name,
        COUNT(DISTINCT patient_id) AS patient_count,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE
            WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
              OR insurance_group IS NULL
            THEN patient_id END) AS other_patients,
        COUNT(DISTINCT claim_id) AS claims_count,
        MAX(fill_date)           AS last_treatment_date,
        'TERRITORY_ALL_PAYER' AS rollup_level
    FROM base
    GROUP BY territory_id, territory_name
),
national AS (
    SELECT
        'ALL Territories' AS territory_id,
        'All Territories' AS territory_name,
        'ALL Payers'      AS payer_id,
        'All Payers'      AS payer_name,
        'ALL HCOs'        AS hco_veeva_crm_id,
        'All HCOs'        AS hco_name,
        'ALL Parents'     AS parent_id,
        'All Parents'     AS parent_name,
        COUNT(DISTINCT patient_id) AS patient_count,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICARE'   THEN patient_id END) AS medicare_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'MEDICAID'   THEN patient_id END) AS medicaid_patients,
        COUNT(DISTINCT CASE WHEN insurance_group = 'COMMERCIAL' THEN patient_id END) AS commercial_patients,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE
            WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
              OR insurance_group IS NULL
            THEN patient_id END) AS other_patients,
        COUNT(DISTINCT claim_id) AS claims_count,
        MAX(fill_date)           AS last_treatment_date,
        'NATIONAL' AS rollup_level
    FROM base
),
ref_dedup AS (
    SELECT
        hco_veeva_crm_id,
        MAX(hco_city)  AS hco_city,
        MAX(hco_state) AS hco_state
    FROM com_edp_prd.cmpa_insights_internal_schema.reference_file
    GROUP BY hco_veeva_crm_id
)
SELECT
    a.*,
    CASE WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL' ELSE ref.hco_city  END AS hco_city,
    CASE WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL' ELSE ref.hco_state END AS hco_state
FROM (
    SELECT * FROM territory_payer
    UNION ALL
    SELECT * FROM payer_all_territory
    UNION ALL
    SELECT * FROM territory_all_payer
    UNION ALL
    SELECT * FROM national
) a
LEFT JOIN ref_dedup ref
    ON ref.hco_veeva_crm_id = a.hco_veeva_crm_id;
 
 
WITH tx_classification AS (
    SELECT
        patient_id,
        claim_id,
        fill_date,
        CASE
            WHEN code IN ('54092070001','540920700','J1743') THEN 'ELAPRASE'
            WHEN code = '8497600101'
                 OR code IN ('J3490','J3590','J9999') THEN 'AVLAYAH'
            ELSE NULL
        END AS drug_type,
        CASE
            WHEN UPPER(transaction_status) = 'REJECTED' THEN 1
            ELSE 0
        END AS is_denial
    FROM (
        SELECT
            patient_id,
            medical_event_id AS claim_id,
            service_date AS fill_date,
            COALESCE(ndc11, procedure_code) AS code,
            'MEDICAL' AS claim_source,
            'PAID' AS transaction_status
        FROM com_edp_prd.com_raw.kom_medical_events
        UNION ALL
        SELECT
            patient_id,
            pharmacy_event_id AS claim_id,
            fill_date,
            ndc11 AS code,
            'PHARMACY' AS claim_source,
            transaction_result AS transaction_status
        FROM com_edp_prd.com_raw.kom_pharmacy_events
    )
    WHERE code IS NOT NULL
),
patient_age_map AS (
    SELECT
        patient_id,
        YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
),
patient_tx_enriched AS (
    SELECT
        p.*,
        a.patient_age,
        t.claim_id,
        t.fill_date,
        t.drug_type,
        t.is_denial
    FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    LEFT JOIN tx_classification t
        ON p.patient_id = t.patient_id
    LEFT JOIN patient_age_map a
        ON p.patient_id = a.patient_id
),
patient_tx_metrics AS (
    SELECT
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 AND drug_type = 'ELAPRASE' THEN patient_id END) AS elaprase_pt_ct_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 AND drug_type = 'ELAPRASE' THEN claim_id   END) AS elaprase_claims_ct,
        COUNT(DISTINCT CASE WHEN drug_type = 'ELAPRASE' AND insurance_group = 'COMMERCIAL' THEN patient_id END) AS elaprase_commercial_pt_ct,
        COUNT(DISTINCT CASE WHEN drug_type = 'ELAPRASE' AND insurance_group = 'MEDICARE'   THEN patient_id END) AS elaprase_medicare_pt_ct,
        COUNT(DISTINCT CASE WHEN drug_type = 'ELAPRASE'
                            AND insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL') THEN patient_id END) AS elaprase_other_pt_ct,
        COUNT(DISTINCT CASE WHEN patient_age < 17 AND drug_type = 'AVLAYAH' THEN patient_id END) AS avlayah_pt_ct_lt17,
        COUNT(DISTINCT CASE WHEN drug_type = 'AVLAYAH' THEN claim_id END) AS avlayah_claims_ct,
        COUNT(DISTINCT CASE WHEN drug_type = 'AVLAYAH' AND insurance_group = 'COMMERCIAL' THEN patient_id END) AS avlayah_commercial_pt_ct,
        COUNT(DISTINCT CASE WHEN drug_type = 'AVLAYAH' AND insurance_group = 'MEDICARE'   THEN patient_id END) AS avlayah_medicare_pt_ct,
        COUNT(DISTINCT CASE WHEN drug_type = 'AVLAYAH'
                            AND insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL') THEN patient_id END) AS avlayah_other_pt_ct,
        CASE
            WHEN COUNT(CASE WHEN drug_type = 'ELAPRASE' THEN claim_id END) = 0 THEN 0
            ELSE ROUND(
                100.0 * SUM(CASE WHEN drug_type = 'ELAPRASE' AND is_denial = 1 THEN 1 ELSE 0 END)
                / COUNT(CASE WHEN drug_type = 'ELAPRASE' THEN claim_id END), 2)
        END AS elaprase_denial_rate,
        CASE
            WHEN COUNT(CASE WHEN drug_type = 'AVLAYAH' THEN claim_id END) = 0 THEN 0
            ELSE ROUND(
                100.0 * SUM(CASE WHEN drug_type = 'AVLAYAH' AND is_denial = 1 THEN 1 ELSE 0 END)
                / COUNT(CASE WHEN drug_type = 'AVLAYAH' THEN claim_id END), 2)
        END AS avlayah_denial_rate,
        COUNT(DISTINCT CASE WHEN drug_type = 'ELAPRASE' THEN hcp_npi          END) AS elaprase_hcp,
        COUNT(DISTINCT CASE WHEN drug_type = 'AVLAYAH'  THEN hcp_npi          END) AS avlayah_hcp,
        COUNT(DISTINCT CASE WHEN drug_type = 'ELAPRASE' THEN hco_veeva_crm_id END) AS elaprase_hco,
        COUNT(DISTINCT CASE WHEN drug_type = 'AVLAYAH'  THEN hco_veeva_crm_id END) AS avlayah_hco
    FROM patient_tx_enriched
)
SELECT * FROM patient_tx_metrics;
 
 
CREATE OR REPLACE TABLE cmpa_insights_internal_schema.patient360_master_enriched AS
WITH tx_classification AS (
    SELECT
        patient_id,
        claim_id,
        fill_date,
        CASE
            WHEN code IN ('54092070001','540920700','J1743') THEN 'ELAPRASE'
            WHEN code = '8497600101'
                 OR code IN ('J3490','J3590','J9999') THEN 'AVLAYAH'
        END AS drug_type,
        CASE
            WHEN UPPER(transaction_status) = 'REJECTED' THEN 1
            ELSE 0
        END AS is_denial
    FROM (
        SELECT
            patient_id,
            medical_event_id AS claim_id,
            service_date AS fill_date,
            COALESCE(ndc11, procedure_code) AS code,
            'PAID' AS transaction_status
        FROM com_edp_prd.com_raw.kom_medical_events
        UNION ALL
        SELECT
            patient_id,
            pharmacy_event_id AS claim_id,
            fill_date,
            ndc11 AS code,
            transaction_result AS transaction_status
        FROM com_edp_prd.com_raw.kom_pharmacy_events
    )
),
patient_age_map AS (
    SELECT
        patient_id,
        YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
),
base AS (
    SELECT
        p.patient_id,
        p.insurance_group,
        a.patient_age,
        t.claim_id,
        t.drug_type,
        t.is_denial
    FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
    LEFT JOIN tx_classification t
        ON p.patient_id = t.patient_id
    LEFT JOIN patient_age_map a
        ON p.patient_id = a.patient_id
),
patient_tx_patient_level AS (
    SELECT
        patient_id,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,
        CASE WHEN MAX(patient_age) < 17 THEN 1 ELSE 0 END AS total_patients_lt17,
        MAX(CASE WHEN patient_age < 17 AND drug_type = 'ELAPRASE' THEN 1 ELSE 0 END) AS elaprase_pt_ct_lt17,
        COUNT(DISTINCT CASE WHEN drug_type = 'ELAPRASE' THEN claim_id END) AS elaprase_claims_ct,
        MAX(CASE WHEN patient_age < 17 AND drug_type = 'AVLAYAH' THEN 1 ELSE 0 END) AS avlayah_pt_ct_lt17,
        COUNT(DISTINCT CASE WHEN drug_type = 'AVLAYAH' THEN claim_id END) AS avlayah_claims_ct,
        MAX(CASE WHEN drug_type = 'ELAPRASE' AND insurance_group = 'COMMERCIAL' THEN 1 ELSE 0 END) AS elaprase_commercial_pt_ct,
        MAX(CASE WHEN drug_type = 'ELAPRASE' AND insurance_group = 'MEDICARE'   THEN 1 ELSE 0 END) AS elaprase_medicare_pt_ct,
        MAX(CASE WHEN drug_type = 'ELAPRASE'
                  AND insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL') THEN 1 ELSE 0 END) AS elaprase_other_pt_ct,
        MAX(CASE WHEN drug_type = 'AVLAYAH' AND insurance_group = 'COMMERCIAL' THEN 1 ELSE 0 END) AS avlayah_commercial_pt_ct,
        MAX(CASE WHEN drug_type = 'AVLAYAH' AND insurance_group = 'MEDICARE'   THEN 1 ELSE 0 END) AS avlayah_medicare_pt_ct,
        MAX(CASE WHEN drug_type = 'AVLAYAH'
                  AND insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL') THEN 1 ELSE 0 END) AS avlayah_other_pt_ct,
        CASE
            WHEN COUNT(CASE WHEN drug_type = 'ELAPRASE' THEN claim_id END) = 0 THEN 0
            ELSE ROUND(
                100.0 * SUM(CASE WHEN drug_type = 'ELAPRASE' AND is_denial = 1 THEN 1 ELSE 0 END)
                / COUNT(CASE WHEN drug_type = 'ELAPRASE' THEN claim_id END), 2)
        END AS elaprase_denial_rate,
        CASE
            WHEN COUNT(CASE WHEN drug_type = 'AVLAYAH' THEN claim_id END) = 0 THEN 0
            ELSE ROUND(
                100.0 * SUM(CASE WHEN drug_type = 'AVLAYAH' AND is_denial = 1 THEN 1 ELSE 0 END)
                / COUNT(CASE WHEN drug_type = 'AVLAYAH' THEN claim_id END), 2)
        END AS avlayah_denial_rate
    FROM base
    GROUP BY patient_id
)
SELECT
    m.*,
    t.elaprase_pt_ct_lt17,
    t.avlayah_pt_ct_lt17,
    t.elaprase_denial_rate,
    t.avlayah_denial_rate,
    t.avlayah_commercial_pt_ct,
    t.avlayah_medicare_pt_ct,
    t.avlayah_other_pt_ct,
    t.elaprase_commercial_pt_ct,
    t.elaprase_medicare_pt_ct,
    t.elaprase_other_pt_ct,
    t.total_claims_lt17,
    t.total_patients_lt17,
    t.avlayah_claims_ct,
    t.elaprase_claims_ct
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master m
LEFT JOIN patient_tx_patient_level t
    ON m.patient_id = t.patient_id;
 
 
WITH patient_age_map AS (
    SELECT
        patient_id,
        YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY patient_id
),
tx_enriched_final AS (
    SELECT
        t.*,
        pm.territory_id,
        pm.territory AS territory_name,
        pm.payer_name,
        pm.hco_veeva_crm_id,
        pm.insurance_group,
        pa.patient_age
    FROM tx_enriched t
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
        ON t.patient_id = pm.patient_id
    LEFT JOIN patient_age_map pa
        ON t.patient_id = pa.patient_id
),
payer_tx_metrics AS (
    SELECT
        territory_id,
        payer_name,
        hco_veeva_crm_id,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id   END) AS total_claims_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 AND drug_type = 'ELAPRASE' THEN patient_id END) AS elaprase_pt_ct_lt17,
        COUNT(DISTINCT CASE WHEN patient_age < 17 AND drug_type = 'AVLAYAH'  THEN patient_id END) AS avlayah_pt_ct_lt17,
        COUNT(DISTINCT CASE WHEN drug_type = 'ELAPRASE' THEN claim_id END) AS elaprase_claims_ct,
        COUNT(DISTINCT CASE WHEN drug_type = 'AVLAYAH'  THEN claim_id END) AS avlayah_claims_ct,
        ROUND(
            100.0 * SUM(CASE WHEN drug_type = 'ELAPRASE' AND is_denial = 1 THEN 1 ELSE 0 END)
            / NULLIF(COUNT(CASE WHEN drug_type = 'ELAPRASE' THEN claim_id END), 0)
        , 2) AS elaprase_denial_rate,
        ROUND(
            100.0 * SUM(CASE WHEN drug_type = 'AVLAYAH' AND is_denial = 1 THEN 1 ELSE 0 END)
            / NULLIF(COUNT(CASE WHEN drug_type = 'AVLAYAH' THEN claim_id END), 0)
        , 2) AS avlayah_denial_rate
    FROM tx_enriched_final
    GROUP BY territory_id, payer_name, hco_veeva_crm_id
)
SELECT
    p.*,
    t.total_patients_lt17,
    t.total_claims_lt17,
    t.elaprase_pt_ct_lt17,
    t.avlayah_pt_ct_lt17,
    t.elaprase_claims_ct,
    t.avlayah_claims_ct,
    t.elaprase_denial_rate,
    t.avlayah_denial_rate
FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL p
LEFT JOIN payer_tx_metrics t
    ON p.territory_id      = t.territory_id
   AND p.payer_name        = t.payer_name
   AND p.hco_veeva_crm_id  = t.hco_veeva_crm_id;

In [0]:
-- CREATE OR REPLACE TEMP VIEW tx_enriched AS

-- SELECT
--     a.*,

--     CASE 
--         WHEN t.code IN ('54092070001','540920700','J1743') THEN 'ELAPRASE'
--         WHEN t.code = '8479600101' 
--              OR t.code IN ('J3490','J3590','J9999') THEN 'AVLAYAH'
--     END AS drug_type,

--     CASE 
--         WHEN UPPER(t.transaction_status) = 'REJECTED' THEN 1 
--         ELSE 0 
--     END AS is_denial

-- FROM all_patient_claims a   -- ✅ FIXED

-- LEFT JOIN (
--     SELECT
--         patient_id,
--         medical_event_id AS claim_id,
--         service_date AS fill_date,
--         COALESCE(ndc11, procedure_code) AS code,
--         'PAID' AS transaction_status
--     FROM com_edp_prd.com_raw.kom_medical_events

--     UNION ALL

--     SELECT
--         patient_id,
--         pharmacy_event_id AS claim_id,
--         fill_date,
--         ndc11 AS code,
--         transaction_result AS transaction_status
--     FROM com_edp_prd.com_raw.kom_pharmacy_events
-- ) t
-- ON a.patient_id = t.patient_id
-- AND a.claim_id = t.claim_id;

In [0]:
-- /*
-- PURPOSE
-- - Create HCO-level rollup metrics across Territory and Payer dimensions.
-- - Provide patient count, claims count, and most recent treatment date.
-- - Support multiple aggregation levels using GROUPING SETS.

-- BUSINESS LOGIC
-- 1) Use enriched claim-level dataset (all_patient_claims_expanded).
-- 2) Aggregate metrics at:
--    - Territory + Payer + HCO (most granular; HCP removed)
--    - Payer (all territories) with HCO breakdown
--    - Territory (all payers)
--    - National
-- 3) Default rolled-up dimension values to 'ALL ...'.
-- 4) Label each aggregation level using GROUPING().
-- */

-- CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL AS

-- WITH patient_age AS (
-- SELECT
--     patient_id,
--     YEAR(CURRENT_DATE) - YEAR(patient_yob) AS patient_age
-- FROM (
--     SELECT patient_id, MAX(patient_yob) AS patient_yob
--     FROM com_edp_prd.com_raw.kom_patient_demographics
--     GROUP BY patient_id
-- )
-- ),

-- base AS (

-- SELECT
--     a.*,

--     pm.hco_veeva_crm_id,
--     pm.hco_name,

--     pm.territory_id,
--     pm.territory AS territory_name,

--     pa.patient_age,   -- ⚠️ required for LT17 logic

--     pm.insurance_group,
--     pm.payer_name AS payer_group,

--     CAST(pm.payer_id AS STRING)  AS payer_id_rollup,
--     CAST(pm.parent_id AS STRING) AS parent_id_rollup,
--     pm.parent_name AS parent_name_rollup

-- FROM tx_enriched a

-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
--     ON a.patient_id = pm.patient_id

-- LEFT JOIN patient_age pa
--     ON a.patient_id = pa.patient_id
-- ),

-- /* ============================================================
-- 1. TERRITORY + PAYER + HCO
-- ============================================================ */

-- territory_payer AS (

-- SELECT

-- territory_id,
-- territory_name,
-- payer_id_rollup AS payer_id,
-- payer_group AS payer_name,

-- hco_veeva_crm_id,
-- MAX(hco_name) AS hco_name,

-- MAX(parent_id_rollup) AS parent_id,
-- MAX(parent_name_rollup) AS parent_name,

-- COUNT(DISTINCT patient_id) AS patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) AS other_patients,

-- COUNT(DISTINCT claim_id) AS claims_count,
-- MAX(fill_date) AS last_treatment_date,

-- 'TERRITORY_PAYER' AS rollup_level

-- FROM base

-- GROUP BY
-- territory_id,
-- territory_name,
-- payer_id_rollup,
-- payer_group,
-- hco_veeva_crm_id
-- ),

-- /* ============================================================
-- 2. PAYER ALL TERRITORY
-- ============================================================ */

-- payer_all_territory AS (

-- SELECT

-- 'ALL Territories' AS territory_id,
-- 'All Territories' AS territory_name,

-- payer_id_rollup AS payer_id,
-- payer_group AS payer_name,

-- hco_veeva_crm_id,
-- MAX(hco_name) AS hco_name,

-- MAX(parent_id_rollup) AS parent_id,
-- MAX(parent_name_rollup) AS parent_name,

-- COUNT(DISTINCT patient_id) AS patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) AS other_patients,

-- COUNT(DISTINCT claim_id) AS claims_count,
-- MAX(fill_date) AS last_treatment_date,

-- 'PAYER_ALL_TERRITORY' AS rollup_level

-- FROM base

-- GROUP BY
-- payer_id_rollup,
-- payer_group,
-- hco_veeva_crm_id
-- ),

-- /* ============================================================
-- 3. TERRITORY ALL PAYER
-- ============================================================ */

-- territory_all_payer AS (

-- SELECT

-- territory_id,
-- territory_name,

-- 'ALL Payers' AS payer_id,
-- 'All Payers' AS payer_name,

-- 'ALL HCOs' AS hco_veeva_crm_id,
-- 'All HCOs' AS hco_name,

-- 'ALL Parents' AS parent_id,
-- 'All Parents' AS parent_name,

-- COUNT(DISTINCT patient_id) AS patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) AS other_patients,

-- COUNT(DISTINCT claim_id) AS claims_count,
-- MAX(fill_date) AS last_treatment_date,

-- 'TERRITORY_ALL_PAYER' AS rollup_level

-- FROM base

-- GROUP BY
-- territory_id,
-- territory_name
-- ),

-- /* ============================================================
-- 4. NATIONAL
-- ============================================================ */

-- national AS (

-- SELECT

-- 'ALL Territories' AS territory_id,
-- 'All Territories' AS territory_name,

-- 'ALL Payers' AS payer_id,
-- 'All Payers' AS payer_name,

-- 'ALL HCOs' AS hco_veeva_crm_id,
-- 'All HCOs' AS hco_name,

-- 'ALL Parents' AS parent_id,
-- 'All Parents' AS parent_name,

-- COUNT(DISTINCT patient_id) AS patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) AS other_patients,

-- COUNT(DISTINCT claim_id) AS claims_count,
-- MAX(fill_date) AS last_treatment_date,

-- 'NATIONAL' AS rollup_level

-- FROM base
-- ),
-- ref_dedup AS (
--     SELECT
--         hco_veeva_crm_id,
--         MAX(hco_city)  AS hco_city,
--         MAX(hco_state) AS hco_state
--     FROM com_edp_prd.cmpa_insights_internal_schema.reference_file
--     GROUP BY hco_veeva_crm_id
-- )
-- /* ============================================================
-- FINAL OUTPUT
-- ============================================================ */
-- SELECT 
--     a.*,
--     CASE 
--         WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL'
--         ELSE ref.hco_city
--     END AS hco_city,
    
--     CASE 
--         WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL'
--         ELSE ref.hco_state
--     END AS hco_state

-- FROM (
--     SELECT * FROM territory_payer
--     UNION ALL
--     SELECT * FROM payer_all_territory
--     UNION ALL
--     SELECT * FROM territory_all_payer
--     UNION ALL
--     SELECT * FROM national
-- ) a

-- LEFT JOIN ref_dedup ref
--     ON ref.hco_veeva_crm_id = a.hco_veeva_crm_id;

In [0]:
-- CREATE OR REPLACE TEMP VIEW all_patient_claims_expanded AS

-- SELECT
--     c.patient_id,
--     c.claim_id,
--     c.fill_date,

--     /* ================= HCP ================= */
--     p.hcp_npi,
--     p.hcp_name,
--     p.hcp_specialty,

--     /* ===============================
--    STANDARDIZED HCO ATTRIBUTION
--    (single canonical missing value)
--    =============================== */

--    CASE
--       WHEN p.hco_veeva_crm_id IS NULL
--          OR TRIM(p.hco_veeva_crm_id) = ''
--          OR p.hco_veeva_crm_id = '-'
--          OR UPPER(p.hco_veeva_crm_id) = 'UNKNOWN'
--       THEN 'Unknown HCO'
--       ELSE p.hco_veeva_crm_id
--    END AS hco_veeva_crm_id,

--    CASE
--       WHEN p.hco_veeva_crm_id IS NULL
--          OR TRIM(p.hco_veeva_crm_id) = ''
--          OR p.hco_veeva_crm_id = '-'
--          OR UPPER(p.hco_veeva_crm_id) = 'UNKNOWN'
--       THEN 'Unknown HCO'
--       ELSE COALESCE(p.hco_name,'Unknown HCO')
--    END AS hco_name,

--     /* ================= GEOGRAPHY ================= */
--     p.territory_id,
--     p.territory AS territory_name,
--     p.region_id,
--     p.region AS region_name,

--     /* ================= PAYER GROUPING ================= */
--    COALESCE(r.canonical_payer_id, CAST(p.PAYER_ID AS STRING)) AS PAYER_ID,
--    COALESCE(r.canonical_payer_name, p.PAYER_NAME) AS PAYER_NAME,

--    -- COALESCE(r.canonical_payer_id,'Unknown') AS parent_id,
--    COALESCE(CAST(r.canonical_payer_id AS STRING),'Unknown') AS parent_id,
--    COALESCE(r.canonical_payer_name,'Unknown') AS parent_name,
--     (year(current_date)- year(pat_dem.patient_yob)) as patient_age,
--     p.avlayah_pt_lt_17,
--     p.elaprase_pt_lt_17

-- FROM all_patient_claims c

-- INNER JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
--     ON c.patient_id = p.patient_id

-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_rollup_dim r
--    ON p.PAYER_ID = r.source_payer_id
-- LEFT JOIN (
--         SELECT patient_id, MAX(patient_yob) patient_yob
--         FROM com_edp_prd.com_raw.kom_patient_demographics
--         GROUP BY patient_id
--     ) pat_dem on c.patient_id = pat_dem.patient_id;

In [0]:
-- -- CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL AS

-- -- WITH base AS (

-- -- SELECT
-- --     a.*,

-- --     pm.insurance_group,
-- --     pm.payer_name AS payer_group,

-- --     CAST(pm.payer_id AS STRING)  AS payer_id_rollup,
-- --     CAST(pm.parent_id AS STRING) AS parent_id_rollup,
-- --     pm.parent_name AS parent_name_rollup

-- -- FROM tx_enriched a

-- -- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
-- --     ON a.patient_id = pm.patient_id
-- -- ),

-- WITH patient_age AS (
-- SELECT
--     patient_id,
--     YEAR(CURRENT_DATE) - YEAR(patient_yob) AS patient_age
-- FROM (
--     SELECT patient_id, MAX(patient_yob) AS patient_yob
--     FROM com_edp_prd.com_raw.kom_patient_demographics
--     GROUP BY patient_id
-- )
-- ),

-- base AS (

-- SELECT
--     a.patient_id,
--     a.claim_id,
--     a.fill_date,
--     pm.hcp_npi,

--     /* bring HCP details if available in a */
--     pm.hcp_name,
--     pm.hcp_specialty,

--     /* payer + territory + HCO */
--     pm.territory_id,
--     pm.territory AS territory_name,

--     pm.hco_veeva_crm_id,
--     pm.hco_name,

--     pm.insurance_group,
--     pm.payer_name AS payer_group,

--     CAST(pm.payer_id AS STRING)  AS payer_id_rollup,
--     CAST(pm.parent_id AS STRING) AS parent_id_rollup,
--     pm.parent_name AS parent_name_rollup,

--     /* age */
--     pa.patient_age,

--     /* bring flags if available in tx_enriched OR join from payer360 */
--     pm.avlayah_pt_lt_17,
--     pm.elaprase_pt_lt_17

-- FROM tx_enriched a

-- LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
--     ON a.patient_id = pm.patient_id

-- LEFT JOIN patient_age pa
--     ON a.patient_id = pa.patient_id
-- ),

-- /* ============================================================
-- 1. TERRITORY + PAYER + HCP
-- ============================================================ */

-- territory_payer AS (

-- SELECT

-- territory_id,
-- territory_name,
-- payer_id_rollup AS payer_id,
-- payer_group AS payer_name,

-- hcp_npi,
-- hcp_name,
-- hcp_specialty,

-- hco_veeva_crm_id,
-- MAX(hco_name) hco_name,

-- MAX(parent_id_rollup) parent_id,
-- MAX(parent_name_rollup) parent_name,

-- COUNT(DISTINCT patient_id) patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) other_patients,

-- COUNT(DISTINCT CASE WHEN patient_age<5 THEN patient_id END) patient_count_age_less_than_5,
-- COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5 AND 10 THEN patient_id END) patient_count_age_5_10,
-- COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16 THEN patient_id END) patient_count_age_11_16,
-- COUNT(DISTINCT CASE WHEN patient_age>=17 THEN patient_id END) patient_count_age_greater_than_17,

-- COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17 = 1 THEN patient_id END) avlayah_pt_lt_17,
-- COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) elaprase_pt_lt_17,

-- COUNT(DISTINCT claim_id) claims_count,
-- COUNT(DISTINCT hco_veeva_crm_id) total_hcos,
-- MAX(fill_date) last_treatment_date,

-- 'TERRITORY_PAYER' rollup_level

-- FROM base

-- GROUP BY
-- territory_id,
-- territory_name,
-- payer_id_rollup,
-- payer_group,
-- hcp_npi,
-- hcp_name,
-- hcp_specialty,
-- hco_veeva_crm_id
-- ),

-- /* ============================================================
-- 2. PAYER ALL TERRITORY
-- ============================================================ */

-- payer_all_territory AS (

-- SELECT

-- 'ALL Territories' territory_id,
-- 'All Territories' territory_name,

-- payer_id_rollup AS payer_id,
-- payer_group AS payer_name,

-- hcp_npi,
-- hcp_name,
-- hcp_specialty,

-- hco_veeva_crm_id,
-- MAX(hco_name) hco_name,

-- MAX(parent_id_rollup) parent_id,
-- MAX(parent_name_rollup) parent_name,

-- COUNT(DISTINCT patient_id) patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) other_patients,

-- COUNT(DISTINCT CASE WHEN patient_age<5 THEN patient_id END) patient_count_age_less_than_5,
-- COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5 AND 10 THEN patient_id END) patient_count_age_5_10,
-- COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16 THEN patient_id END) patient_count_age_11_16,
-- COUNT(DISTINCT CASE WHEN patient_age>=17 THEN patient_id END) patient_count_age_greater_than_17,

-- COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17 = 1 THEN patient_id END) avlayah_pt_lt_17,
-- COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) elaprase_pt_lt_17,

-- COUNT(DISTINCT claim_id) claims_count,
-- COUNT(DISTINCT hco_veeva_crm_id) total_hcos,
-- MAX(fill_date) last_treatment_date,

-- 'PAYER_ALL_TERRITORY' rollup_level

-- FROM base

-- GROUP BY
-- payer_id_rollup,
-- payer_group,
-- hcp_npi,
-- hcp_name,
-- hcp_specialty,
-- hco_veeva_crm_id
-- ),

-- /* ============================================================
-- 3. TERRITORY ALL PAYER
-- ============================================================ */

-- territory_all_payer AS (

-- SELECT

-- territory_id,
-- territory_name,

-- 'ALL Payers' payer_id,
-- 'All Payers' payer_name,

-- 'ALL HCPs' hcp_npi,
-- 'All HCPs' hcp_name,
-- 'All HCPs' hcp_specialty,

-- 'ALL HCOs' hco_veeva_crm_id,
-- 'All HCOs' hco_name,

-- 'ALL Parents' parent_id,
-- 'All Parents' parent_name,

-- COUNT(DISTINCT patient_id) patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) other_patients,

-- COUNT(DISTINCT CASE WHEN patient_age<5 THEN patient_id END) patient_count_age_less_than_5,
-- COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5 AND 10 THEN patient_id END) patient_count_age_5_10,
-- COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16 THEN patient_id END) patient_count_age_11_16,
-- COUNT(DISTINCT CASE WHEN patient_age>=17 THEN patient_id END) patient_count_age_greater_than_17,

-- COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17 = 1 THEN patient_id END) avlayah_pt_lt_17,
-- COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) elaprase_pt_lt_17,

-- COUNT(DISTINCT claim_id) claims_count,
-- COUNT(DISTINCT hco_veeva_crm_id) total_hcos,
-- MAX(fill_date) last_treatment_date,

-- 'TERRITORY_ALL_PAYER' rollup_level

-- FROM base

-- GROUP BY
-- territory_id,
-- territory_name
-- ),

-- /* ============================================================
-- 4. NATIONAL
-- ============================================================ */

-- national AS (

-- SELECT

-- 'ALL Territories' territory_id,
-- 'All Territories' territory_name,

-- 'ALL Payers' payer_id,
-- 'All Payers' payer_name,

-- 'ALL HCPs' hcp_npi,
-- 'All HCPs' hcp_name,
-- 'All HCPs' hcp_specialty,

-- 'ALL HCOs' hco_veeva_crm_id,
-- 'All HCOs' hco_name,

-- 'ALL Parents' parent_id,
-- 'All Parents' parent_name,

-- COUNT(DISTINCT patient_id) patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) other_patients,

-- COUNT(DISTINCT CASE WHEN patient_age<5 THEN patient_id END) patient_count_age_less_than_5,
-- COUNT(DISTINCT CASE WHEN patient_age BETWEEN 5 AND 10 THEN patient_id END) patient_count_age_5_10,
-- COUNT(DISTINCT CASE WHEN patient_age BETWEEN 11 AND 16 THEN patient_id END) patient_count_age_11_16,
-- COUNT(DISTINCT CASE WHEN patient_age>=17 THEN patient_id END) patient_count_age_greater_than_17,

-- COUNT(DISTINCT CASE WHEN avlayah_pt_lt_17 = 1 THEN patient_id END) avlayah_pt_lt_17,
-- COUNT(DISTINCT CASE WHEN elaprase_pt_lt_17 = 1 THEN patient_id END) elaprase_pt_lt_17,

-- COUNT(DISTINCT claim_id) claims_count,
-- COUNT(DISTINCT hco_veeva_crm_id) total_hcos,
-- MAX(fill_date) last_treatment_date,

-- 'NATIONAL' rollup_level

-- FROM base
-- )

-- SELECT * FROM territory_payer
-- UNION ALL
-- SELECT * FROM payer_all_territory
-- UNION ALL
-- SELECT * FROM territory_all_payer
-- UNION ALL
-- SELECT * FROM national;

In [0]:
-- CREATE OR REPLACE table cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL_PATIENT_LEVEL AS

-- WITH patient_age_map AS (
--     SELECT 
--         patient_id,
--         YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
--     FROM com_edp_prd.com_raw.kom_patient_demographics
--     GROUP BY patient_id
-- ),

-- base AS (
--     SELECT
--         a.*,

--         /* ✅ territory + provider */
--         pm.territory_id,
--         pm.territory AS territory_name,

--         pm.hcp_npi,
--         pm.hcp_name,
--         pm.hcp_specialty,

--         pm.hco_veeva_crm_id,
--         pm.hco_name,

--         /* payer */
--         pm.insurance_group,
--         pm.payer_name AS payer_group,

--         CAST(pm.payer_id  AS STRING) AS payer_id_rollup,
--         CAST(pm.parent_id AS STRING) AS parent_id_rollup,
--         pm.parent_name             AS parent_name_rollup,

--         /* ✅ FIXED: age comes from demographics, NOT pm */
--         pa.patient_age

--     FROM tx_enriched a

--     LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
--         ON a.patient_id = pm.patient_id

--     LEFT JOIN patient_age_map pa
--         ON a.patient_id = pa.patient_id
-- ),

-- /* ============================================================
--    1. TERRITORY + PAYER  (patient-level)
--    Join key: territory_id, payer_id, hcp_npi, hco_npi
-- ============================================================ */
-- territory_payer AS (
--     SELECT
--         territory_id,
--         territory_name,
--         payer_id_rollup   AS payer_id,
--         payer_group       AS payer_name,
--         hcp_npi,
--         hcp_name,
--         hcp_specialty,
--         hco_veeva_crm_id,
--         hco_name,
--         parent_id_rollup  AS parent_id,
--         parent_name_rollup AS parent_name,
--         patient_id,
--         insurance_group,
--         claim_id,
--         fill_date,
--         patient_age,
--         'TERRITORY_PAYER'  AS rollup_level
--     FROM base
-- ),

-- /* ============================================================
--    2. PAYER ALL TERRITORY  (patient-level)
--    Territory collapsed → 'ALL Territories'
--    Join key: payer_id, hcp_npi, hco_npi
-- ============================================================ */
-- payer_all_territory AS (
--     SELECT
--         'ALL Territories'  AS territory_id,
--         'All Territories'  AS territory_name,
--         payer_id_rollup    AS payer_id,
--         payer_group        AS payer_name,
--         hcp_npi,
--         hcp_name,
--         hcp_specialty,
--         hco_veeva_crm_id,
--         hco_name,
--         parent_id_rollup   AS parent_id,
--         parent_name_rollup AS parent_name,
--         patient_id,
--         insurance_group,
--         claim_id,
--         fill_date,
--         patient_age,
--         'PAYER_ALL_TERRITORY' AS rollup_level
--     FROM base
-- ),

-- /* ============================================================
--    3. TERRITORY ALL PAYER  (patient-level)
--    Payer/HCP/HCO/Parent collapsed → 'ALL ...'
--    Join key: territory_id
-- ============================================================ */
-- territory_all_payer AS (
--     SELECT
--         territory_id,
--         territory_name,
--         'ALL Payers'   AS payer_id,
--         'All Payers'   AS payer_name,
--         'ALL HCPs'     AS hcp_npi,
--         'All HCPs'     AS hcp_name,
--         'All HCPs'     AS hcp_specialty,
--         'ALL HCOs'     AS hco_veeva_crm_id,
--         'All HCOs'     AS hco_name,
--         'ALL Parents'  AS parent_id,
--         'All Parents'  AS parent_name,
--         patient_id,
--         insurance_group,
--         claim_id,
--         fill_date,
--         patient_age,
--         'TERRITORY_ALL_PAYER' AS rollup_level
--     FROM base
-- ),

-- /* ============================================================
--    4. NATIONAL  (patient-level)
--    Everything collapsed → 'ALL ...'
--    Join key: rollup_level only
-- ============================================================ */
-- national AS (
--     SELECT
--         'ALL Territories' AS territory_id,
--         'All Territories' AS territory_name,
--         'ALL Payers'      AS payer_id,
--         'All Payers'      AS payer_name,
--         'ALL HCPs'        AS hcp_npi,
--         'All HCPs'        AS hcp_name,
--         'All HCPs'        AS hcp_specialty,
--         'ALL HCOs'        AS hco_veeva_crm_id,
--         'All HCOs'        AS hco_name,
--         'ALL Parents'     AS parent_id,
--         'All Parents'     AS parent_name,
--         patient_id,
--         insurance_group,
--         claim_id,
--         fill_date,
--         patient_age,
--         'NATIONAL' AS rollup_level
--     FROM base
-- ),

-- final_rollups as (
--   SELECT * FROM territory_payer
-- UNION ALL
-- SELECT * FROM payer_all_territory
-- UNION ALL
-- SELECT * FROM territory_all_payer
-- UNION ALL
-- SELECT * FROM national
-- )

-- SELECT DISTINCT
--     territory_id,
--     territory_name,
--     payer_id,
--     payer_name,
--     hcp_npi,
--     hcp_name,
--     hcp_specialty,
--     hco_veeva_crm_id,
--     hco_name,
--     parent_id,
--     parent_name,
--     patient_id,
--     insurance_group,
--     fill_date,
--     patient_age,
--     rollup_level
-- FROM final_rollups

In [0]:
-- /*
-- PURPOSE
-- - Create HCO-level rollup metrics across Territory and Payer dimensions.
-- - Provide patient count, claims count, and most recent treatment date.
-- - Support multiple aggregation levels using GROUPING SETS.

-- BUSINESS LOGIC
-- 1) Use enriched claim-level dataset (tx_enriched).
-- 2) Aggregate metrics at:
--    - Territory + Payer + HCO (most granular; HCP removed)
--    - Payer (all territories) with HCO breakdown
--    - Territory (all payers)
--    - National
-- 3) Default rolled-up dimension values to 'ALL ...'.
-- 4) Label each aggregation level using GROUPING().
-- */

-- CREATE OR REPLACE TABLE cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL AS

-- WITH patient_age AS (
--     SELECT
--         patient_id,
--         YEAR(CURRENT_DATE) - YEAR(patient_yob) AS patient_age
--     FROM (
--         SELECT 
--             patient_id, 
--             MAX(patient_yob) AS patient_yob
--         FROM com_edp_prd.com_raw.kom_patient_demographics
--         GROUP BY patient_id
--     )
-- ),

-- base AS (
--     SELECT
--         a.*,

--         /* ✅ ADD THESE */
--         pm.territory_id,
--         pm.territory AS territory_name,

--         pm.hcp_npi,
--         pm.hcp_name,
--         pm.hcp_specialty,

--         pm.hco_veeva_crm_id,
--         pm.hco_name,

--         pm.insurance_group,
--         pm.payer_name AS payer_group,

--         pa.patient_age,

--         CAST(pm.payer_id   AS STRING) AS payer_id_rollup,
--         CAST(pm.parent_id  AS STRING) AS parent_id_rollup,
--         pm.parent_name                AS parent_name_rollup

--     FROM tx_enriched a

--     LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
--         ON a.patient_id = pm.patient_id

--     LEFT JOIN patient_age pa
--         ON a.patient_id = pa.patient_id
-- ),

-- /* ============================================================
-- 1. TERRITORY + PAYER + HCO
-- ============================================================ */

-- territory_payer AS (

-- SELECT

-- territory_id,
-- territory_name,
-- payer_id_rollup AS payer_id,
-- payer_group AS payer_name,

-- hco_veeva_crm_id,
-- MAX(hco_name) AS hco_name,

-- MAX(parent_id_rollup) AS parent_id,
-- MAX(parent_name_rollup) AS parent_name,

-- COUNT(DISTINCT patient_id) AS patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) AS other_patients,

-- COUNT(DISTINCT claim_id) AS claims_count,
-- MAX(fill_date) AS last_treatment_date,

-- 'TERRITORY_PAYER' AS rollup_level

-- FROM base

-- GROUP BY
-- territory_id,
-- territory_name,
-- payer_id_rollup,
-- payer_group,
-- hco_veeva_crm_id
-- ),

-- /* ============================================================
-- 2. PAYER ALL TERRITORY
-- ============================================================ */

-- payer_all_territory AS (

-- SELECT

-- 'ALL Territories' AS territory_id,
-- 'All Territories' AS territory_name,

-- payer_id_rollup AS payer_id,
-- payer_group AS payer_name,

-- hco_veeva_crm_id,
-- MAX(hco_name) AS hco_name,

-- MAX(parent_id_rollup) AS parent_id,
-- MAX(parent_name_rollup) AS parent_name,

-- COUNT(DISTINCT patient_id) AS patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) AS other_patients,

-- COUNT(DISTINCT claim_id) AS claims_count,
-- MAX(fill_date) AS last_treatment_date,

-- 'PAYER_ALL_TERRITORY' AS rollup_level

-- FROM base

-- GROUP BY
-- payer_id_rollup,
-- payer_group,
-- hco_veeva_crm_id
-- ),

-- /* ============================================================
-- 3. TERRITORY ALL PAYER
-- ============================================================ */

-- territory_all_payer AS (

-- SELECT

-- territory_id,
-- territory_name,

-- 'ALL Payers' AS payer_id,
-- 'All Payers' AS payer_name,

-- 'ALL HCOs' AS hco_veeva_crm_id,
-- 'All HCOs' AS hco_name,

-- 'ALL Parents' AS parent_id,
-- 'All Parents' AS parent_name,

-- COUNT(DISTINCT patient_id) AS patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) AS other_patients,

-- COUNT(DISTINCT claim_id) AS claims_count,
-- MAX(fill_date) AS last_treatment_date,

-- 'TERRITORY_ALL_PAYER' AS rollup_level

-- FROM base

-- GROUP BY
-- territory_id,
-- territory_name
-- ),

-- /* ============================================================
-- 4. NATIONAL
-- ============================================================ */

-- national AS (

-- SELECT

-- 'ALL Territories' AS territory_id,
-- 'All Territories' AS territory_name,

-- 'ALL Payers' AS payer_id,
-- 'All Payers' AS payer_name,

-- 'ALL HCOs' AS hco_veeva_crm_id,
-- 'All HCOs' AS hco_name,

-- 'ALL Parents' AS parent_id,
-- 'All Parents' AS parent_name,

-- COUNT(DISTINCT patient_id) AS patient_count,

-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICARE' THEN patient_id END) AS medicare_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='MEDICAID' THEN patient_id END) AS medicaid_patients,
-- COUNT(DISTINCT CASE WHEN insurance_group='COMMERCIAL' THEN patient_id END) AS commercial_patients,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

-- COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

-- COUNT(DISTINCT CASE
-- WHEN insurance_group NOT IN ('MEDICARE','MEDICAID','COMMERCIAL')
-- OR insurance_group IS NULL
-- THEN patient_id
-- END) AS other_patients,

-- COUNT(DISTINCT claim_id) AS claims_count,
-- MAX(fill_date) AS last_treatment_date,

-- 'NATIONAL' AS rollup_level

-- FROM base
-- ),
-- ref_dedup AS (
--     SELECT
--         hco_veeva_crm_id,
--         MAX(hco_city)  AS hco_city,
--         MAX(hco_state) AS hco_state
--     FROM com_edp_prd.cmpa_insights_internal_schema.reference_file
--     GROUP BY hco_veeva_crm_id
-- )
-- /* ============================================================
-- FINAL OUTPUT
-- ============================================================ */
-- SELECT 
--     a.*,
--     CASE 
--         WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL'
--         ELSE ref.hco_city
--     END AS hco_city,
    
--     CASE 
--         WHEN a.hco_veeva_crm_id = 'ALL HCOs' THEN 'ALL'
--         ELSE ref.hco_state
--     END AS hco_state

-- FROM (
--     SELECT * FROM territory_payer
--     UNION ALL
--     SELECT * FROM payer_all_territory
--     UNION ALL
--     SELECT * FROM territory_all_payer
--     UNION ALL
--     SELECT * FROM national
-- ) a

-- LEFT JOIN ref_dedup ref
--     ON ref.hco_veeva_crm_id = a.hco_veeva_crm_id;

In [0]:
-- WITH tx_classification AS (

-- SELECT
--     patient_id,
--     claim_id,
--     fill_date,

--     /* ================= DRUG FLAG ================= */
--     CASE 
--         WHEN 
--             /* ELAPRASE */
--             (
--                 code IN ('54092070001','540920700','J1743')
--             )
--         THEN 'ELAPRASE'

--         WHEN 
--             /* AVLayah */
--             (
--                 code = '8479600101'
--                 OR code IN ('J3490','J3590','J9999')
--             )
--         THEN 'AVLAYAH'

--         ELSE NULL
--     END AS drug_type,

--     /* ================= DENIAL FLAG ================= */
--     CASE 
--         WHEN UPPER(transaction_status) = 'REJECTED' THEN 1
--         ELSE 0
--     END AS is_denial

-- FROM (

--     /* MEDICAL */
--     SELECT
--         patient_id,
--         medical_event_id AS claim_id,
--         service_date AS fill_date,
--         COALESCE(ndc11, procedure_code) AS code,
--         'MEDICAL' AS claim_source,
--         'PAID' AS transaction_status
--     FROM com_edp_prd.com_raw.kom_medical_events

--     UNION ALL

--     /* PHARMACY */
--     SELECT
--         patient_id,
--         pharmacy_event_id AS claim_id,
--         fill_date,
--         ndc11 AS code,
--         'PHARMACY',
--         transaction_result AS transaction_status
--     FROM com_edp_prd.com_raw.kom_pharmacy_events

-- )
-- WHERE code IS NOT NULL
-- ),

-- patient_age_map AS (
--     SELECT 
--         patient_id,
--         YEAR(CURRENT_DATE) - YEAR(MAX(patient_yob)) AS patient_age
--     FROM com_edp_prd.com_raw.kom_patient_demographics
--     GROUP BY patient_id
-- ),

-- -- patient_age_map AS (
-- --     SELECT 
-- --         patient_id,
-- --         CASE 
-- --             WHEN MAX(patient_yob) IS NOT NULL 
-- --             THEN CAST(date_format(CURRENT_DATE, 'yyyy') AS INT) - CAST(MAX(patient_yob) AS INT)
-- --         END AS patient_age
-- --     FROM com_edp_prd.com_raw.kom_patient_demographics
-- --     GROUP BY patient_id
-- -- ),
-- -- patient_age_map AS (
-- --     SELECT 
-- --         patient_id,
-- --         YEAR(CURRENT_DATE) - MAX(patient_yob) AS patient_age
-- --     FROM com_edp_prd.com_raw.kom_patient_demographics
-- --     GROUP BY patient_id
-- -- ),

-- patient_tx_enriched AS (
-- SELECT 
--     p.*,
--     a.patient_age,
--     t.claim_id,
--     t.fill_date,
--     t.drug_type,
--     t.is_denial

-- FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p

-- LEFT JOIN tx_classification t
--     ON p.patient_id = t.patient_id

-- LEFT JOIN patient_age_map a
--     ON p.patient_id = a.patient_id
-- ),

--  patient_tx_metrics AS (

-- SELECT

--     /* ================= BASE ================= */
--     COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

--     COUNT(DISTINCT CASE 
--         WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

--     /* ================= ELAPRASE ================= */
--     COUNT(DISTINCT CASE 
--         WHEN patient_age < 17 AND drug_type = 'ELAPRASE'
--         THEN patient_id END) AS elaprase_pt_ct_lt17,

--     COUNT(DISTINCT CASE 
--         WHEN patient_age < 17 AND drug_type = 'ELAPRASE'
--         THEN claim_id END) AS elaprase_claims_ct,

--     COUNT(DISTINCT CASE 
--         WHEN drug_type = 'ELAPRASE' AND insurance_group = 'COMMERCIAL'
--         THEN patient_id END) AS elaprase_commercial_pt_ct,

--     COUNT(DISTINCT CASE 
--         WHEN drug_type = 'ELAPRASE' AND insurance_group = 'MEDICARE'
--         THEN patient_id END) AS elaprase_medicare_pt_ct,

--     COUNT(DISTINCT CASE 
--         WHEN drug_type = 'ELAPRASE' 
--         AND insurance_group NOT IN ('MEDICARE','COMMERCIAL')
--         THEN patient_id END) AS elaprase_other_pt_ct,

--     /* ================= AVLayah ================= */
--     COUNT(DISTINCT CASE 
--         WHEN patient_age < 17 AND drug_type = 'AVLAYAH'
--         THEN patient_id END) AS avlayah_pt_ct_lt17,

--     COUNT(DISTINCT CASE 
--         WHEN drug_type = 'AVLAYAH'
--         THEN claim_id END) AS avlayah_claims_ct,

--     COUNT(DISTINCT CASE 
--         WHEN drug_type = 'AVLAYAH' AND insurance_group = 'COMMERCIAL'
--         THEN patient_id END) AS avlayah_commercial_pt_ct,

--     COUNT(DISTINCT CASE 
--         WHEN drug_type = 'AVLAYAH' AND insurance_group = 'MEDICARE'
--         THEN patient_id END) AS avlayah_medicare_pt_ct,

--     COUNT(DISTINCT CASE 
--         WHEN drug_type = 'AVLAYAH' 
--         AND insurance_group NOT IN ('MEDICARE','COMMERCIAL')
--         THEN patient_id END) AS avlayah_other_pt_ct,

--     /* ================= DENIAL RATES ================= */
--     CASE 
--         WHEN COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END) = 0 THEN 0
--         ELSE ROUND(
--             100.0 * SUM(CASE WHEN drug_type='ELAPRASE' AND is_denial=1 THEN 1 ELSE 0 END)
--             / COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END), 2)
--     END AS elaprase_denial_rate,

--     CASE 
--         WHEN COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END) = 0 THEN 0
--         ELSE ROUND(
--             100.0 * SUM(CASE WHEN drug_type='AVLAYAH' AND is_denial=1 THEN 1 ELSE 0 END)
--             / COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END), 2)
--     END AS avlayah_denial_rate,

--     /* ================= HCP / HCO ================= */
--     COUNT(DISTINCT CASE WHEN drug_type='ELAPRASE' THEN hcp_npi END) AS elaprase_hcp,
--     COUNT(DISTINCT CASE WHEN drug_type='AVLAYAH' THEN hcp_npi END) AS avlayah_hcp,

--     COUNT(DISTINCT CASE WHEN drug_type='ELAPRASE' THEN hco_veeva_crm_id END) AS elaprase_hco,
--     COUNT(DISTINCT CASE WHEN drug_type='AVLAYAH' THEN hco_veeva_crm_id END) AS avlayah_hco

-- FROM patient_tx_enriched
-- )
-- SELECT * FROM patient_tx_metrics;

In [0]:
-- CREATE OR REPLACE TABLE cmpa_insights_internal_schema.patient360_master_enriched AS

-- WITH tx_classification AS (
--     SELECT
--         patient_id,
--         claim_id,
--         fill_date,

--         CASE 
--             WHEN code IN ('54092070001','540920700','J1743') THEN 'ELAPRASE'
--             WHEN code = '8479600101' 
--                  OR code IN ('J3490','J3590','J9999') THEN 'AVLAYAH'
--         END AS drug_type,

--         CASE 
--             WHEN UPPER(transaction_status) = 'REJECTED' THEN 1 
--             ELSE 0 
--         END AS is_denial

--     FROM (
--         SELECT
--             patient_id,
--             medical_event_id AS claim_id,
--             service_date AS fill_date,
--             COALESCE(ndc11, procedure_code) AS code,
--             'PAID' AS transaction_status
--         FROM com_edp_prd.com_raw.kom_medical_events

--         UNION ALL

--         SELECT
--             patient_id,
--             pharmacy_event_id AS claim_id,
--             fill_date,
--             ndc11 AS code,
--             transaction_result AS transaction_status
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--     )
-- ),

-- patient_age_map AS (
--     SELECT 
--         patient_id,
--         YEAR(CURRENT_DATE()) - YEAR(MAX(patient_yob)) AS patient_age
--     FROM com_edp_prd.com_raw.kom_patient_demographics
--     GROUP BY patient_id
-- ),

-- base AS (
--     SELECT 
--         p.patient_id,
--         p.insurance_group,
--         a.patient_age,
--         t.claim_id,
--         t.drug_type,
--         t.is_denial
--     FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level p
--     LEFT JOIN tx_classification t 
--         ON p.patient_id = t.patient_id
--     LEFT JOIN patient_age_map a 
--         ON p.patient_id = a.patient_id
-- ),

-- patient_tx_patient_level AS (
--     SELECT
--         patient_id,

--         /* BASE */
--         COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,
--         CASE WHEN MAX(patient_age) < 17 THEN 1 ELSE 0 END AS total_patients_lt17,

--         /* ELAPRASE */
--         MAX(CASE WHEN patient_age < 17 AND drug_type='ELAPRASE' THEN 1 ELSE 0 END) AS elaprase_pt_ct_lt17,
--         COUNT(DISTINCT CASE WHEN drug_type='ELAPRASE' THEN claim_id END) AS elaprase_claims_ct,

--         /* AVLAYAH */
--         MAX(CASE WHEN patient_age < 17 AND drug_type='AVLAYAH' THEN 1 ELSE 0 END) AS avlayah_pt_ct_lt17,
--         COUNT(DISTINCT CASE WHEN drug_type='AVLAYAH' THEN claim_id END) AS avlayah_claims_ct,

--         /* INSURANCE SPLIT */
--         MAX(CASE WHEN drug_type='ELAPRASE' AND insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) AS elaprase_commercial_pt_ct,
--         MAX(CASE WHEN drug_type='ELAPRASE' AND insurance_group='MEDICARE' THEN 1 ELSE 0 END) AS elaprase_medicare_pt_ct,
--         MAX(CASE WHEN drug_type='ELAPRASE' AND insurance_group NOT IN ('MEDICARE','COMMERCIAL') THEN 1 ELSE 0 END) AS elaprase_other_pt_ct,

--         MAX(CASE WHEN drug_type='AVLAYAH' AND insurance_group='COMMERCIAL' THEN 1 ELSE 0 END) AS avlayah_commercial_pt_ct,
--         MAX(CASE WHEN drug_type='AVLAYAH' AND insurance_group='MEDICARE' THEN 1 ELSE 0 END) AS avlayah_medicare_pt_ct,
--         MAX(CASE WHEN drug_type='AVLAYAH' AND insurance_group NOT IN ('MEDICARE','COMMERCIAL') THEN 1 ELSE 0 END) AS avlayah_other_pt_ct,

--         /* DENIAL RATE */
--         CASE 
--             WHEN COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END)=0 THEN 0
--             ELSE ROUND(
--                 100.0 * SUM(CASE WHEN drug_type='ELAPRASE' AND is_denial=1 THEN 1 ELSE 0 END)
--                 / COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END),2)
--         END AS elaprase_denial_rate,

--         CASE 
--             WHEN COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END)=0 THEN 0
--             ELSE ROUND(
--                 100.0 * SUM(CASE WHEN drug_type='AVLAYAH' AND is_denial=1 THEN 1 ELSE 0 END)
--                 / COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END),2)
--         END AS avlayah_denial_rate

--     FROM base
--     GROUP BY patient_id
-- )

-- SELECT
--     m.*,

--     t.elaprase_pt_ct_lt17,
--     t.avlayah_pt_ct_lt17,
--     t.elaprase_denial_rate,
--     t.avlayah_denial_rate,

--     t.avlayah_commercial_pt_ct,
--     t.avlayah_medicare_pt_ct,
--     t.avlayah_other_pt_ct,

--     t.elaprase_commercial_pt_ct,
--     t.elaprase_medicare_pt_ct,
--     t.elaprase_other_pt_ct,

--     t.total_claims_lt17,
--     t.total_patients_lt17,

--     t.avlayah_claims_ct,
--     t.elaprase_claims_ct

-- FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master m
-- LEFT JOIN patient_tx_patient_level t
--     ON m.patient_id = t.patient_id;

In [0]:
-- with patient_age_map AS (
--     SELECT 
--         patient_id,
--         YEAR(CURRENT_DATE) - MAX(patient_yob) AS patient_age
--     GROUP BY patient_id
-- ),

-- tx_enriched_final AS (
--     SELECT
--         t.*,

--         /* ✅ bring ALL required dimensions */
--         pm.territory_id,
--         pm.territory AS territory_name,

--         pm.payer_name,
--         pm.hco_veeva_crm_id,

--         pm.insurance_group,

--         /* ✅ age */
--         pa.patient_age

--     FROM tx_enriched t

--     LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level pm
--         ON t.patient_id = pm.patient_id

--     LEFT JOIN patient_age_map pa
--         ON t.patient_id = pa.patient_id
-- ),

-- payer_tx_metrics AS (

-- SELECT
--     territory_id,
--     payer_name,
--     hco_veeva_crm_id,

--     /* ================= BASE ================= */
--     COUNT(DISTINCT CASE WHEN patient_age < 17 THEN patient_id END) AS total_patients_lt17,

--     COUNT(DISTINCT CASE WHEN patient_age < 17 THEN claim_id END) AS total_claims_lt17,

--     /* ================= DRUG ================= */
--     COUNT(DISTINCT CASE 
--         WHEN patient_age < 17 AND drug_type='ELAPRASE' 
--         THEN patient_id END) AS elaprase_pt_ct_lt17,

--     COUNT(DISTINCT CASE 
--         WHEN patient_age < 17 AND drug_type='AVLAYAH' 
--         THEN patient_id END) AS avlayah_pt_ct_lt17,

--     COUNT(DISTINCT CASE 
--         WHEN drug_type='ELAPRASE' 
--         THEN claim_id END) AS elaprase_claims_ct,

--     COUNT(DISTINCT CASE 
--         WHEN drug_type='AVLAYAH' 
--         THEN claim_id END) AS avlayah_claims_ct,

--     /* ================= DENIAL ================= */
--     ROUND(
--         100.0 * SUM(CASE WHEN drug_type='ELAPRASE' AND is_denial=1 THEN 1 ELSE 0 END)
--         / NULLIF(COUNT(CASE WHEN drug_type='ELAPRASE' THEN claim_id END),0)
--     ,2) AS elaprase_denial_rate,

--     ROUND(
--         100.0 * SUM(CASE WHEN drug_type='AVLAYAH' AND is_denial=1 THEN 1 ELSE 0 END)
--         / NULLIF(COUNT(CASE WHEN drug_type='AVLAYAH' THEN claim_id END),0)
--     ,2) AS avlayah_denial_rate

-- FROM tx_enriched_final   -- ✅ FIXED

-- GROUP BY
--     territory_id,
--     payer_name,
--     hco_veeva_crm_id
-- )

-- SELECT 
--     p.*,

--     t.total_patients_lt17,
--     t.total_claims_lt17,

--     t.elaprase_pt_ct_lt17,
--     t.avlayah_pt_ct_lt17,

--     t.elaprase_claims_ct,
--     t.avlayah_claims_ct,

--     t.elaprase_denial_rate,
--     t.avlayah_denial_rate

-- FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL p

-- LEFT JOIN payer_tx_metrics t
-- ON p.territory_id = t.territory_id
-- AND p.payer_name = t.payer_name
-- AND p.hco_veeva_crm_id = t.hco_veeva_crm_id;

In [0]:
select * from cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL ;

In [0]:
select distinct TRANSACTION_status from com_edp_prd.com_raw.kom_pharmacy_events